# Module 1

In [1]:
"""
Phase 6 -- Module 01, cell 1 of 3: settings, output folders, shared helpers (P6-12).

Run this first, once per kernel. It reads nothing from earlier phases.
It creates only Phase 6 output folders. Every file Phase 6 writes goes through
atomic_write_bytes(), which refuses any path outside the Phase 6 output folder
(this is the read-only guard for Phases 1-5).
"""

import hashlib
import json
import os
import platform
import sys
import tempfile
from datetime import datetime
from pathlib import Path

# ----------------------------------------------------------------------------
# Locations (Planning Section 10, P6-11)
# ----------------------------------------------------------------------------
ROOT = Path(r"T:\Thesis - Development")
ART = ROOT / "Resources" / "Artifacts"
DIRS = {
    "p1": ART / "Phase 1 - P2",
    "p2": ART / "Phase 2 - P3",
    "p3": ART / "Phase 3 - P1",
    "p4": ART / "Phase 4 - P1",
    "p5": ART / "Phase 5 - P1",
}
OUT = ART / "Phase 6 - P1"
SUBDIRS = [
    "rules", "baseline_predictions", "oof", "claim1", "baselines",
    "holdout_check", "intervals", "demographic_intervals", "assembly", "final",
]

# ----------------------------------------------------------------------------
# Fixed settings (Planning Sections 4-6). Everything later reads these.
# ----------------------------------------------------------------------------
SETTINGS = {
    "alpha": 0.05,
    "holm_family_size": 7,
    "bootstrap_resamples": 1000,
    "bootstrap_seed": 42,
    "ci_percent": 95,
    "undefined_resample_flag_share": 0.05,
    "n_folds": 5,
    "n_classes": 7,
}
# Counts the earlier phases recorded. Module 01's checks compare the files against these.
EXPECTED = {
    "cv_pool_images": 8583,
    "holdout_images": 1432,
    "holdout_lesions": 1067,
    "bcn_scored_rows": 16843,
    "bcn_primary_images": 16284,
    "bcn_primary_lesions": 4591,
    "holdout_demographic_cells_per_model": 168,
    "bcn_demographic_cells_per_model": 112,
    "per_image_scores_rows": 5728,
    "claim2_comparison_rows": 17,
}

# Root-level files that are not "results" (the rules-note gate ignores them)
EXEMPT_ROOT_FILES = {"phase6_start_fingerprints.json", "phase6_input_checks.json"}


# ----------------------------------------------------------------------------
# Shared helpers
# ----------------------------------------------------------------------------
def now_iso():
    return datetime.now().astimezone().isoformat(timespec="seconds")


def banner(title):
    print("\n" + "=" * 78 + "\n" + title + "\n" + "=" * 78)


def sha256_file(path, block=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(block), b""):
            h.update(chunk)
    return h.hexdigest()


def inside_out(path):
    try:
        Path(path).resolve().relative_to(OUT.resolve())
        return True
    except ValueError:
        return False


def atomic_write_bytes(path, data):
    """Write to a temp file in the same folder, then rename. Refuses paths outside OUT."""
    path = Path(path)
    if not inside_out(path):
        raise RuntimeError(f"READ-ONLY GUARD: refusing to write outside the Phase 6 folder: {path}")
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=str(path.parent), suffix=".tmp")
    try:
        with os.fdopen(fd, "wb") as f:
            f.write(data)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp, path)
    except BaseException:
        if os.path.exists(tmp):
            os.remove(tmp)
        raise


def json_bytes(obj):
    return json.dumps(obj, indent=2, ensure_ascii=False).encode("utf-8")


def write_json_once(path, obj):
    """Write obj only if the file is absent; read it back. Returns 'written' or 'exists'."""
    path = Path(path)
    if path.exists():
        return "exists"
    atomic_write_bytes(path, json_bytes(obj))
    back = json.loads(path.read_text(encoding="utf-8"))
    if back != json.loads(json.dumps(obj)):
        raise RuntimeError(f"Read-back mismatch for {path}")
    return "written"


def run_checks(title, checks):
    """checks = [(name, function returning (ok, detail))]. Prints every result; hides nothing."""
    banner(title)
    rows = []
    for name, fn in checks:
        try:
            ok, detail = fn()
        except Exception as exc:  # a check that crashes counts as a failure
            ok, detail = False, f"{type(exc).__name__}: {exc}"
        print(f"  [{'PASS' if ok else 'FAIL'}] {name}  --  {detail}")
        rows.append({"check": name, "passed": bool(ok), "detail": str(detail)})
    n_fail = sum(not r["passed"] for r in rows)
    print(f"\n  {len(rows) - n_fail} of {len(rows)} passed" + ("" if n_fail == 0 else f"  --  {n_fail} FAILED"))
    return rows


def make_output_folders():
    OUT.mkdir(parents=True, exist_ok=True)
    for name in SUBDIRS:
        (OUT / name).mkdir(exist_ok=True)


# ----------------------------------------------------------------------------
# Read-back of this cell
# ----------------------------------------------------------------------------
make_output_folders()
missing = [n for n in SUBDIRS if not (OUT / n).is_dir()]
assert not missing, f"Output subfolders missing after creation: {missing}"
assert not inside_out(DIRS["p3"]), "Guard self-test failed: Phase 3 folder counted as inside Phase 6"
assert inside_out(OUT / "rules" / "x.json"), "Guard self-test failed: Phase 6 folder not recognised"

print("Module 01 cell 1 loaded.")
print(f"  Output folder : {OUT}")
print(f"  Subfolders    : {len(SUBDIRS)} present")
print(f"  Python {sys.version.split()[0]} on {platform.system()}")
print("  Next: run 01_01, then  P6 = initialise_phase6()")

Module 01 cell 1 loaded.
  Output folder : T:\Thesis - Development\Resources\Artifacts\Phase 6 - P1
  Subfolders    : 10 present
  Python 3.11.15 on Windows
  Next: run 01_01, then  P6 = initialise_phase6()


In [2]:
"""
Phase 6 -- Module 01, cell 2 of 3: frozen-input registry, fingerprints, read-only guard.

Requires cell 01_00.  Run:   P6 = initialise_phase6()

What it does
  1. Lists every frozen Phase 1-5 input Phase 6 reads (exact paths from the output inventory).
  2. Stops if a core input is missing. If only the Module 04 inputs (ResNet50/ViT-B/16
     checkpoints, processed images) are missing, it says so and flags Modules 04 and 08.
  3. Fingerprints (SHA-256) every input. First run: records phase6_start_fingerprints.json.
     Later runs: re-hash and compare; any difference stops the run. Module 12 repeats this.
  Nothing in Phases 1-5 is written to.
"""

import hashlib
import json
import os
import platform
import sys


def build_registry():
    p1, p2, p3, p4, p5 = (DIRS[k] for k in ("p1", "p2", "p3", "p4", "p5"))
    reg = []

    def add(name, path, group="core", kind="file"):
        reg.append({"name": name, "path": Path(path), "group": group, "kind": kind})

    # Phase 1
    add("p1_cv_folds", p1 / "cv_fold_assignments.csv")
    add("p1_holdout_split", p1 / "holdout_split.csv")
    add("p1_images_dir", p1 / "Final Output Phase 1" / "images", "module04", "dir")

    # Phase 2 - P3 (run configs sit at the root of the Phase 2 folder)
    for s in ("script1_efficientnet_b3", "script2_resnet50", "script3_vit_b_16"):
        add(f"p2_{s}_run_config", p2 / f"{s}_run_config.json")
    for s in ("script2_resnet50", "script3_vit_b_16"):
        for k in range(5):
            add(f"p2_{s}_fold{k}_best", p2 / s / f"fold_{k}_best.pt", "module04")

    # Phase 3
    add("p3_run_config", p3 / "cv_training" / "run_config.json")
    for m in ("multimodal", "ablation"):
        for k in range(5):
            add(f"p3_{m}_fold{k}_val_predictions",
                p3 / "cv_training" / m / f"fold_{k}_val_predictions.npz")
    for f in ("per_fold_metrics", "mean_sd_summary", "paired_auc_roc_difference",
              "per_class_metrics", "per_class_mean_sd"):
        add(f"p3_results_{f}", p3 / "results_summary" / f"{f}.csv")

    # Phase 4
    add("p4_multimodal_holdout_predictions", p4 / "holdout_predictions" / "multimodal_holdout_predictions.npz")
    add("p4_ablation_holdout_predictions", p4 / "holdout_predictions" / "ablation_fold3_holdout_predictions.npz")
    add("p4_per_image_scores", p4 / "per_image_scores.csv")
    add("p4_claim2_paired_comparison", p4 / "results_summary" / "claim2_paired_comparison.csv")

    # Phase 5
    for m in ("multimodal", "ablation"):
        add(f"p5_bcn_predictions_{m}", p5 / "bcn_predictions" / f"bcn_predictions_{m}.npz")
        add(f"p5_holdout_per_class_{m}", p5 / "holdout_results" / f"holdout_per_class_fp32_{m}.csv")
        add(f"p5_bcn_per_class_{m}", p5 / "bcn_results" / f"bcn_per_class_fp32_{m}.csv")
        add(f"p5_holdout_demographics_{m}", p5 / "demographics" / f"holdout_demographics_fp32_{m}.csv")
        add(f"p5_bcn_demographics_{m}", p5 / "demographics" / f"bcn_demographics_fp32_{m}_EXPLORATORY.csv")
    add("p5_bcn_metadata_encoded", p5 / "bcn_metadata" / "bcn_metadata_encoded.csv")
    add("p5_holdout_overall", p5 / "holdout_results" / "holdout_overall_fp32.csv")
    add("p5_holdout_metrics", p5 / "holdout_results" / "holdout_metrics_fp32.json")
    add("p5_bcn_overall", p5 / "bcn_results" / "bcn_overall_fp32.csv")
    add("p5_bcn_metrics", p5 / "bcn_results" / "bcn_metrics_fp32.json")
    add("p5_bcn_gap_table", p5 / "bcn_results" / "bcn_gap_table.csv")
    add("p5_bcn_gap_all_headline", p5 / "bcn_results" / "bcn_gap_all_headline_metrics.csv")
    add("p5_bcn_gap_json", p5 / "bcn_results" / "bcn_gap.json")
    add("p5_holdout_demographics_sizes", p5 / "demographics" / "holdout_demographics_subgroup_sizes.csv")
    add("p5_holdout_demographics_status", p5 / "demographics" / "holdout_demographics_status_summary.csv")
    add("p5_bcn_demographics_sizes", p5 / "demographics" / "bcn_demographics_subgroup_sizes_EXPLORATORY.csv")
    add("p5_bcn_demographics_status", p5 / "demographics" / "bcn_demographics_status_summary_EXPLORATORY.csv")
    add("p5_rules_note", p5 / "rules" / "phase5_rules_note.json")
    add("p5_rules_note_sha", p5 / "rules" / "phase5_rules_note.sha256.json")
    add("p5_run_config", p5 / "final" / "run_config.json")
    add("p5_exit_gate", p5 / "final" / "phase5_exit_gate.json")
    add("p5_results_digest", p5 / "final" / "phase5_results_digest.md")
    for folder, mf in (("holdout_results", "holdout_manifest"),
                       ("bcn_inventory", "bcn_inventory_manifest"),
                       ("bcn_metadata", "bcn_metadata_manifest"),
                       ("bcn_preprocessed", "bcn_preprocessing_manifest"),
                       ("bcn_predictions", "bcn_predictions_manifest"),
                       ("bcn_results", "bcn_results_manifest"),
                       ("demographics", "demographics_manifest"),
                       ("additions", "additions_manifest")):
        add(f"p5_manifest_{mf}", p5 / folder / f"{mf}.json")
    return reg


def fingerprint_entry(entry):
    p = entry["path"]
    if p.is_dir():
        with os.scandir(p) as it:
            items = sorted((c.name, c.stat().st_size) for c in it if c.is_file())
        digest = hashlib.sha256("\n".join(f"{n}|{s}" for n, s in items).encode("utf-8")).hexdigest()
        return {"kind": "dir", "n_files": len(items), "bytes": sum(s for _, s in items), "sha256": digest}
    return {"kind": "file", "bytes": p.stat().st_size, "sha256": sha256_file(p)}


def environment_snapshot():
    env = {"python": sys.version.split()[0], "platform": platform.platform()}
    for mod in ("numpy", "pandas", "scipy", "sklearn", "torch", "torchvision", "cv2"):
        try:
            env[mod] = __import__(mod).__version__
        except Exception:
            env[mod] = "not importable"
    return env


def initialise_phase6():
    banner("PHASE 6 -- MODULE 01: SETUP, FINGERPRINTS, READ-ONLY GUARD")
    make_output_folders()
    registry = build_registry()

    missing_core = [e for e in registry if e["group"] == "core" and not e["path"].exists()]
    missing_m4 = [e for e in registry if e["group"] == "module04" and not e["path"].exists()]
    if missing_core:
        for e in missing_core:
            print(f"  MISSING (core): {e['path']}")
        raise RuntimeError(f"{len(missing_core)} core input(s) missing. Nothing was written. Tell Claude which ones.")
    if missing_m4:
        for e in missing_m4:
            print(f"  MISSING (Module 04 input): {e['path']}")
        print("  -> Modules 04 and 08 cannot run without these (cut order 3, P6-5). Everything else can.")

    present = [e for e in registry if e["path"].exists()]
    print(f"  Hashing {len(present)} frozen inputs (the ViT and ResNet checkpoints take a little while)...")
    current = {}
    for e in present:
        current[e["name"]] = {"path": str(e["path"]), "group": e["group"], **fingerprint_entry(e)}

    fp_path = OUT / "phase6_start_fingerprints.json"
    if not fp_path.exists():
        record = {
            "format_version": 1,
            "recorded_at": now_iso(),
            "n_entries": len(current),
            "absent_module04_inputs": [e["name"] for e in missing_m4],
            "environment": environment_snapshot(),
            "entries": current,
        }
        write_json_once(fp_path, record)
        status = "recorded (first run)"
    else:
        saved = json.loads(fp_path.read_text(encoding="utf-8"))
        old = saved["entries"]
        changed = [n for n in current if n in old and
                   (old[n]["sha256"] != current[n]["sha256"] or old[n]["bytes"] != current[n]["bytes"])]
        gone = [n for n in old if n not in current]
        new = [n for n in current if n not in old]
        if changed or gone or new:
            for n in changed:
                print(f"  CHANGED: {n}")
            for n in gone:
                print(f"  NOW MISSING: {n}")
            for n in new:
                print(f"  NOT IN THE RECORD (was absent on the first run): {n}")
            raise RuntimeError("Frozen inputs differ from the first-run record. Stop and tell Claude; nothing was changed.")
        status = "matched the first-run record"
        env_now, env_then = environment_snapshot(), saved.get("environment", {})
        for k, v in env_now.items():
            if env_then.get(k) != v:
                print(f"  Note (environment, not a stop): {k} was {env_then.get(k)}, now {v}")

    # Read-back: the record on disk covers exactly what was just hashed
    back = json.loads(fp_path.read_text(encoding="utf-8"))
    assert set(back["entries"]) == set(current), "Fingerprint record does not list the inputs just hashed"

    n_file = sum(1 for v in current.values() if v["kind"] == "file")
    n_dir = sum(1 for v in current.values() if v["kind"] == "dir")
    gb = sum(v["bytes"] for v in current.values()) / 1e9
    print(f"\n  Fingerprints {status}: {n_file} files + {n_dir} folder digest(s), {gb:.2f} GB")
    print(f"  Record: {fp_path}")

    P6 = {
        "dirs": DIRS,
        "out": OUT,
        "settings": SETTINGS,
        "expected": EXPECTED,
        "registry": {e["name"]: e["path"] for e in registry},
        "fingerprints_path": fp_path,
        "fingerprint_status": status,
        "module04_available": not missing_m4,
        "input_checks_passed": False,   # set by check_inputs() in cell 01_02
    }
    print("  Module 04 inputs present:", P6["module04_available"])
    print("\n  Next: run 01_02, then  P6 = check_inputs(P6)")
    return P6


print("Module 01 cell 2 loaded.  Run:  P6 = initialise_phase6()")

Module 01 cell 2 loaded.  Run:  P6 = initialise_phase6()


In [3]:
"""
Phase 6 -- Module 01, cell 3 of 3: structural read-back checks on the frozen inputs.

Requires cells 01_00 and 01_01.  Run:   P6 = check_inputs(P6)

Checks the SHAPE of what Phase 6 will read (columns, row counts, keys, alignment).
It reads no metric into any analysis and computes no AUC. It also prints how the
Phase 2 run configs store the locked per-fold AUC-ROC (needed to confirm the wiring-proof
tolerances before the rules note is hashed). Sets P6["input_checks_passed"].
"""

import json

import numpy as np
import pandas as pd

HOLDOUT_PER_CLASS_COLUMNS = [
    "Model", "Precision mode", "Class", "N", "N predicted", "AUC-ROC", "Sensitivity (recall)",
    "Specificity", "Precision", "F1", "PR-AUC", "Sens @ 95% spec", "Sens @ 90% spec", "Flag",
]
DEMOGRAPHIC_COLUMNS = [
    "Dataset", "Model", "Axis", "Subgroup", "Class", "N positive", "N negative",
    "N positive lesions", "AUC-ROC", "Sensitivity", "Tier", "Status",
]
P3_PRED_KEYS = {"image_ids", "labels", "probs", "best_epoch", "class_names"}
P4_PRED_KEYS = {"image_ids", "labels", "probs", "pred"}
P5_PRED_KEYS = {"isic_ids", "lesion_ids", "groups", "labels", "probs", "pred"}


def check_inputs(P6):
    R, X, S = P6["registry"], P6["expected"], P6["settings"]
    D = {}          # loaded tables, reused between checks
    LOCKED = {}     # locked per-fold AUC-ROC values as stored (for the wiring proof later)
    STATUS = {}     # distinct Status values per demographic file (for Module 10)

    def cv_table():
        cv = pd.read_csv(R["p1_cv_folds"])
        need = {"image_id", "lesion_id", "dx", "localization", "strat_label", "cv_fold"}
        assert need <= set(cv.columns), f"missing columns {sorted(need - set(cv.columns))}"
        D["cv"] = cv
        n_les = int(cv.groupby("lesion_id")["cv_fold"].nunique().max())
        ok = (len(cv) == X["cv_pool_images"] and cv["image_id"].is_unique
              and set(cv["cv_fold"].unique()) == set(range(S["n_folds"])) and n_les == 1)
        return ok, f"{len(cv)} rows, folds {sorted(cv['cv_fold'].unique().tolist())}, max folds per lesion = {n_les}"

    def holdout_table():
        ho = pd.read_csv(R["p1_holdout_split"])
        cv = D["cv"]
        D["ho"] = ho
        img_overlap = len(set(ho["image_id"]) & set(cv["image_id"]))
        les_overlap = len(set(ho["lesion_id"]) & set(cv["lesion_id"]))
        ok = (len(ho) == X["holdout_images"] and ho["image_id"].is_unique
              and ho["lesion_id"].nunique() == X["holdout_lesions"] and img_overlap == 0 and les_overlap == 0)
        return ok, (f"{len(ho)} images, {ho['lesion_id'].nunique()} lesions; "
                    f"overlap with pool: {img_overlap} images, {les_overlap} lesions")

    def p3_predictions():
        problems, totals = [], {}
        cv = D["cv"]
        for m in ("multimodal", "ablation"):
            total = 0
            for k in range(S["n_folds"]):
                z = np.load(R[f"p3_{m}_fold{k}_val_predictions"], allow_pickle=False)
                if not P3_PRED_KEYS <= set(z.files):
                    problems.append(f"{m} fold {k}: keys {sorted(z.files)}")
                    continue
                ids, probs = z["image_ids"], z["probs"]
                want = set(cv.loc[cv["cv_fold"] == k, "image_id"])
                if probs.shape != (len(ids), S["n_classes"]):
                    problems.append(f"{m} fold {k}: probs shape {probs.shape}")
                if len(set(ids.tolist())) != len(ids) or set(ids.tolist()) != want:
                    problems.append(f"{m} fold {k}: image IDs differ from cv_fold_assignments")
                if not np.isfinite(probs).all():
                    problems.append(f"{m} fold {k}: non-finite probabilities")
                total += len(ids)
            totals[m] = total
            if total != X["cv_pool_images"]:
                problems.append(f"{m}: {total} rows across folds, expected {X['cv_pool_images']}")
        return not problems, ("; ".join(problems) if problems else f"10 files, rows per model: {totals}, IDs match folds")

    def p3_run_config():
        cfg = json.loads(R["p3_run_config"].read_text(encoding="utf-8"))
        res = cfg["results"]
        for m in ("multimodal", "ablation"):
            LOCKED[f"p3_{m}"] = [res["per_fold"][m][str(k)]["best_val_auc_roc"] for k in range(S["n_folds"])]
        return bool(res.get("complete")), f"results.complete = {res.get('complete')}; per-fold best_val_auc_roc found for both arms"

    def p2_run_configs():
        out, ok = [], True
        for s in ("script1_efficientnet_b3", "script2_resnet50", "script3_vit_b_16"):
            cfg = json.loads(R[f"p2_{s}_run_config"].read_text(encoding="utf-8"))
            fm = cfg["fold_metrics"]
            vals = [d["auc_roc"] for d in fm]
            LOCKED[f"p2_{s}"] = vals
            ok = ok and len(fm) == S["n_folds"] and all(isinstance(v, float) for v in vals)
            out.append(f"{s}: {len(fm)} folds, fold-0 auc_roc stored as {vals[0]!r}")
        return ok, " | ".join(out)

    def checkpoints():
        if not P6["module04_available"]:
            return False, "Module 04 inputs absent (see the list printed by initialise_phase6)"
        sizes = {n: R[n].stat().st_size for n in R if n.startswith("p2_") and n.endswith("_best")}
        small = [n for n, s in sizes.items() if s < 1_000_000]
        return (len(sizes) == 10 and not small), f"{len(sizes)} checkpoints found, none under 1 MB: {not small} (loaded strictly in Module 04)"

    def p4_predictions():
        ho, problems = D["ho"], []
        for m, key in (("multimodal", "p4_multimodal_holdout_predictions"), ("ablation", "p4_ablation_holdout_predictions")):
            z = np.load(R[key], allow_pickle=False)
            if not P4_PRED_KEYS <= set(z.files):
                problems.append(f"{m}: keys {sorted(z.files)[:8]}...")
                continue
            if z["probs"].shape != (X["holdout_images"], S["n_classes"]) or set(z["image_ids"].tolist()) != set(ho["image_id"]):
                problems.append(f"{m}: shape {z['probs'].shape} or image IDs differ from holdout_split")
        return not problems, ("; ".join(problems) if problems else "both files: 1432 x 7, IDs match holdout_split")

    def p4_tables():
        a = pd.read_csv(R["p4_per_image_scores"], usecols=["image_id", "lesion_id", "model", "layer"])
        b = pd.read_csv(R["p4_claim2_paired_comparison"])
        ok = len(a) == X["per_image_scores_rows"] and len(b) == X["claim2_comparison_rows"]
        return ok, f"per_image_scores {len(a)} rows; claim2_paired_comparison {len(b)} rows"

    def p5_predictions():
        problems, ids = [], {}
        for m in ("multimodal", "ablation"):
            z = np.load(R[f"p5_bcn_predictions_{m}"], allow_pickle=False)
            if not P5_PRED_KEYS <= set(z.files):
                problems.append(f"{m}: keys missing {sorted(P5_PRED_KEYS - set(z.files))}")
                continue
            labels, les = z["labels"], z["lesion_ids"]
            primary = labels >= 0
            n_les = len(set(les[primary].tolist()))
            ids[m] = z["isic_ids"].tolist()
            if (len(labels) != X["bcn_scored_rows"] or int(primary.sum()) != X["bcn_primary_images"]
                    or n_les != X["bcn_primary_lesions"] or z["probs"].shape != (len(labels), S["n_classes"])):
                problems.append(f"{m}: rows {len(labels)}, primary {int(primary.sum())}, primary lesions {n_les}")
        if len(ids) == 2 and ids["multimodal"] != ids["ablation"]:
            problems.append("multimodal and ablation rows are in a different order")
        return not problems, ("; ".join(problems) if problems else "16843 rows, 16284 primary, 4591 primary lesions, same row order in both models")

    def p5_holdout_per_class():
        cols = {m: pd.read_csv(R[f"p5_holdout_per_class_{m}"]).columns.tolist() for m in ("multimodal", "ablation")}
        ok = all(c == HOLDOUT_PER_CLASS_COLUMNS for c in cols.values())
        return ok, "columns as expected in both files" if ok else f"columns differ: {cols}"

    def p5_demographics():
        problems, info = [], []
        for ds, per_model in (("holdout", X["holdout_demographic_cells_per_model"]), ("bcn", X["bcn_demographic_cells_per_model"])):
            for m in ("multimodal", "ablation"):
                df = pd.read_csv(R[f"p5_{ds}_demographics_{m}"])
                if df.columns.tolist() != DEMOGRAPHIC_COLUMNS:
                    problems.append(f"{ds}/{m}: columns {df.columns.tolist()}")
                if len(df) != per_model:
                    problems.append(f"{ds}/{m}: {len(df)} rows, expected {per_model}")
                STATUS[f"{ds}_{m}"] = df["Status"].value_counts().to_dict()
                info.append(f"{ds}/{m}: {STATUS[f'{ds}_{m}']}")
        return not problems, ("; ".join(problems) if problems else "four files, 12 columns, 168 / 112 rows per model. Status counts: " + " | ".join(info))

    def p5_bcn_per_class_info():
        cols = pd.read_csv(R["p5_bcn_per_class_multimodal"]).columns.tolist()
        return True, f"information only, columns: {cols}"

    checks = [
        ("Phase 1 fold table: 8583 rows, folds 0-4, one fold per lesion", cv_table),
        ("Phase 1 holdout: 1432 images / 1067 lesions, no overlap with the pool", holdout_table),
        ("Phase 3 validation predictions: 10 files, IDs match their folds", p3_predictions),
        ("Phase 3 run config: complete, per-fold AUC-ROC present", p3_run_config),
        ("Phase 2 run configs: 5 fold entries each; how AUC-ROC is stored", p2_run_configs),
        ("Phase 2 ResNet50 / ViT-B/16 checkpoints: 10 present", checkpoints),
        ("Phase 4 holdout predictions: 1432 x 7, IDs match holdout_split", p4_predictions),
        ("Phase 4 tables: per_image_scores and Claim 2 comparison", p4_tables),
        ("Phase 5 BCN20000 predictions: counts, lesions, same row order", p5_predictions),
        ("Phase 5 holdout per-class tables: columns", p5_holdout_per_class),
        ("Phase 5 demographic grids: columns, rows, Status values", p5_demographics),
        ("Phase 5 BCN per-class table: columns (information)", p5_bcn_per_class_info),
    ]
    rows = run_checks("PHASE 6 -- MODULE 01: INPUT CHECKS (structure only, no analysis)", checks)

    passed = all(r["passed"] for r in rows)
    record = {"format_version": 1, "recorded_at": now_iso(), "all_passed": passed, "checks": rows,
              "locked_auc_as_stored": LOCKED, "demographic_status_values": STATUS}
    atomic_write_bytes(OUT / "phase6_input_checks.json", json_bytes(record))
    back = json.loads((OUT / "phase6_input_checks.json").read_text(encoding="utf-8"))
    assert back["all_passed"] == passed and len(back["checks"]) == len(checks), "Input-check record read-back failed"

    P6["input_checks_passed"] = passed
    P6["locked_auc"] = LOCKED
    P6["demographic_status_values"] = STATUS
    if passed:
        print("\n  All checks passed. Next: run cell 02_00, then  RULES6 = record_rules_note(P6)")
    else:
        print("\n  STOP: do not run Module 02. Send Claude the FAIL lines above (nothing in Phases 1-5 was changed).")
    return P6


print("Module 01 cell 3 loaded.  Run:  P6 = check_inputs(P6)")

Module 01 cell 3 loaded.  Run:  P6 = check_inputs(P6)


# Module 2

In [10]:
"""
Phase 6 -- Module 02: the rules note, written BEFORE any result exists.

Requires cells 01_00, 01_01, 01_02 (and P6["input_checks_passed"] = True).

  First run :  RULES6 = record_rules_note(P6)
               -> prints the wiring-proof tolerances and stops (nothing written).
  Second run:  RULES6 = record_rules_note(P6, tolerances_confirmed=True)
               -> writes rules/phase6_rules_note.json, a readable .md copy and a SHA-256 sidecar.

The note is write-once. It refuses to record if any Phase 6 result file already exists.
Later modules call  check_rules_note()  to confirm the note has not changed.
Module 12 also checks that every result file is newer than the note.
"""

import json

# Wiring-proof tolerances (absolute difference in per-fold macro AUC-ROC against the locked value).
# The run configs store the locked values at full float precision, so rounding is not the limit;
# these values allow for bf16 and GPU run-to-run noise. A wrong checkpoint or wrong preprocessing
# would miss by far more than either number.
WIRING_TOLERANCE_ABS = {"resnet50": 0.001, "vit_b_16": 0.003}

NOTE_PATH = OUT / "rules" / "phase6_rules_note.json"
MD_PATH = OUT / "rules" / "phase6_rules_note.md"
SIDECAR_PATH = OUT / "rules" / "phase6_rules_note.sha256.json"


def build_rules(P6):
    S = SETTINGS
    return {
        "format_version": 1,
        "phase": 6,
        "title": "Phase 6 rules note: statistical comparison and results assembly",
        "source": "Phase-6_Planning.md decisions P6-1 to P6-12 (Ghani, 2026-10-09)",
        "populations": {
            "verdict_claim1": "Pooled out-of-fold predictions, 8,583 images in the CV pool: every image scored by the model of the fold that held it out, for all four models",
            "holdout_check": "EXPLORATORY: multimodal vs ablation on the 1,432-image / 1,067-lesion holdout, using the Phase 4 fp32 prediction files (fold-3 models)",
            "resnet50_vit": "Out-of-fold only, from a read-only scoring pass (no holdout check)",
            "intervals": "Internal holdout and BCN20000 primary images, from stored predictions (no new inference for the frozen models); BCN20000 SCC images (label -1) belong to no metric",
        },
        "comparisons": [
            {"name": "multimodal vs Phase 3 matched ablation", "role": "VERDICT (Claim 1)"},
            {"name": "multimodal vs ResNet50", "role": "SECONDARY"},
            {"name": "multimodal vs ViT-B/16", "role": "SECONDARY"},
            {"name": "Phase 2 EfficientNet-B3 (B3)", "role": "baseline reference row only; B4 was not run"},
        ],
        "tests": {
            "alpha": S["alpha"],
            "primary_test": "Paired DeLong, one-vs-rest, per class (7 tests per comparison)",
            "macro_auc": "Descriptive headline only; not tested",
            "correction": "Holm step-down across the 7 class p-values of each comparison (not Bonferroni)",
            "holm_family_size": S["holm_family_size"],
            "holm_untestable_class_rule": "The family is always 7. A class that cannot be tested is entered with p = 1.0 and is never dropped from the family",
            "reporting": "The corrected p-value is never reported without delta AUC-ROC (multimodal minus comparator) and its 95% CI in the same row",
        },
        "bootstrap": {
            "unit": "lesion (never image); all images of a sampled lesion come along",
            "pairing": "paired: both models in a comparison are scored on the same resampled lesions",
            "resamples": S["bootstrap_resamples"],
            "seed": S["bootstrap_seed"],
            "interval": f"percentile {S['ci_percent']}%",
            "undefined_resample_rule": "A resample where a metric is undefined is dropped from that metric's percentile calculation and counted; the count is printed beside the interval, and a share above 5% flags the row",
            "undefined_flag_share": S["undefined_resample_flag_share"],
            "macro_precision": "Taken over the classes where it is defined in that resample; the number left out is printed",
            "significance_rule": "A class is named significant only if the Holm-corrected DeLong p-value AND the lesion-bootstrap CI both support it. Disagreements are listed as disagreements, not resolved silently",
        },
        "claim1": {
            "supports": "Holm-corrected p < 0.05 AND delta AUC (multimodal minus ablation) > 0 AND lesion-bootstrap 95% CI excludes 0",
            "opposes": "Holm-corrected p < 0.05 AND delta AUC < 0 AND CI excludes 0 (ablation better)",
            "not_detectable": "Neither of the above; includes any DeLong/bootstrap disagreement (listed as such)",
            "minimum_effect_size": "None required (Methodology 9.1)",
            "reported_as": "Supports in K1 of 7, opposes in K2 of 7, not detectable in K3 of 7, naming the classes, each with delta AUC and CI",
            "categories": {
                "supported": {
                    "condition": "K1 >= 1 and K2 = 0",
                    "sentence": "In this setup, adding age, sex and site via MetaBlock was associated with higher AUC-ROC in [K₁] of 7 classes ([names], ΔAUC [values], 95% CI [values]) and no detectable difference in the rest.",
                },
                "mixed": {
                    "condition": "K1 >= 1 and K2 >= 1",
                    "sentence": "The effect of metadata fusion differed by class: higher in [names] and lower in [names], with no detectable difference in [names] (ΔAUC and CI in Table X).",
                },
                "not_detectable": {
                    "condition": "K1 = 0 and K2 = 0",
                    "sentence": "No class showed a detectable AUC-ROC difference between the multimodal model and the matched image-only ablation in this setup. This does not show metadata cannot help; it shows no benefit was detected with these three fields, this fusion method and this dataset.",
                },
                "opposed": {
                    "condition": "K1 = 0 and K2 >= 1",
                    "sentence": "The image-only ablation scored higher than the multimodal model in [names] (ΔAUC and CI in Table X) and no class favoured the multimodal model. Metadata fusion did not improve classification in this setup.",
                },
            },
            "limits_for_all_four": [
                "Each sentence is followed by the fold-2 sensitivity result (does the category change without that fold?) and the holdout-check result, labelled exploratory",
                "Any reason offered for the result is written as a hypothesis that was not tested; the Phase 5 site-blind run is not a measure of what the metadata is worth and is not offered as the explanation",
                "The sentence says 'in this setup', never 'metadata does not help'",
                "The category names describe the counts; they are not a pass or fail (Methodology 9.1 sets no minimum K)",
            ],
            "fold2_sensitivity_label": "EXPLORATORY: chosen after seeing the Phase 3 results. Same analysis with ablation fold 2's validation images removed from the pool for both models. Shown beside the primary row; never replaces it",
        },
        "status_labels": {
            "verdict": ["Claim 1 on pooled out-of-fold predictions"],
            "exploratory": [
                "holdout check (multimodal vs ablation)",
                "fold-2 sensitivity row",
                "site-blind intervals",
                "BCN20000 demographic-cell intervals",
                "paired multimodal minus ablation difference on each dataset (descriptive, not a test)",
            ],
            "secondary": ["multimodal vs ResNet50 tests", "multimodal vs ViT-B/16 tests"],
        },
        "scoring_pass": {
            "models": ["ResNet50", "ViT-B/16"],
            "source": "fold_{k}_best.pt from Phase 2 - P3, scored on fold k's validation images only; strict load, eval(), no training, no file changed",
            "precision": "bf16 autocast (matches how these models' validation AUC-ROC was recorded)",
            "wiring_proof": "Each fold's recomputed macro AUC-ROC must reproduce the locked per-fold value in the Phase 2 run config",
            "wiring_tolerance_abs_auc": WIRING_TOLERANCE_ABS,
            "tolerance_reason": "ResNet50 tight; ViT-B/16 wider because ViT is not bit-reproducible",
            "on_failure": "Module 04 stops and the fold is investigated. The tolerance is not widened after this note is recorded",
            "if_not_run": "Modules 04 and 08 are cut; the baselines' locked means are reported without tests",
        },
        "intervals": {
            "gets_intervals": [
                "holdout and BCN20000 headline metrics and per-class metrics, both models",
                "generalisation gap, per model, all eight headline metrics (each dataset bootstrapped separately; gap interval from the paired draws' difference)",
                "multimodal minus ablation difference on each dataset (descriptive)",
                "demographic cells with 'reported' status only (22 holdout; 75 BCN20000 exploratory), per model",
            ],
            "gets_no_interval": "Tier 1 low-sample and Tier 2 undefined demographic cells keep their Phase 5 flags exactly",
            "not_estimable_wording": "not estimable at this sample size (AKIEC, DF and VASC subgroups)",
            "gap_anchors": "Taken from Phase 5 bcn_gap.json (verified 2026-10-07): AUROC drop 0.075; 23.2 and 22.3 points. Worded 'closest published figures, not like-for-like'. No pass mark",
        },
        "wording_rules": [
            "No table or sentence uses 'significant' without delta AUC-ROC and its CI in the same row",
            "Claims 3 and 4: numbers first, criterion second; 'met' is never a headline",
            "Claims 1 and 2 are two findings with one cross-reference sentence; no link between them was tested",
            "Any analysis added after seeing results is labelled as such",
        ],
        "priority_and_cut_order": {
            "protected": ["Claim 1 on OOF", "intervals on headline, per-class and gap numbers", "success-criteria check", "literature table"],
            "cut_order": ["1. demographic-cell intervals (Module 10)", "2. holdout check (Module 07)", "3. ResNet50 and ViT-B/16 DeLong tests (Modules 04 and 08)"],
        },
        "inputs": {
            "fingerprint_record": str(P6["fingerprints_path"]),
            "fingerprint_record_sha256": sha256_file(P6["fingerprints_path"]),
        },
    }


def render_markdown(rules, written_at):
    c1 = rules["claim1"]
    lines = [f"# {rules['title']}", "", f"Recorded {written_at}. Source: {rules['source']}.", "",
             "This file is a readable copy. The JSON file is the one that is hashed.", ""]
    lines += ["## Populations"] + [f"- **{k}**: {v}" for k, v in rules["populations"].items()] + [""]
    lines += ["## Comparisons"] + [f"- {c['name']}: {c['role']}" for c in rules["comparisons"]] + [""]
    lines += ["## Tests"] + [f"- **{k}**: {v}" for k, v in rules["tests"].items()] + [""]
    lines += ["## Bootstrap"] + [f"- **{k}**: {v}" for k, v in rules["bootstrap"].items()] + [""]
    lines += ["## Claim 1", f"- **Supports**: {c1['supports']}", f"- **Opposes**: {c1['opposes']}",
              f"- **Not detectable**: {c1['not_detectable']}", f"- **Minimum effect size**: {c1['minimum_effect_size']}", ""]
    lines += ["### Pre-set sentences"]
    for name, cat in c1["categories"].items():
        lines += [f"- **{name}** ({cat['condition']}): {cat['sentence']}"]
    lines += ["", "### Limits for all four"] + [f"- {x}" for x in c1["limits_for_all_four"]]
    lines += [f"- **Fold-2 sensitivity**: {c1['fold2_sensitivity_label']}", ""]
    lines += ["## Verdict / exploratory / secondary"]
    for k, v in rules["status_labels"].items():
        lines += [f"- **{k}**: " + "; ".join(v)]
    lines += ["", "## Scoring pass"] + [f"- **{k}**: {v}" for k, v in rules["scoring_pass"].items()] + [""]
    lines += ["## Intervals"] + [f"- **{k}**: {v}" for k, v in rules["intervals"].items()] + [""]
    lines += ["## Wording rules"] + [f"- {x}" for x in rules["wording_rules"]] + [""]
    lines += ["## Priority and cut order", f"- Protected: {'; '.join(rules['priority_and_cut_order']['protected'])}"]
    lines += [f"- Cut order: {'; '.join(rules['priority_and_cut_order']['cut_order'])}", ""]
    return "\n".join(lines)


def find_result_files():
    """Any file under the Phase 6 folder that is not the rules folder or an exempt root file."""
    found = []
    for p in OUT.rglob("*"):
        if not p.is_file():
            continue
        rel = p.relative_to(OUT).parts
        if rel[0] == "rules" or (len(rel) == 1 and rel[0] in EXEMPT_ROOT_FILES):
            continue
        found.append(p)
    return found


def check_rules_note(P6=None):
    """Call this at the start of every later module. Returns the rules dict or raises."""
    if not NOTE_PATH.exists() or not SIDECAR_PATH.exists():
        raise RuntimeError("Rules note not recorded. Run Module 02 first.")
    side = json.loads(SIDECAR_PATH.read_text(encoding="utf-8"))
    if sha256_file(NOTE_PATH) != side["sha256"]:
        raise RuntimeError("The rules note has changed since it was recorded. Stop and tell Claude.")
    return json.loads(NOTE_PATH.read_text(encoding="utf-8"))["rules"]


def record_rules_note(P6, tolerances_confirmed=False):
    banner("PHASE 6 -- MODULE 02: RULES NOTE (before any result exists)")
    if not P6.get("input_checks_passed"):
        raise RuntimeError("Module 01 input checks have not passed. Run cell 01_02 first.")

    if NOTE_PATH.exists():
        rules = check_rules_note(P6)
        side = json.loads(SIDECAR_PATH.read_text(encoding="utf-8"))
        print(f"  Rules note already recorded and unchanged.\n  Recorded {side['recorded_at']}  SHA-256 {side['sha256']}")
        return rules

    print("  Wiring-proof tolerances (absolute difference in per-fold macro AUC-ROC):")
    for k, v in WIRING_TOLERANCE_ABS.items():
        print(f"    {k:<9} {v}")
    print("  Locked AUC-ROC as stored in the Phase 2 run configs (from cell 01_02):")
    for name in ("p2_script2_resnet50", "p2_script3_vit_b_16"):
        print(f"    {name}: {P6['locked_auc'][name]}")
    if not tolerances_confirmed:
        print("\n  NOTHING WRITTEN. Check that the tolerances above suit the stored precision,")
        print("  then run:  RULES6 = record_rules_note(P6, tolerances_confirmed=True)")
        return None

    found = find_result_files()
    if found:
        for p in found[:20]:
            print(f"  RESULT FILE ALREADY EXISTS: {p}")
        raise RuntimeError("Phase 6 result files already exist. The rules note must come first. Nothing was written.")

    rules = build_rules(P6)
    written_at = now_iso()
    note = {"written_at": written_at, "written_before_any_result": True,
            "result_files_found_at_write": 0, "rules": rules}
    atomic_write_bytes(NOTE_PATH, json_bytes(note))
    atomic_write_bytes(MD_PATH, render_markdown(rules, written_at).encode("utf-8"))
    sidecar = {"file": NOTE_PATH.name, "bytes": NOTE_PATH.stat().st_size, "sha256": sha256_file(NOTE_PATH),
               "md_file": MD_PATH.name, "md_sha256": sha256_file(MD_PATH), "recorded_at": written_at}
    atomic_write_bytes(SIDECAR_PATH, json_bytes(sidecar))

    # Read-back: hash matches, content parses, the four sentences and the tolerances are what was meant
    back = check_rules_note(P6)
    cats = back["claim1"]["categories"]
    assert set(cats) == {"supported", "mixed", "not_detectable", "opposed"}, "Claim 1 categories missing"
    assert back["scoring_pass"]["wiring_tolerance_abs_auc"] == WIRING_TOLERANCE_ABS, "Tolerances differ on read-back"
    assert back["tests"]["holm_family_size"] == 7 and back["bootstrap"]["seed"] == 42, "Core settings differ on read-back"

    print(f"\n  Recorded: {NOTE_PATH}")
    print(f"  Readable copy: {MD_PATH}")
    print(f"  SHA-256: {sidecar['sha256']}")
    print("  Next: Module 03 (statistics library). Every later module starts with check_rules_note().")
    return back


print("Module 02 loaded.  Run:  RULES6 = record_rules_note(P6)")

Module 02 loaded.  Run:  RULES6 = record_rules_note(P6)


In [8]:
P6 = initialise_phase6()


PHASE 6 -- MODULE 01: SETUP, FINGERPRINTS, READ-ONLY GUARD
  Hashing 71 frozen inputs (the ViT and ResNet checkpoints take a little while)...

  Fingerprints matched the first-run record: 70 files + 1 folder digest(s), 2.55 GB
  Record: T:\Thesis - Development\Resources\Artifacts\Phase 6 - P1\phase6_start_fingerprints.json
  Module 04 inputs present: True

  Next: run 01_02, then  P6 = check_inputs(P6)


In [9]:
P6 = check_inputs(P6)


PHASE 6 -- MODULE 01: INPUT CHECKS (structure only, no analysis)
  [PASS] Phase 1 fold table: 8583 rows, folds 0-4, one fold per lesion  --  8583 rows, folds [0, 1, 2, 3, 4], max folds per lesion = 1
  [PASS] Phase 1 holdout: 1432 images / 1067 lesions, no overlap with the pool  --  1432 images, 1067 lesions; overlap with pool: 0 images, 0 lesions
  [PASS] Phase 3 validation predictions: 10 files, IDs match their folds  --  10 files, rows per model: {'multimodal': 8583, 'ablation': 8583}, IDs match folds
  [PASS] Phase 3 run config: complete, per-fold AUC-ROC present  --  results.complete = True; per-fold best_val_auc_roc found for both arms
  [PASS] Phase 2 run configs: 5 fold entries each; how AUC-ROC is stored  --  script1_efficientnet_b3: 5 folds, fold-0 auc_roc stored as 0.8819440747222641 | script2_resnet50: 5 folds, fold-0 auc_roc stored as 0.9637826359570456 | script3_vit_b_16: 5 folds, fold-0 auc_roc stored as 0.9646075539701037
  [PASS] Phase 2 ResNet50 / ViT-B/16 checkpoint

In [11]:
RULES6 = record_rules_note(P6)


PHASE 6 -- MODULE 02: RULES NOTE (before any result exists)
  Wiring-proof tolerances (absolute difference in per-fold macro AUC-ROC):
    resnet50  0.001
    vit_b_16  0.003
  Locked AUC-ROC as stored in the Phase 2 run configs (from cell 01_02):
    p2_script2_resnet50: [0.9637826359570456, 0.961477216374117, 0.9544816947068104, 0.9625398059885676, 0.9616713728005832]
    p2_script3_vit_b_16: [0.9646075539701037, 0.9632192403618764, 0.9543537883634994, 0.9629503517117525, 0.959482087544222]

  NOTHING WRITTEN. Check that the tolerances above suit the stored precision,
  then run:  RULES6 = record_rules_note(P6, tolerances_confirmed=True)


In [12]:
RULES6 = record_rules_note(P6, tolerances_confirmed=True)


PHASE 6 -- MODULE 02: RULES NOTE (before any result exists)
  Wiring-proof tolerances (absolute difference in per-fold macro AUC-ROC):
    resnet50  0.001
    vit_b_16  0.003
  Locked AUC-ROC as stored in the Phase 2 run configs (from cell 01_02):
    p2_script2_resnet50: [0.9637826359570456, 0.961477216374117, 0.9544816947068104, 0.9625398059885676, 0.9616713728005832]
    p2_script3_vit_b_16: [0.9646075539701037, 0.9632192403618764, 0.9543537883634994, 0.9629503517117525, 0.959482087544222]

  Recorded: T:\Thesis - Development\Resources\Artifacts\Phase 6 - P1\rules\phase6_rules_note.json
  Readable copy: T:\Thesis - Development\Resources\Artifacts\Phase 6 - P1\rules\phase6_rules_note.md
  SHA-256: ccb068c346db6e1c6abcfaf4ad283dd2d9edd85b1f67b2791854d93b8dbd4c5e
  Next: Module 03 (statistics library). Every later module starts with check_rules_note().


# Module 3

In [13]:
"""
Phase 6 -- Module 03, cell 1 of 5: constants the Phase 5 metric library needs.

Requires cells 01_00 to 02_00, and P6 from the earlier modules.  Run the cell (no function call).
Stops if the rules note is missing or altered.

The metric library (next cell) is an unchanged copy of Phase 5's, so it needs the same named
constants. They are set here with the Phase 5 values, and the class list is read from the
prediction files (never typed by hand) and checked against the Phase 5 holdout table.
"""

import numpy as np
import pandas as pd

RULES6 = check_rules_note()          # stops here if the rules note is missing or has changed


def _class_names_from_phase3():
    found = {}
    for m in ("multimodal", "ablation"):
        for k in range(SETTINGS["n_folds"]):
            z = np.load(P6["registry"][f"p3_{m}_fold{k}_val_predictions"], allow_pickle=False)
            found[f"{m}/fold{k}"] = tuple(str(c) for c in z["class_names"])
    distinct = set(found.values())
    assert len(distinct) == 1, f"class_names differ between Phase 3 files: {distinct}"
    return list(distinct.pop())


CLASSES = _class_names_from_phase3()
UNDEFINED = "undefined"
ECE_BINS = 15
LOW_SAMPLE_N = 30
CELL_MIN = 30
SPECIFICITY_TARGETS = (0.95, 0.90)
NULL_N_SHUFFLES = 20
NULL_AUC_TOL = 0.02
NULL_BALANCED_ACC_TOL = 0.02
RANDOM_SEED = 42

# ----------------------------------------------------------------------------
# Read-back checks
# ----------------------------------------------------------------------------


def _check_phase5_class_order():
    t = pd.read_csv(P6["registry"]["p5_holdout_per_class_multimodal"])
    names = [c for c in t["Class"].tolist() if c != "MACRO"]
    return names == [c.upper() for c in CLASSES], f"Phase 5 table order {names}"


def _check_phase4_holdout_labels():
    z = np.load(P6["registry"]["p4_multimodal_holdout_predictions"], allow_pickle=False)
    names = [str(c) for c in z["class_names"]]
    return names == CLASSES, f"Phase 4 holdout class_names {names}"


_rows = run_checks("PHASE 6 -- MODULE 03 (1/5): CONSTANTS", [
    ("Class list read from the 10 Phase 3 files; identical across all of them", lambda: (len(CLASSES) == SETTINGS["n_classes"], f"{CLASSES}")),
    ("Same order as the Phase 5 holdout per-class table", _check_phase5_class_order),
    ("Same order as the Phase 4 holdout predictions", _check_phase4_holdout_labels),
    ("Rules note present and unchanged", lambda: (True, f"{len(RULES6)} sections")),
])
assert all(r["passed"] for r in _rows), "Constants cell failed; send Claude the FAIL lines."
print("\n  Constants set. Next: run 03_01 (the Phase 5 metric library, unchanged).")


PHASE 6 -- MODULE 03 (1/5): CONSTANTS
  [PASS] Class list read from the 10 Phase 3 files; identical across all of them  --  ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
  [PASS] Same order as the Phase 5 holdout per-class table  --  Phase 5 table order ['AKIEC', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'VASC']
  [PASS] Same order as the Phase 4 holdout predictions  --  Phase 4 holdout class_names ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
  [PASS] Rules note present and unchanged  --  15 sections

  4 of 4 passed

  Constants set. Next: run 03_01 (the Phase 5 metric library, unchanged).


In [14]:
"""
03_00_metric_library.txt
Phase 5 -- Module 03: Holdout metrics -- The metric library (reusable by Modules 03, 08, 09, 09B)

TITLE: Metric library

TITLE DESCRIPTION
  One set of functions computes every metric Phase 5 reports, for the holdout now and for BCN20000 and the demographic
  tables later, so that the same definition is used everywhere. Every definition is the one fixed in the rules note:
  one-vs-rest per class, argmax decision rule, F1 = 2TP / (2TP + FP + FN), a 0/0 metric is "undefined" and never 0, a
  macro-average is taken over the classes where the metric is defined with the number left out kept beside it, balanced
  accuracy = mean of the per-class recalls, ECE top-label with equal-width bins, Brier = multiclass.

CODE DESCRIPTION
  - compute_metric_set() returns, for one (dataset, model), every metric of the rules note: per-class TP/FP/FN/TN, AUC-ROC,
    recall, specificity, precision, F1, PR-AUC, sensitivity at 95% / 90% specificity (A3), the macro values with the number
    of classes left out, balanced accuracy, accuracy, the confusion matrix and (unweighted only) ECE and Brier. Weights are for A1.
  - metric_table / overall_row / confusion_frames / flatten_metrics / cell_metrics / run_null_test (A4) / atomic_write_csv make
    the tables ("undefined" for 0/0, low-sample flag written by code), apply the demographic cell rule and the null test.
    Defines functions only: nothing is read from disk and nothing is written when this cell runs.

Requires: 02_00 already run (uses CLASSES, ECE_BINS, LOW_SAMPLE_N, CELL_MIN, UNDEFINED, SPECIFICITY_TARGETS, NULL_*).
"""

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score, roc_curve

NAN = float("nan")
SENS_AT_SPEC_KEYS = {t: f"sens_at_spec_{int(round(t * 100))}" for t in SPECIFICITY_TARGETS}
SENS_AT_SPEC_COLUMNS = {t: f"Sens @ {int(round(t * 100))}% spec" for t in SPECIFICITY_TARGETS}
MACRO_FIELDS = ("auc_roc", "recall", "specificity", "precision", "f1", "pr_auc")
CELL_TIER1_STATUS = "low sample -- interpret with caution"
CELL_TIER2_NO_POSITIVES = "undefined -- 0 positive cases"
CELL_TIER2_NO_NEGATIVES = "undefined -- no negative cases"


# ---------------------------------------------------------------------------
# Small helpers
# ---------------------------------------------------------------------------

def is_nan(value):
    return isinstance(value, (float, np.floating)) and bool(np.isnan(value))


def safe_div(numerator, denominator):
    """numerator / denominator, or NaN when the denominator is 0 (a 0/0 metric is undefined, never 0)."""
    return float(numerator) / float(denominator) if denominator > 0 else NAN


def undef(value):
    """NaN -> the string 'undefined'; anything else unchanged (so full precision survives in the CSV files)."""
    return UNDEFINED if is_nan(value) else value


def fmt(value, digits=4):
    """Display text: NaN / None -> 'undefined'; strings unchanged; integers without decimals."""
    if value is None:
        return UNDEFINED
    if isinstance(value, str):
        return value
    if isinstance(value, (bool, np.bool_)):
        return str(bool(value))
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if is_nan(value):
        return UNDEFINED
    return f"{float(value):.{digits}f}"


def show(df, digits=4):
    """A copy of a table with every value formatted for printing."""
    return df.map(lambda value: fmt(value, digits))


def to_float(value):
    """Reads a table value back: the string 'undefined' becomes NaN."""
    if isinstance(value, str):
        return NAN if value.strip().lower() == UNDEFINED else float(value)
    return float(value)


def atomic_write_csv(df, path, index=False):
    atomic_write_bytes(path, df.to_csv(index=index).encode("utf-8"))


# ---------------------------------------------------------------------------
# One-vs-rest building blocks
# ---------------------------------------------------------------------------

def binary_auc(y_bin, scores, sample_weight=None):
    """One-vs-rest AUC-ROC. Undefined (NaN) when there are no positives or no negatives."""
    y_bin = np.asarray(y_bin).astype(np.int64)
    n_pos = int(y_bin.sum())
    if n_pos == 0 or n_pos == len(y_bin):
        return NAN
    return float(roc_auc_score(y_bin, scores, sample_weight=sample_weight))


def binary_ap(y_bin, scores, sample_weight=None):
    """PR-AUC (average precision). Undefined (NaN) when there are no positives."""
    y_bin = np.asarray(y_bin).astype(np.int64)
    if int(y_bin.sum()) == 0:
        return NAN
    return float(average_precision_score(y_bin, scores, sample_weight=sample_weight))


def sensitivity_at_specificity(y_bin, scores, target):
    """A3. The highest sensitivity among the ROC points whose specificity is at least `target` (ties: the highest
    specificity). Read from this dataset's own ROC curve; no threshold is applied or changed anywhere.
    Returns (sensitivity, achieved_specificity); both NaN when there are no positives or no negatives."""
    y_bin = np.asarray(y_bin).astype(np.int64)
    n_pos = int(y_bin.sum())
    if n_pos == 0 or n_pos == len(y_bin):
        return NAN, NAN
    fpr, tpr, _ = roc_curve(y_bin, scores, drop_intermediate=False)
    specificity = 1.0 - fpr
    feasible = np.flatnonzero(specificity >= target - 1e-12)      # always contains the first point (fpr = 0)
    best = tpr[feasible].max()
    candidates = feasible[tpr[feasible] == best]
    pick = candidates[np.argmax(specificity[candidates])]
    return float(tpr[pick]), float(specificity[pick])


def ece_top_label(y_true, probs, n_bins=ECE_BINS):
    """Top-label ECE with equal-width bins on [0, 1] (bins are (low, high], the first also holds 0), plus the bin table."""
    y_true = np.asarray(y_true).astype(np.int64)
    probs = np.asarray(probs, dtype=np.float64)
    confidence = probs.max(axis=1)
    correct = (probs.argmax(axis=1) == y_true).astype(np.float64)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    n = len(y_true)
    ece, mce, rows = 0.0, 0.0, []
    for i in range(n_bins):
        low, high = edges[i], edges[i + 1]
        in_bin = (confidence >= low) & (confidence <= high) if i == 0 else (confidence > low) & (confidence <= high)
        count = int(in_bin.sum())
        if count == 0:
            rows.append({"bin": i, "low": low, "high": high, "n": 0, "mean_confidence": NAN, "accuracy": NAN, "gap": NAN})
            continue
        mean_conf, acc = float(confidence[in_bin].mean()), float(correct[in_bin].mean())
        gap = abs(acc - mean_conf)
        ece += (count / n) * gap
        mce = max(mce, gap)
        rows.append({"bin": i, "low": low, "high": high, "n": count, "mean_confidence": mean_conf, "accuracy": acc, "gap": gap})
    return {"ece": float(ece), "mce": float(mce), "n_bins": int(n_bins), "bins": pd.DataFrame(rows),
            "mean_confidence": float(confidence.mean()), "accuracy": float(correct.mean()),
            "overconfidence": float(confidence.mean() - correct.mean())}


def multiclass_brier(y_true, probs):
    """Mean over images of the squared error of the full probability vector against the one-hot truth."""
    y_true = np.asarray(y_true).astype(np.int64)
    probs = np.asarray(probs, dtype=np.float64)
    onehot = np.zeros_like(probs)
    onehot[np.arange(len(y_true)), y_true] = 1.0
    return float(((probs - onehot) ** 2).sum(axis=1).mean())


# ---------------------------------------------------------------------------
# The full metric set for one (dataset, model)
# ---------------------------------------------------------------------------

def _macro(per_class, field, classes):
    """Mean over the classes where `field` is defined, and how many classes were left out."""
    values = [per_class[c][field] for c in classes if field in per_class[c]]
    defined = [v for v in values if not is_nan(v)]
    return (float(np.mean(defined)) if defined else NAN), len(values) - len(defined)


def compute_metric_set(y_true, probs, classes=None, sample_weight=None, with_calibration=True, with_sens_at_spec=True):
    """Everything Section 2 of the Planning asks for, for one (dataset, model). Decision rule: argmax of `probs`
    (ties go to the lower class index). `sample_weight` is used ONLY by addition A1; with it, the calibration and
    sensitivity-at-specificity parts are skipped and the result says weighted=True."""
    classes = list(classes) if classes is not None else list(CLASSES)
    y_true = np.asarray(y_true).astype(np.int64)
    probs = np.asarray(probs, dtype=np.float64)
    if probs.ndim != 2 or probs.shape[1] != len(classes) or len(y_true) != probs.shape[0]:
        raise ValueError(f"probs must be (n, {len(classes)}) and match y_true; got {probs.shape} and {y_true.shape}")
    if not np.isfinite(probs).all():
        raise ValueError("probs contains NaN or infinity")
    if y_true.min() < 0 or y_true.max() >= len(classes):
        raise ValueError("y_true holds a class index outside the class list")
    weighted = sample_weight is not None
    weights = np.asarray(sample_weight, dtype=np.float64) if weighted else np.ones(len(y_true))
    if weights.shape != y_true.shape or (weights < 0).any() or weights.sum() <= 0:
        raise ValueError("sample_weight must be non-negative, match y_true and have a positive sum")
    pred = probs.argmax(axis=1)

    per_class = {}
    for index, name in enumerate(classes):
        is_pos, is_pred = (y_true == index), (pred == index)
        tp = float(weights[is_pos & is_pred].sum())
        fp = float(weights[~is_pos & is_pred].sum())
        fn = float(weights[is_pos & ~is_pred].sum())
        tn = float(weights[~is_pos & ~is_pred].sum())
        w_arg = weights if weighted else None
        entry = {"n": int(is_pos.sum()), "n_predicted": int(is_pred.sum()), "tp": tp, "fp": fp, "fn": fn, "tn": tn,
                 "recall": safe_div(tp, tp + fn), "specificity": safe_div(tn, tn + fp), "precision": safe_div(tp, tp + fp),
                 "f1": safe_div(2 * tp, 2 * tp + fp + fn),
                 "auc_roc": binary_auc(is_pos, probs[:, index], w_arg), "pr_auc": binary_ap(is_pos, probs[:, index], w_arg)}
        if with_sens_at_spec and not weighted:
            for target in SPECIFICITY_TARGETS:
                sens, achieved = sensitivity_at_specificity(is_pos, probs[:, index], target)
                entry[SENS_AT_SPEC_KEYS[target]] = sens
                entry[SENS_AT_SPEC_KEYS[target] + "_achieved_specificity"] = achieved
        per_class[name] = entry

    macro = {}
    for field in MACRO_FIELDS:
        macro[field], macro[field + "_left_out"] = _macro(per_class, field, classes)
    if with_sens_at_spec and not weighted:
        for target in SPECIFICITY_TARGETS:
            key = SENS_AT_SPEC_KEYS[target]
            macro[key], macro[key + "_left_out"] = _macro(per_class, key, classes)

    confusion = np.zeros((len(classes), len(classes)), dtype=np.float64)
    np.add.at(confusion, (y_true, pred), weights)
    row_sums = confusion.sum(axis=1, keepdims=True)
    with np.errstate(invalid="ignore", divide="ignore"):
        confusion_norm = np.where(row_sums > 0, confusion / np.where(row_sums > 0, row_sums, 1.0), NAN)

    result = {"n_images": int(len(y_true)), "weighted": weighted, "classes": classes, "per_class": per_class, "macro": macro,
              "balanced_accuracy": macro["recall"],              # by definition: the mean of the per-class recalls
              "balanced_accuracy_left_out": macro["recall_left_out"],
              "accuracy": float(weights[pred == y_true].sum() / weights.sum()),
              "confusion_counts": confusion, "confusion_row_normalised": confusion_norm,
              "class_distribution": {c: int((y_true == i).sum()) for i, c in enumerate(classes)},
              "predicted_distribution": {c: int((pred == i).sum()) for i, c in enumerate(classes)}}
    if with_calibration and not weighted:
        calibration = ece_top_label(y_true, probs)
        result["calibration"] = {k: v for k, v in calibration.items() if k != "bins"}
        result["calibration_bins"] = calibration["bins"]
        result["brier"] = multiclass_brier(y_true, probs)
    return result


# ---------------------------------------------------------------------------
# Tables
# ---------------------------------------------------------------------------

def metric_table(ms, precision_label="fp32", low_sample_n=LOW_SAMPLE_N):
    """Per-class table plus a MACRO row. 'undefined' and the low-sample flag are written by code on every row."""
    rows = []
    for name in ms["classes"]:
        e = ms["per_class"][name]
        notes = []
        if e["n"] < low_sample_n:
            notes.append(f"low sample (N < {low_sample_n}) -- interpret with caution")
        if is_nan(e["precision"]):
            notes.append("precision undefined (class never predicted)")
        if is_nan(e["recall"]):
            notes.append("recall undefined (no positive cases)")
        if is_nan(e["auc_roc"]):
            notes.append("AUC-ROC undefined (no positives or no negatives)")
        row = {"Precision mode": precision_label, "Class": name.upper(), "N": e["n"], "N predicted": e["n_predicted"],
               "AUC-ROC": undef(e["auc_roc"]), "Sensitivity (recall)": undef(e["recall"]), "Specificity": undef(e["specificity"]),
               "Precision": undef(e["precision"]), "F1": undef(e["f1"]), "PR-AUC": undef(e["pr_auc"])}
        for target in SPECIFICITY_TARGETS:
            if SENS_AT_SPEC_KEYS[target] in e:
                row[SENS_AT_SPEC_COLUMNS[target]] = undef(e[SENS_AT_SPEC_KEYS[target]])
        row["Flag"] = "; ".join(notes)
        rows.append(row)

    macro = ms["macro"]
    summary = {"Precision mode": precision_label, "Class": "MACRO", "N": ms["n_images"], "N predicted": ms["n_images"],
               "AUC-ROC": undef(macro["auc_roc"]), "Sensitivity (recall)": undef(macro["recall"]),
               "Specificity": undef(macro["specificity"]), "Precision": undef(macro["precision"]), "F1": undef(macro["f1"]),
               "PR-AUC": undef(macro["pr_auc"])}
    for target in SPECIFICITY_TARGETS:
        if SENS_AT_SPEC_KEYS[target] in macro:
            summary[SENS_AT_SPEC_COLUMNS[target]] = undef(macro[SENS_AT_SPEC_KEYS[target]])
    left_out = [f"{label} macro excludes {macro[field + '_left_out']} undefined class(es)"
                for field, label in (("auc_roc", "AUC-ROC"), ("recall", "recall"), ("precision", "precision"), ("f1", "F1"), ("pr_auc", "PR-AUC"))
                if macro[field + "_left_out"] > 0]
    summary["Flag"] = "; ".join(left_out) if left_out else f"all {len(ms['classes'])} classes defined"
    rows.append(summary)
    return pd.DataFrame(rows)


def overall_row(ms, model, precision_label="fp32"):
    """One row of dataset-level numbers; accuracy comes last."""
    macro = ms["macro"]
    row = {"Model": model, "Precision mode": precision_label, "N images": ms["n_images"],
           "Macro AUC-ROC (primary)": undef(macro["auc_roc"]), "Balanced accuracy": undef(ms["balanced_accuracy"]),
           "Macro-F1": undef(macro["f1"]), "Macro precision": undef(macro["precision"]),
           "Macro precision, classes left out": macro["precision_left_out"], "Macro PR-AUC": undef(macro["pr_auc"])}
    if "calibration" in ms:
        row["ECE (top-label, %d bins)" % ms["calibration"]["n_bins"]] = ms["calibration"]["ece"]
        row["Brier (multiclass)"] = ms["brier"]
        row["Mean confidence"] = ms["calibration"]["mean_confidence"]
        row["Overconfidence (confidence - accuracy)"] = ms["calibration"]["overconfidence"]
    row["Accuracy (comparability only)"] = ms["accuracy"]
    return pd.DataFrame([row])


def confusion_frames(ms):
    """(counts, row-normalised) as DataFrames; rows = true class, columns = predicted class."""
    labels = [c.upper() for c in ms["classes"]]
    counts = pd.DataFrame(ms["confusion_counts"], index=labels, columns=labels)
    normalised = pd.DataFrame(ms["confusion_row_normalised"], index=labels, columns=labels)
    if not ms["weighted"]:
        counts = counts.round(0).astype(np.int64)
    counts.index.name = normalised.index.name = "true \\ predicted"
    return counts, normalised


def flatten_metrics(ms):
    """{metric name: float} for gap arithmetic. NaN stays NaN, so a gap against an undefined value is itself undefined."""
    flat = {"macro_auc_roc": ms["macro"]["auc_roc"], "balanced_accuracy": ms["balanced_accuracy"], "macro_f1": ms["macro"]["f1"],
            "macro_precision": ms["macro"]["precision"], "macro_recall": ms["macro"]["recall"],
            "macro_specificity": ms["macro"]["specificity"], "macro_pr_auc": ms["macro"]["pr_auc"], "accuracy": ms["accuracy"]}
    if "calibration" in ms:
        flat.update({"ece": ms["calibration"]["ece"], "brier": ms["brier"], "mean_confidence": ms["calibration"]["mean_confidence"]})
    for name, e in ms["per_class"].items():
        for field in ("auc_roc", "recall", "specificity", "precision", "f1", "pr_auc"):
            flat[f"{field}_{name}"] = e[field]
        for target in SPECIFICITY_TARGETS:
            if SENS_AT_SPEC_KEYS[target] in e:
                flat[f"{SENS_AT_SPEC_KEYS[target]}_{name}"] = e[SENS_AT_SPEC_KEYS[target]]
    return flat


# ---------------------------------------------------------------------------
# One demographic cell (used by Module 09)
# ---------------------------------------------------------------------------

def cell_metrics(y_bin, scores, pred_bin):
    """AUC-ROC and sensitivity of ONE class in ONE subgroup, with its tier. Rules from the rules note:
    0 positives -> Tier 2 'undefined -- 0 positive cases' (AUC and sensitivity undefined);
    positives but 0 negatives -> Tier 2 'undefined -- no negative cases' (AUC undefined, sensitivity reported);
    otherwise min(n_pos, n_neg) < CELL_MIN -> Tier 1 'low sample -- interpret with caution'; else reported."""
    y_bin = np.asarray(y_bin).astype(np.int64)
    pred_bin = np.asarray(pred_bin).astype(np.int64)
    n_pos = int(y_bin.sum())
    n_neg = int(len(y_bin) - n_pos)
    if n_pos == 0:
        return {"n_pos": n_pos, "n_neg": n_neg, "auc_roc": NAN, "sensitivity": NAN, "tier": "Tier 2", "status": CELL_TIER2_NO_POSITIVES}
    tp = int(((y_bin == 1) & (pred_bin == 1)).sum())
    sensitivity = tp / n_pos
    if n_neg == 0:
        return {"n_pos": n_pos, "n_neg": n_neg, "auc_roc": NAN, "sensitivity": sensitivity, "tier": "Tier 2", "status": CELL_TIER2_NO_NEGATIVES}
    auc = binary_auc(y_bin, scores)
    if min(n_pos, n_neg) < CELL_MIN:
        return {"n_pos": n_pos, "n_neg": n_neg, "auc_roc": auc, "sensitivity": sensitivity, "tier": "Tier 1", "status": CELL_TIER1_STATUS}
    return {"n_pos": n_pos, "n_neg": n_neg, "auc_roc": auc, "sensitivity": sensitivity, "tier": "", "status": "reported"}


# ---------------------------------------------------------------------------
# A4: metric null test
# ---------------------------------------------------------------------------

def run_null_test(y_true, probs, classes=None, n_shuffles=NULL_N_SHUFFLES, seed=RANDOM_SEED,
                  auc_tol=NULL_AUC_TOL, ba_tol=NULL_BALANCED_ACC_TOL):
    """Shuffle the true labels against the real probabilities `n_shuffles` times (seed 42). The metric code must then give
    chance: mean macro AUC-ROC near 0.5 and mean balanced accuracy near 1/n_classes, within the tolerances of the rules note."""
    classes = list(classes) if classes is not None else list(CLASSES)
    y_true = np.asarray(y_true).astype(np.int64)
    rng = np.random.default_rng(seed)
    aucs, bas = [], []
    for _ in range(n_shuffles):
        ms = compute_metric_set(rng.permutation(y_true), probs, classes, with_calibration=False, with_sens_at_spec=False)
        aucs.append(ms["macro"]["auc_roc"])
        bas.append(ms["balanced_accuracy"])
    chance = 1.0 / len(classes)
    mean_auc, mean_ba = float(np.mean(aucs)), float(np.mean(bas))
    auc_ok = (not np.isnan(mean_auc)) and abs(mean_auc - 0.5) < auc_tol
    ba_ok = (not np.isnan(mean_ba)) and abs(mean_ba - chance) < ba_tol
    return {"n_images": int(len(y_true)), "n_shuffles": int(n_shuffles), "seed": int(seed),
            "macro_auc_roc": {"mean": mean_auc, "sd": float(np.std(aucs)), "min": float(np.min(aucs)), "max": float(np.max(aucs)),
                              "expected": 0.5, "tolerance": auc_tol, "passed": bool(auc_ok)},
            "balanced_accuracy": {"mean": mean_ba, "sd": float(np.std(bas)), "min": float(np.min(bas)), "max": float(np.max(bas)),
                                  "expected": chance, "tolerance": ba_tol, "passed": bool(ba_ok)},
            "passed": bool(auc_ok and ba_ok)}


print("Metric library loaded.")
print(f"  definitions : one-vs-rest, argmax, F1 = 2TP/(2TP+FP+FN), 0/0 -> '{UNDEFINED}' (never 0), ECE {ECE_BINS} bins, low-sample N < {LOW_SAMPLE_N}, cell rule min(n_pos, n_neg) < {CELL_MIN}")
print(f"  A3 targets  : sensitivity at {[int(round(t * 100)) for t in SPECIFICITY_TARGETS]}% specificity (ROC read only)")
print(f"  A4          : {NULL_N_SHUFFLES} shuffles, tolerances {NULL_AUC_TOL} (AUC) and {NULL_BALANCED_ACC_TOL} (balanced accuracy)")

Metric library loaded.
  definitions : one-vs-rest, argmax, F1 = 2TP/(2TP+FP+FN), 0/0 -> 'undefined' (never 0), ECE 15 bins, low-sample N < 30, cell rule min(n_pos, n_neg) < 30
  A3 targets  : sensitivity at [95, 90]% specificity (ROC read only)
  A4          : 20 shuffles, tolerances 0.02 (AUC) and 0.02 (balanced accuracy)


In [15]:
"""
Phase 6 -- Module 03, cell 3 of 5: the statistics library (functions only; nothing is read or written).

Requires 03_00 and 03_01.

  delong_paired(y_bin, score_a, score_b)   paired DeLong test for two AUCs on the same cases (fast Sun & Xu form)
  holm_adjust(pvalues, family_size)        Holm step-down; a missing p counts as 1.0 (P6-14)
  percentile_interval(values, ...)         percentile interval; undefined resamples dropped, counted and flagged
  LesionBootstrap(lesion_ids, B, seed)     lesion-level resampling: every image of a drawn lesion comes along
  read_class(...)                          the per-class reading, written by code
  compare(y_true, scores_a, scores_b, boot, name_a, name_b)   the ONE function for every paired comparison

Sign convention everywhere: difference = A minus B.
"""

import numpy as np
import pandas as pd
from scipy.stats import norm, rankdata

NAN = float("nan")


def _auc(y, s):
    """Rank-based AUC for a boolean y; NaN if either side is empty."""
    m = int(y.sum())
    n = len(y) - m
    if m == 0 or n == 0:
        return NAN
    return (rankdata(s)[y].sum() - m * (m + 1) / 2.0) / (m * n)


def delong_paired(y_bin, score_a, score_b):
    """Paired DeLong test, two-sided, for AUC(A) - AUC(B) on the same cases."""
    y = np.asarray(y_bin).astype(bool)
    m, n = int(y.sum()), int((~y).sum())
    out = {"defined": False, "n_pos": m, "n_neg": n, "auc_a": NAN, "auc_b": NAN, "delta": NAN,
           "var_delta": NAN, "se": NAN, "z": NAN, "p": NAN}
    if m < 2 or n < 2:
        return out
    aucs, v10, v01 = [], [], []
    for s in (score_a, score_b):
        s = np.asarray(s, dtype=np.float64)
        overall, pos, neg = rankdata(s), rankdata(s[y]), rankdata(s[~y])
        aucs.append((overall[y].sum() - m * (m + 1) / 2.0) / (m * n))
        v10.append((overall[y] - pos) / n)               # per positive: share of negatives it beats
        v01.append(1.0 - (overall[~y] - neg) / m)        # per negative: share of positives that beat it
    cov = np.cov(np.array(v10)) / m + np.cov(np.array(v01)) / n
    var = float(cov[0, 0] + cov[1, 1] - 2.0 * cov[0, 1])
    delta = float(aucs[0] - aucs[1])
    out.update(defined=True, auc_a=float(aucs[0]), auc_b=float(aucs[1]), delta=delta, var_delta=var)
    if var <= 1e-18:                                      # the two score sets rank the cases the same way
        out.update(se=0.0, z=0.0 if abs(delta) < 1e-12 else float(np.sign(delta) * np.inf),
                   p=1.0 if abs(delta) < 1e-12 else 0.0)
    else:
        z = delta / np.sqrt(var)
        out.update(se=float(np.sqrt(var)), z=float(z), p=float(2.0 * norm.sf(abs(z))))
    return out


def holm_adjust(pvalues, family_size):
    """Holm step-down adjusted p-values, in the order given. The family size is fixed, never 'whatever was testable'."""
    p = np.array([1.0 if (v is None or not np.isfinite(v)) else float(v) for v in pvalues])
    if len(p) != family_size:
        raise ValueError(f"Holm family is {family_size}, got {len(p)} p-values")
    order = np.argsort(p, kind="stable")
    adjusted = np.empty(len(p))
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (family_size - rank) * p[i])
        adjusted[i] = min(1.0, running)
    return adjusted


def percentile_interval(values, level=None, flag_share=None):
    """Percentile interval of a bootstrap column. NaN resamples are dropped, counted and, above the flag share, flagged."""
    level = SETTINGS["ci_percent"] if level is None else level
    flag_share = SETTINGS["undefined_resample_flag_share"] if flag_share is None else flag_share
    v = np.asarray(values, dtype=np.float64)
    valid = v[np.isfinite(v)]
    share = (len(v) - len(valid)) / len(v) if len(v) else NAN
    tail = (100.0 - level) / 2.0
    lo, hi = (np.percentile(valid, [tail, 100.0 - tail]) if len(valid) else (NAN, NAN))
    return {"lo": float(lo), "hi": float(hi), "n_valid": int(len(valid)), "n_undefined": int(len(v) - len(valid)),
            "undefined_share": float(share), "flagged": bool(share > flag_share)}


class LesionBootstrap:
    """Resamples LESIONS with replacement; each drawn lesion brings all of its images (siblings are never split)."""

    def __init__(self, lesion_ids, n_resamples=None, seed=None):
        self.B = SETTINGS["bootstrap_resamples"] if n_resamples is None else int(n_resamples)
        self.seed = SETTINGS["bootstrap_seed"] if seed is None else int(seed)
        _, self.codes = np.unique(np.asarray(lesion_ids).astype(str), return_inverse=True)
        self.n_images, self.n_lesions = len(self.codes), int(self.codes.max()) + 1
        draws = np.random.default_rng(self.seed).integers(0, self.n_lesions, size=(self.B, self.n_lesions))
        self.counts = np.stack([np.bincount(d, minlength=self.n_lesions) for d in draws]).astype(np.int32)

    def indices(self, b):
        """Image positions for resample b (an image appears as many times as its lesion was drawn)."""
        return np.repeat(np.arange(self.n_images), self.counts[b][self.codes])

    def run(self, fn):
        """fn(indices) -> 1-D array. Returns an array (B, k)."""
        return np.array([fn(self.indices(b)) for b in range(self.B)], dtype=np.float64)


def read_class(defined, p_holm, delta, ci_lo, ci_hi, name_a, name_b, alpha=None):
    """The per-class reading. Supports needs BOTH tests to agree on a direction (rules note, Claim 1)."""
    alpha = SETTINGS["alpha"] if alpha is None else alpha
    if not defined:
        return "not testable"
    delong_sig = bool(p_holm < alpha)
    boot_sig = bool(np.isfinite(ci_lo) and np.isfinite(ci_hi) and (ci_lo > 0 or ci_hi < 0))
    if delong_sig and boot_sig:
        if (delta > 0) == (ci_lo > 0):
            return f"{name_a} higher" if delta > 0 else f"{name_b} higher"
        return "disagreement: direction differs"
    if delong_sig:
        return "disagreement: DeLong only"
    if boot_sig:
        return "disagreement: bootstrap only"
    return "not detectable"


def compare(y_true, scores_a, scores_b, boot, name_a, name_b, classes=None, alpha=None):
    """Per-class one-vs-rest paired comparison of A and B on the same images.
    Returns {'table': DataFrame (one row per class + a descriptive MACRO row), 'boot_deltas': (B, K+1) array}."""
    classes = list(CLASSES if classes is None else classes)
    K = len(classes)
    y_true = np.asarray(y_true)
    A, Bm = np.asarray(scores_a, dtype=np.float64), np.asarray(scores_b, dtype=np.float64)
    if not (A.shape == Bm.shape == (len(y_true), K)) or boot.n_images != len(y_true):
        raise ValueError("scores, labels and the bootstrap must describe the same images and classes")
    masks = [y_true == k for k in range(K)]

    tests = [delong_paired(masks[k], A[:, k], Bm[:, k]) for k in range(K)]
    p_holm = holm_adjust([t["p"] if t["defined"] else NAN for t in tests], K)

    def deltas(idx):
        d = [_auc(masks[k][idx], A[idx, k]) - _auc(masks[k][idx], Bm[idx, k]) for k in range(K)]
        return np.array(d + [np.nanmean(d) if np.isfinite(d).any() else NAN])

    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        boot_deltas = boot.run(deltas)

    rows = []
    for k, name in enumerate(classes):
        t, ci = tests[k], percentile_interval(boot_deltas[:, k])
        reading = read_class(t["defined"], p_holm[k], t["delta"], ci["lo"], ci["hi"], name_a, name_b, alpha)
        rows.append({"Class": name.upper(), "N positive": t["n_pos"], "N negative": t["n_neg"],
                     f"AUC {name_a}": t["auc_a"], f"AUC {name_b}": t["auc_b"], "Delta (A - B)": t["delta"],
                     "SE": t["se"], "z": t["z"], "p (raw)": t["p"] if t["defined"] else NAN, "p (Holm)": float(p_holm[k]),
                     "CI low": ci["lo"], "CI high": ci["hi"], "Undefined resamples": ci["n_undefined"],
                     "Undefined share": ci["undefined_share"], "Flag": "undefined share above 5%" if ci["flagged"] else "",
                     "Reading": reading})
    ok = [t for t in tests if t["defined"]]
    ci = percentile_interval(boot_deltas[:, K])
    rows.append({"Class": "MACRO (descriptive)", "N positive": len(y_true), "N negative": NAN,
                 f"AUC {name_a}": float(np.mean([t["auc_a"] for t in ok])) if ok else NAN,
                 f"AUC {name_b}": float(np.mean([t["auc_b"] for t in ok])) if ok else NAN,
                 "Delta (A - B)": float(np.mean([t["delta"] for t in ok])) if ok else NAN,
                 "SE": NAN, "z": NAN, "p (raw)": NAN, "p (Holm)": NAN, "CI low": ci["lo"], "CI high": ci["hi"],
                 "Undefined resamples": ci["n_undefined"], "Undefined share": ci["undefined_share"],
                 "Flag": "undefined share above 5%" if ci["flagged"] else "", "Reading": "descriptive only (no test)"})
    return {"table": pd.DataFrame(rows), "boot_deltas": boot_deltas}


print("Module 03 cell 3 loaded: delong_paired, holm_adjust, percentile_interval, LesionBootstrap, read_class, compare.")
print("  Next: run 03_03 (known-answer tests).")

Module 03 cell 3 loaded: delong_paired, holm_adjust, percentile_interval, LesionBootstrap, read_class, compare.
  Next: run 03_03 (known-answer tests).


In [16]:
"""
Phase 6 -- Module 03, cell 4 of 5: known-answer tests for the statistics library.

Requires 03_00 to 03_02.  Run the cell (no function call).  Starts with check_rules_note().
Every expected value is worked out by hand, by a slow brute-force route, or by scikit-learn;
none is read from the library itself. The last two checks reproduce Phase 5's holdout tables with
the reused metric library (they print "matches" and the largest difference, never a metric value).
Result is kept in M3_KAT_ROWS and used by cell 03_04.
"""

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

check_rules_note()


def _brute(y, a, b):
    """Slow, textbook DeLong: all positive-negative pairs, row and column means, ordinary covariance."""
    y = np.asarray(y).astype(bool)
    m, n = int(y.sum()), int((~y).sum())
    psi = []
    for s in (a, b):
        P, N = s[y], s[~y]
        psi.append((P[:, None] > N[None, :]) + 0.5 * (P[:, None] == N[None, :]))
    cov = np.cov(np.array([p.mean(axis=1) for p in psi])) / m + np.cov(np.array([p.mean(axis=0) for p in psi])) / n
    return psi[0].mean(), psi[1].mean(), cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]


def t_hand_auc():
    y = [1, 1, 1, 0, 0, 0]
    r = delong_paired(y, [.9, .8, .4, .5, .3, .1], [.6, .5, .2, .5, .4, .1])
    ok = abs(r["auc_a"] - 8 / 9) < 1e-12 and abs(r["auc_b"] - 6.5 / 9) < 1e-12 and abs(r["delta"] - 1.5 / 9) < 1e-12
    return ok, f"AUC A {r['auc_a']:.6f} (hand 8/9), AUC B {r['auc_b']:.6f} (hand 6.5/9, with one tie)"


def t_brute_force():
    rng, worst_var, worst_auc = np.random.default_rng(7), 0.0, 0.0
    for _ in range(25):
        n = int(rng.integers(40, 120))
        y = (rng.random(n) < 0.35).astype(int)
        a = np.round(rng.normal(size=n) + 1.0 * y, 1)                  # rounding creates ties on purpose
        b = np.round(a + rng.normal(scale=0.8, size=n), 1)
        r = delong_paired(y, a, b)
        auc_a, auc_b, var = _brute(y, a, b)
        worst_var = max(worst_var, abs(r["var_delta"] - var))
        worst_auc = max(worst_auc, abs(r["auc_a"] - roc_auc_score(y, a)), abs(r["auc_b"] - roc_auc_score(y, b)),
                        abs(r["auc_a"] - auc_a))
    return worst_var < 1e-12 and worst_auc < 1e-12, f"25 random sets with ties; largest variance difference {worst_var:.1e}, AUC difference {worst_auc:.1e}"


def t_identical():
    rng = np.random.default_rng(3)
    y, s = (rng.random(200) < 0.4).astype(int), rng.normal(size=200)
    r = delong_paired(y, s, s.copy())
    return r["p"] == 1.0 and r["delta"] == 0.0, f"p = {r['p']}, delta = {r['delta']}"


def t_perfect():
    rng = np.random.default_rng(5)
    y = (rng.random(150) < 0.4).astype(int)
    noisy = rng.normal(size=150)
    r1 = delong_paired(y, y + 0.0, noisy)
    r2 = delong_paired(y, y + 0.0, y * 2.0)
    ok = r1["auc_a"] == 1.0 and abs(r1["delta"] - (1 - roc_auc_score(y, noisy))) < 1e-12 and r1["p"] < 1e-6 and r2["p"] == 1.0
    return ok, f"perfect vs noise: AUC 1.0, p = {r1['p']:.1e}; two perfect models: p = {r2['p']}"


def t_undefined():
    none = delong_paired([0, 0, 0, 0], [.1, .2, .3, .4], [.4, .3, .2, .1])
    one = delong_paired([1, 0, 0, 0, 0], [.9, .2, .3, .4, .1], [.8, .3, .2, .1, .4])
    return (not none["defined"]) and (not one["defined"]) and np.isnan(none["p"]) and np.isnan(one["p"]), \
        f"no positives -> defined {none['defined']}; one positive -> defined {one['defined']}"


def t_holm():
    got = holm_adjust([0.01, 0.04, 0.03, 0.005], 4)
    bonf = np.minimum(1, 4 * np.array([0.01, 0.04, 0.03, 0.005]))
    seven = holm_adjust([0.01] * 7, 7)
    nan_case = holm_adjust([np.nan, 0.001, 0.2, 0.9, 0.9, 0.9, 0.9], 7)
    ok = (np.allclose(got, [0.03, 0.06, 0.06, 0.02]) and not np.allclose(got, bonf)
          and np.allclose(seven, 0.07) and nan_case[0] == 1.0 and abs(nan_case[1] - 0.007) < 1e-12)
    note = ""
    try:
        from statsmodels.stats.multitest import multipletests
        rng = np.random.default_rng(11)
        worst = 0.0
        for _ in range(20):
            p = rng.random(7) ** 3
            worst = max(worst, np.abs(holm_adjust(p, 7) - multipletests(p, method="holm")[1]).max())
        ok = ok and worst < 1e-12
        note = f"; statsmodels agrees (largest difference {worst:.1e})"
    except ImportError:
        note = "; statsmodels not installed (cross-check skipped)"
    return ok, f"hand example [0.03, 0.06, 0.06, 0.02]; 7 x 0.01 -> 0.07; missing p -> 1.0" + note


def t_interval():
    r = percentile_interval(np.arange(1, 1001, dtype=float))
    flag = lambda k: percentile_interval(np.r_[np.full(k, np.nan), np.arange(100 - k, dtype=float)])
    f10, f4, f5 = flag(10), flag(4), flag(5)
    ok = (abs(r["lo"] - 25.975) < 1e-9 and abs(r["hi"] - 975.025) < 1e-9 and f10["flagged"] and not f4["flagged"]
          and not f5["flagged"] and f10["n_undefined"] == 10)
    return ok, f"1..1000 -> [{r['lo']:.3f}, {r['hi']:.3f}] (hand 25.975, 975.025); 10% undefined flagged, 4% and exactly 5% not"


def t_bootstrap():
    ids = np.repeat([f"L{i}" for i in range(100)], 3)
    b1, b2, b3 = LesionBootstrap(ids, 1000, 42), LesionBootstrap(ids, 1000, 42), LesionBootstrap(ids, 1000, 43)
    sums_ok = bool((b1.counts.sum(axis=1) == 100).all())
    sib_ok = True
    for b in (0, 1, 2):                    # every image appears exactly as often as its lesion was drawn
        per_image = np.bincount(b1.indices(b), minlength=len(ids))
        sib_ok &= bool((per_image == b1.counts[b][b1.codes]).all())
    same = bool((b1.counts == b2.counts).all()) and not bool((b1.counts == b3.counts).all())
    distinct = float((b1.counts > 0).sum(axis=1).mean())
    ok = sums_ok and sib_ok and same and abs(distinct - 63.4) < 1.0
    return ok, f"row sums = lesions: {sums_ok}; siblings move together: {sib_ok}; seed 42 repeats, 43 differs: {same}; mean distinct lesions {distinct:.1f} (theory 63.4)"


def t_compare_synthetic():
    rng = np.random.default_rng(21)
    n_les, K = 700, len(CLASSES)
    lab = rng.integers(0, K, n_les)
    y = np.repeat(lab, 2)
    ids = np.repeat([f"S{i}" for i in range(n_les)], 2)
    A = rng.normal(size=(len(y), K))
    A[np.arange(len(y)), y] += 1.0
    B = A.copy()
    B[y == 0, 0] += 1.5                                             # B is truly better for class 0 only
    boot = LesionBootstrap(ids, 200, 42)
    t = compare(y, A, B, boot, "A", "B")["table"]
    reads = t["Reading"].tolist()
    ok = reads[0] == "B higher" and reads[1:K] == ["not detectable"] * (K - 1) and t["Delta (A - B)"].iloc[0] < 0 \
        and t["Reading"].iloc[K] == "descriptive only (no test)"
    return ok, f"class 0: '{reads[0]}' (delta {t['Delta (A - B)'].iloc[0]:.3f}); other classes: {sorted(set(reads[1:K]))}"


def _reproduce_per_class(model):
    z = np.load(P6["registry"][f"p4_{model}_holdout_predictions"], allow_pickle=False)
    mine = metric_table(compute_metric_set(z["labels"], z["probs"]))
    theirs = pd.read_csv(P6["registry"][f"p5_holdout_per_class_{model}"])
    assert mine["Class"].tolist() == theirs["Class"].tolist(), "class rows differ"
    worst, mismatched = 0.0, []
    for col in mine.columns:
        if col in ("Precision mode", "Class", "Flag"):
            continue
        for a, b in zip(mine[col], theirs[col]):
            x, y = to_float(a), to_float(b)
            if np.isnan(x) and np.isnan(y):
                continue
            d = abs(x - y)
            if np.isnan(d):
                mismatched.append(col)
            else:
                worst = max(worst, d)
    flags = bool((mine["Flag"].fillna("") == theirs["Flag"].fillna("")).all())
    return worst, mismatched, flags


def t_reproduce_per_class():
    out = {m: _reproduce_per_class(m) for m in ("multimodal", "ablation")}
    ok = all(w < 1e-9 and not mm and fl for w, mm, fl in out.values())
    return ok, "; ".join(f"{m}: largest difference {w:.1e}, undefined cells agree {not mm}, flags identical {fl}" for m, (w, mm, fl) in out.items())


def t_reproduce_overall():
    theirs = pd.read_csv(P6["registry"]["p5_holdout_overall"])
    cols = [c for c in theirs.columns if c not in ("Model", "Precision mode")]
    used, notes = set(), []
    for m in ("multimodal", "ablation"):
        z = np.load(P6["registry"][f"p4_{m}_holdout_predictions"], allow_pickle=False)
        mine = overall_row(compute_metric_set(z["labels"], z["probs"]), m).iloc[0]
        def gap(x, y):
            x, y = to_float(x), to_float(y)
            return 0.0 if (np.isnan(x) and np.isnan(y)) else abs(x - y)
        diffs = [max(gap(mine[c], theirs.iloc[i][c]) for c in cols) for i in range(len(theirs))]
        best = int(np.argmin(diffs))
        used.add(best)
        notes.append(f"{m}: closest row {best}, largest difference {diffs[best]:.1e}")
        if not diffs[best] < 1e-9:
            return False, "; ".join(notes)
    return len(used) == 2, "; ".join(notes) + " (rows matched by value, not by label)"


M3_KAT_ROWS = run_checks("PHASE 6 -- MODULE 03 (4/5): KNOWN-ANSWER TESTS", [
    ("AUC worked by hand (8/9 and 6.5/9, one tie)", t_hand_auc),
    ("DeLong variance equals the slow brute-force version; AUC equals scikit-learn", t_brute_force),
    ("Two identical models: p = 1, difference 0", t_identical),
    ("Perfect separation behaves", t_perfect),
    ("No positives / one positive: undefined, never a number", t_undefined),
    ("Holm: hand example, not Bonferroni, 7 x 0.01 -> 0.07, missing p -> 1.0", t_holm),
    ("Percentile interval and the undefined-share flag", t_interval),
    ("Lesion bootstrap: sums, siblings together, seeded, ~63% distinct", t_bootstrap),
    ("compare() on built data: only class 0 differs, and it is found", t_compare_synthetic),
    ("Reused metric library reproduces the Phase 5 holdout per-class tables", t_reproduce_per_class),
    ("Reused metric library reproduces the Phase 5 holdout overall table", t_reproduce_overall),
])
if all(r["passed"] for r in M3_KAT_ROWS):
    print("\n  Known-answer tests all passed. Next: run 03_04 (null checks).")
else:
    print("\n  STOP: send Claude the FAIL lines above. Do not run 03_04.")


PHASE 6 -- MODULE 03 (4/5): KNOWN-ANSWER TESTS
  [PASS] AUC worked by hand (8/9 and 6.5/9, one tie)  --  AUC A 0.888889 (hand 8/9), AUC B 0.722222 (hand 6.5/9, with one tie)
  [PASS] DeLong variance equals the slow brute-force version; AUC equals scikit-learn  --  25 random sets with ties; largest variance difference 3.5e-18, AUC difference 1.1e-16
  [PASS] Two identical models: p = 1, difference 0  --  p = 1.0, delta = 0.0
  [PASS] Perfect separation behaves  --  perfect vs noise: AUC 1.0, p = 8.0e-23; two perfect models: p = 1.0
  [PASS] No positives / one positive: undefined, never a number  --  no positives -> defined False; one positive -> defined False
  [PASS] Holm: hand example, not Bonferroni, 7 x 0.01 -> 0.07, missing p -> 1.0  --  hand example [0.03, 0.06, 0.06, 0.02]; 7 x 0.01 -> 0.07; missing p -> 1.0; statsmodels not installed (cross-check skipped)
  [PASS] Percentile interval and the undefined-share flag  --  1..1000 -> [25.975, 975.025] (hand 25.975, 975.025); 10% unde

In [17]:
"""
Phase 6 -- Module 03, cell 5 of 5: null checks, then the Module 03 test record.

Requires 03_00 to 03_03, and the known-answer tests must have passed.  Run the cell (no function call).

What "null" means here: two similar models are compared on labels that have been shuffled between
lesions, so there is no real difference to find. A sound test then finds "significant" differences
about as often as chance (5% of raw p-values), and the intervals contain 0 about 95% of the time.
Thresholds are fixed here, before the run, and are wide enough to tolerate chance.
Writes assembly/module03_test_record.json (a verification record, rewritable).
"""

import json

import numpy as np

check_rules_note()
assert "M3_KAT_ROWS" in globals() and all(r["passed"] for r in M3_KAT_ROWS), "Run 03_03 first; all its checks must pass."

NULL_RULES = {"share_raw_p_below_alpha_max": 0.12, "median_raw_p_min": 0.30,
              "share_ci_containing_zero_min": 0.85, "share_shuffles_any_holm_significant_max": 0.25}
N_SHUF, N_LES, N_BOOT = 20, 700, 200
OBSERVED = {}


def _scores(lab, rng):
    """Model A: logit-like scores that carry the true label. Model B: A plus fresh noise (a similar, different model)."""
    y = np.repeat(lab, 2)
    A = rng.normal(size=(len(y), len(CLASSES)))
    A[np.arange(len(y)), y] += 1.0
    return y, A, A + rng.normal(scale=0.7, size=A.shape)


def n_self_comparison():
    rng = np.random.default_rng(99)
    lab = rng.integers(0, len(CLASSES), N_LES)
    y, A, _ = _scores(lab, rng)
    ids = np.repeat([f"S{i}" for i in range(N_LES)], 2)
    t = compare(y, A, A.copy(), LesionBootstrap(ids, N_BOOT, 42), "A", "A2")["table"].iloc[:len(CLASSES)]
    ok = bool((t["Delta (A - B)"] == 0).all() and (t["p (raw)"] == 1).all() and (t["CI low"] == 0).all()
              and (t["CI high"] == 0).all() and (t["Reading"] == "not detectable").all())
    return ok, "a model against itself: difference 0, p = 1, interval [0, 0], reading 'not detectable' for all 7 classes"


def n_shuffled_labels():
    rng = np.random.default_rng(SETTINGS["bootstrap_seed"])            # seed 42
    ids = np.repeat([f"S{i}" for i in range(N_LES)], 2)
    boot = LesionBootstrap(ids, N_BOOT, SETTINGS["bootstrap_seed"])
    lab0 = rng.integers(0, len(CLASSES), N_LES)
    _, A, B = _scores(lab0, rng)
    raw, covers, any_holm = [], [], 0
    for _ in range(N_SHUF):
        y = np.repeat(rng.permutation(lab0), 2)                        # labels moved between lesions
        t = compare(y, A, B, boot, "A", "B")["table"].iloc[:len(CLASSES)]
        raw += t["p (raw)"].tolist()
        covers += ((t["CI low"] <= 0) & (t["CI high"] >= 0)).tolist()
        any_holm += int((t["p (Holm)"] < SETTINGS["alpha"]).any())
    raw = np.array(raw)
    OBSERVED.update(share_raw_p_below_alpha=float((raw < SETTINGS["alpha"]).mean()), median_raw_p=float(np.median(raw)),
                    share_ci_containing_zero=float(np.mean(covers)), share_shuffles_any_holm_significant=any_holm / N_SHUF,
                    n_p_values=int(len(raw)))
    o, r = OBSERVED, NULL_RULES
    ok = (o["share_raw_p_below_alpha"] <= r["share_raw_p_below_alpha_max"] and o["median_raw_p"] >= r["median_raw_p_min"]
          and o["share_ci_containing_zero"] >= r["share_ci_containing_zero_min"]
          and o["share_shuffles_any_holm_significant"] <= r["share_shuffles_any_holm_significant_max"])
    return ok, (f"{N_SHUF} shuffles x 7 classes: raw p < 0.05 in {o['share_raw_p_below_alpha']:.1%} (limit {r['share_raw_p_below_alpha_max']:.0%}, chance 5%); "
                f"median raw p {o['median_raw_p']:.2f} (min {r['median_raw_p_min']}, chance 0.50); "
                f"intervals containing 0: {o['share_ci_containing_zero']:.1%} (min {r['share_ci_containing_zero_min']:.0%}, chance 95%); "
                f"shuffles with any Holm-significant class: {o['share_shuffles_any_holm_significant']:.0%} (max {r['share_shuffles_any_holm_significant_max']:.0%})")


M3_NULL_ROWS = run_checks("PHASE 6 -- MODULE 03 (5/5): NULL CHECKS", [
    ("A model compared with itself finds nothing", n_self_comparison),
    ("Shuffled labels: false-positive rate, p-value spread, interval coverage", n_shuffled_labels),
])

passed = all(r["passed"] for r in M3_NULL_ROWS)
record = {"format_version": 1, "recorded_at": now_iso(), "all_passed": passed,
          "known_answer_tests": M3_KAT_ROWS, "null_checks": M3_NULL_ROWS,
          "null_rules": NULL_RULES, "null_observed": OBSERVED,
          "null_design": {"shuffles": N_SHUF, "lesions": N_LES, "bootstrap_resamples": N_BOOT, "seed": SETTINGS["bootstrap_seed"]},
          "constants": {"classes": CLASSES, "ece_bins": ECE_BINS, "low_sample_n": LOW_SAMPLE_N,
                        "specificity_targets": list(SPECIFICITY_TARGETS)}}
path = OUT / "assembly" / "module03_test_record.json"
atomic_write_bytes(path, json_bytes(record))
back = json.loads(path.read_text(encoding="utf-8"))
assert back["all_passed"] == passed and len(back["known_answer_tests"]) == len(M3_KAT_ROWS), "Test record read-back failed"
print(f"\n  Test record: {path}")
print("  Module 03 is complete. Next: Module 04 (scoring pass), after the walk-through." if passed
      else "  STOP: send Claude the FAIL lines above.")


PHASE 6 -- MODULE 03 (5/5): NULL CHECKS
  [PASS] A model compared with itself finds nothing  --  a model against itself: difference 0, p = 1, interval [0, 0], reading 'not detectable' for all 7 classes
  [PASS] Shuffled labels: false-positive rate, p-value spread, interval coverage  --  20 shuffles x 7 classes: raw p < 0.05 in 10.0% (limit 12%, chance 5%); median raw p 0.42 (min 0.3, chance 0.50); intervals containing 0: 89.3% (min 85%, chance 95%); shuffles with any Holm-significant class: 10% (max 25%)

  2 of 2 passed

  Test record: T:\Thesis - Development\Resources\Artifacts\Phase 6 - P1\assembly\module03_test_record.json
  Module 03 is complete. Next: Module 04 (scoring pass), after the walk-through.


# Module 4

In [30]:
"""
Phase 6 -- Module 04, cell 1 of 3: definitions for the ResNet50 / ViT-B/16 scoring pass.

Requires cells 01_00 to 03_02 (and P6 from Module 01).  Run the cell (no function call).
Copied from Phase 2 (01_00 dataset, 01_01 metrics, 00_08 model builders) with only these changes:
  - the model builders use weights=None (the checkpoint's own weights replace every ImageNet weight on a
    strict load, so nothing is downloaded and nothing differs);
  - the dataset is validation-only (no augmentation, which Phase 2 never applied to validation either);
  - the image size is checked (300 x 300 as saved) before any resize.
Nothing is trained, loaded or written here. Starts with check_rules_note().
"""

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as tvm
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset

RULES6 = check_rules_note()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = torch.bfloat16
SCORING_BATCH_SIZE = 32                      # Phase 2 used 32 for both scripts
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

# model name -> (builder, resize_to, registry stem, key in P6["locked_auc"], rules-note tolerance key)
SCORING_MODELS = {
    "resnet50": {"resize_to": None, "stem": "p2_script2_resnet50", "tolerance_key": "resnet50"},
    "vit_b_16": {"resize_to": 224, "stem": "p2_script3_vit_b_16", "tolerance_key": "vit_b_16"},
}


def normalize_image(image_bgr):              # Phase 2 / Phase 1 08_06, verbatim
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    return (image_rgb.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD


def build_resnet50(num_classes=7):           # Phase 2 00_08, weights=None (see header)
    model = tvm.resnet50(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def build_vit_b16(num_classes=7):            # Phase 2 00_08, weights=None (see header)
    model = tvm.vit_b_16(weights=None)
    model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    return model


BUILDERS = {"resnet50": build_resnet50, "vit_b_16": build_vit_b16}


class ValidationFoldDataset(Dataset):
    """The validation images of one fold, in cv_fold_assignments.csv order (as in Phase 2's HAM10000FoldDataset)."""

    def __init__(self, cv_table, fold_num, resize_to=None):
        self.df = cv_table[cv_table["cv_fold"] == fold_num].reset_index(drop=True)
        self.resize_to = resize_to
        self.images_dir = P6["registry"]["p1_images_dir"]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = self.images_dir / f"{row['image_id']}.jpg"
        image = cv2.imread(str(path))
        if image is None:
            raise FileNotFoundError(f"Failed to load image: {path}")
        if image.shape[:2] != (300, 300):
            raise ValueError(f"{path.name} is {image.shape[:2]}, expected 300 x 300 as saved in Phase 1")
        if self.resize_to is not None:
            image = cv2.resize(image, (self.resize_to, self.resize_to), interpolation=cv2.INTER_AREA)
        tensor = torch.from_numpy(normalize_image(image).transpose(2, 0, 1)).float()
        return tensor, CLASS_TO_IDX[row["dx"]]


def fold_macro_auc(y_true, probs):           # Phase 2 01_01 compute_fold_metrics, AUC part, verbatim
    return float(roc_auc_score(y_true, probs, multi_class="ovr", average="macro", labels=list(range(7))))


# ----------------------------------------------------------------------------
# Read-back checks (no checkpoint is loaded here)
# ----------------------------------------------------------------------------
def _t_normalise():
    px = np.array([[[0, 128, 255]]], dtype=np.uint8)                       # one BGR pixel
    got = normalize_image(px)[0, 0]
    want = (np.array([255, 128, 0], dtype=np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD
    return bool(np.allclose(got, want, atol=1e-6)), "BGR (0,128,255) -> RGB, /255, ImageNet mean/std as worked by hand"


def _t_dataset():
    cv = pd.read_csv(P6["registry"]["p1_cv_folds"])
    out = []
    for name, spec in SCORING_MODELS.items():
        ds = ValidationFoldDataset(cv, 0, spec["resize_to"])
        x, y = ds[0]
        out.append((name, tuple(x.shape), len(ds), bool((cv[cv["cv_fold"] == 0]["image_id"].values == ds.df["image_id"].values).all())))
    ok = out[0][1] == (3, 300, 300) and out[1][1] == (3, 224, 224) and all(o[3] for o in out)
    return ok, "; ".join(f"{n}: tensor {s}, fold 0 has {c} images in table order" for n, s, c, _ in out)


def _t_builders():
    res = []
    for name, fn in BUILDERS.items():
        m = fn()
        x = torch.zeros(1, 3, 300 if name == "resnet50" else 224, 300 if name == "resnet50" else 224)
        m.eval()
        with torch.no_grad():
            res.append((name, tuple(m(x).shape), sum(p.numel() for p in m.parameters())))
    ok = all(s == (1, 7) for _, s, _ in res)
    return ok, "; ".join(f"{n}: output {s}, {p / 1e6:.1f} M parameters" for n, s, p in res)


def _t_tolerances():
    tol = RULES6["scoring_pass"]["wiring_tolerance_abs_auc"]
    ok = tol == {"resnet50": 0.001, "vit_b_16": 0.003} and all(len(P6["locked_auc"][s["stem"]]) == 5 for s in SCORING_MODELS.values())
    return ok, f"tolerances from the rules note: {tol}; 5 locked values per model found"


_rows = run_checks("PHASE 6 -- MODULE 04 (1/3): SCORING DEFINITIONS", [
    ("A GPU is available (bf16 autocast on CUDA, as in Phase 2)", lambda: (DEVICE.type == "cuda", f"device = {DEVICE}")),
    ("Normalisation works as in Phase 2", _t_normalise),
    ("Validation dataset: tensor shapes, table order, image path", _t_dataset),
    ("Model builders give 7 outputs", _t_builders),
    ("Tolerances and locked values are in place", _t_tolerances),
])
assert all(r["passed"] for r in _rows), "Definitions cell failed; send Claude the FAIL lines."
print("\n  Next: run 04_01, then  check_checkpoints(P6)")


PHASE 6 -- MODULE 04 (1/3): SCORING DEFINITIONS
  [PASS] A GPU is available (bf16 autocast on CUDA, as in Phase 2)  --  device = cuda
  [PASS] Normalisation works as in Phase 2  --  BGR (0,128,255) -> RGB, /255, ImageNet mean/std as worked by hand
  [PASS] Validation dataset: tensor shapes, table order, image path  --  resnet50: tensor (3, 300, 300), fold 0 has 1717 images in table order; vit_b_16: tensor (3, 224, 224), fold 0 has 1717 images in table order
  [PASS] Model builders give 7 outputs  --  resnet50: output (1, 7), 23.5 M parameters; vit_b_16: output (1, 7), 85.8 M parameters
  [PASS] Tolerances and locked values are in place  --  tolerances from the rules note: {'resnet50': 0.001, 'vit_b_16': 0.003}; 5 locked values per model found

  5 of 5 passed

  Next: run 04_01, then  check_checkpoints(P6)


In [31]:
"""
Phase 6 -- Module 04, cell 2 of 3: strict-load check of the 10 checkpoints (no scoring, nothing written).

Requires 04_00.  Just run the cell: it checks the 10 checkpoints straight away.

Loads each fold_k_best.pt on the CPU and fits it into a freshly built model with strict=True, so a wrong
layer name or shape stops here, before the long scoring run. Also confirms each file is the one fingerprinted
in Module 01 (same size) and that the file holds only the key 'model_state_dict', as Phase 2 saved it.
"""

import json


def check_checkpoints(P6):
    check_rules_note()
    fp = json.loads(P6["fingerprints_path"].read_text(encoding="utf-8"))["entries"]
    checks = []
    for name, spec in SCORING_MODELS.items():
        for k in range(SETTINGS["n_folds"]):
            key = f"{spec['stem']}_fold{k}_best"

            def one(name=name, k=k, key=key):
                path = P6["registry"][key]
                obj = torch.load(path, map_location="cpu", weights_only=True)
                if set(obj) != {"model_state_dict"}:
                    return False, f"keys {sorted(obj)}"
                model = BUILDERS[name]()
                model.load_state_dict(obj["model_state_dict"], strict=True)      # raises on any mismatch
                same_size = path.stat().st_size == fp[key]["bytes"]
                return same_size, f"strict load OK, {path.stat().st_size / 1e6:.0f} MB, size equals the Module 01 record: {same_size}"
            checks.append((f"{name} fold {k}", one))
    rows = run_checks("PHASE 6 -- MODULE 04 (2/3): CHECKPOINTS (strict load, nothing scored)", checks)
    if all(r["passed"] for r in rows):
        print("\n  All 10 checkpoints fit. Next: run 04_02 (it starts the scoring pass by itself)")
    else:
        print("\n  STOP: send Claude the FAIL lines.")
    return all(r["passed"] for r in rows)


CHECKPOINTS_OK = check_checkpoints(P6)


PHASE 6 -- MODULE 04 (2/3): CHECKPOINTS (strict load, nothing scored)
  [PASS] resnet50 fold 0  --  strict load OK, 94 MB, size equals the Module 01 record: True
  [PASS] resnet50 fold 1  --  strict load OK, 94 MB, size equals the Module 01 record: True
  [PASS] resnet50 fold 2  --  strict load OK, 94 MB, size equals the Module 01 record: True
  [PASS] resnet50 fold 3  --  strict load OK, 94 MB, size equals the Module 01 record: True
  [PASS] resnet50 fold 4  --  strict load OK, 94 MB, size equals the Module 01 record: True
  [PASS] vit_b_16 fold 0  --  strict load OK, 343 MB, size equals the Module 01 record: True
  [PASS] vit_b_16 fold 1  --  strict load OK, 343 MB, size equals the Module 01 record: True
  [PASS] vit_b_16 fold 2  --  strict load OK, 343 MB, size equals the Module 01 record: True
  [PASS] vit_b_16 fold 3  --  strict load OK, 343 MB, size equals the Module 01 record: True
  [PASS] vit_b_16 fold 4  --  strict load OK, 343 MB, size equals the Module 01 record: True

  1

In [35]:
"""
Phase 6 -- Module 04, cell 3 of 3: the scoring pass with the wiring proof.

Requires 04_00 and 04_01 (whose checkpoint check must have passed).  Just run the cell: it starts at once.
Slow; safe to re-run (finished folds are re-checked and skipped).

For each model (ResNet50, ViT-B/16) and fold k (0-4): load fold_k_best.pt strictly, score the fold's
validation images under bf16 autocast (batch 32, softmax in fp32, as in Phase 2), and compare the fold's macro
AUC-ROC with the locked Phase 2 value. Only a fold INSIDE the tolerance from the rules note is saved.
A fold outside it stops the run and nothing is written for it. Finished folds are re-read, re-checked and
skipped on a re-run. Output (one file per model and fold):
  baseline_predictions/<model>_fold<k>_predictions.npz
  keys: image_ids, lesion_ids, labels, probs (float32, n x 7), pred, fold, model_kind, checkpoint_sha256,
        auc_roc, locked_auc, tolerance, created_at
Plus baseline_predictions/scoring_pass_record.json (every fold's numbers).  No training, no checkpoint changed.
"""

import json
import time

BASE_OUT = OUT / "baseline_predictions"


def _npz_path(model_name, k):
    return BASE_OUT / f"{model_name}_fold{k}_predictions.npz"


def _write_npz(path, arrays):
    import io
    buf = io.BytesIO()
    np.savez(buf, **arrays)
    atomic_write_bytes(path, buf.getvalue())


def _ids_readable(path):
    """True if the image IDs can be read without pickle. The file is always closed again (Windows cannot delete an open file)."""
    try:
        with np.load(path, allow_pickle=False) as z:
            z["image_ids"]
        return True
    except ValueError:
        return False


def _validate_saved(path, fold_ids, model_name, k, locked, tol, sha):
    """Re-read a finished fold from disk and re-prove it. Returns (ok, detail, auc)."""
    with np.load(path, allow_pickle=False) as f:
        z = {key: f[key] for key in f.files}              # everything read, file closed
    need = {"image_ids", "lesion_ids", "labels", "probs", "pred", "fold", "model_kind", "checkpoint_sha256", "auc_roc"}
    if not need <= set(z):
        return False, f"keys missing {sorted(need - set(z))}", NAN
    if int(z["fold"]) != k or str(z["model_kind"]) != model_name or str(z["checkpoint_sha256"]) != sha:
        return False, "file is for a different fold, model or checkpoint than the Module 01 record", NAN
    ids, probs, labels = z["image_ids"].tolist(), z["probs"], z["labels"]
    if ids != fold_ids["image_id"].tolist() or probs.shape != (len(ids), 7) or not np.isfinite(probs).all():
        return False, "image IDs, shape or values differ from the fold", NAN
    if (labels != fold_ids["dx"].map(CLASS_TO_IDX).values).any() or (z["pred"] != probs.argmax(axis=1)).any():
        return False, "labels or predictions differ", NAN
    auc = fold_macro_auc(labels, probs)
    return abs(auc - locked) <= tol, f"re-read OK, AUC recomputed from the saved probabilities {auc:.6f}", auc


def _score_fold(model_name, k, cv, P6):
    spec = SCORING_MODELS[model_name]
    key = f"{spec['stem']}_fold{k}_best"
    obj = torch.load(P6["registry"][key], map_location="cpu", weights_only=True)
    model = BUILDERS[model_name]()
    model.load_state_dict(obj["model_state_dict"], strict=True)
    model.to(DEVICE).eval()
    ds = ValidationFoldDataset(cv, k, spec["resize_to"])
    loader = DataLoader(ds, batch_size=SCORING_BATCH_SIZE, shuffle=False, num_workers=0)
    probs = []
    with torch.no_grad():
        for images, _ in loader:
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                logits = model(images.to(DEVICE))
            probs.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
    del model
    torch.cuda.empty_cache()
    return ds.df, np.concatenate(probs).astype(np.float32)


def run_scoring_pass(P6):
    rules = check_rules_note()
    if not P6.get("input_checks_passed") or not P6.get("module04_available"):
        raise RuntimeError("Module 01 checks have not passed or the Module 04 inputs are missing.")
    import gc, sys
    sys.last_type = sys.last_value = sys.last_traceback = None      # release files held open by an earlier crash in this session
    gc.collect()
    torch.backends.cudnn.deterministic = True        # as in Phase 2's set_all_seeds
    torch.backends.cudnn.benchmark = False
    fp = json.loads(P6["fingerprints_path"].read_text(encoding="utf-8"))["entries"]
    tolerances = rules["scoring_pass"]["wiring_tolerance_abs_auc"]
    cv = pd.read_csv(P6["registry"]["p1_cv_folds"])
    BASE_OUT.mkdir(exist_ok=True)

    banner("PHASE 6 -- MODULE 04 (3/3): SCORING PASS WITH WIRING PROOF")
    print(f"  {'model':<9}{'fold':>5}{'images':>8}{'AUC now':>11}{'AUC locked':>13}{'difference':>12}{'tolerance':>11}   result")
    table = []
    for model_name, spec in SCORING_MODELS.items():
        tol, locked_all = tolerances[spec["tolerance_key"]], P6["locked_auc"][spec["stem"]]
        for k in range(SETTINGS["n_folds"]):
            path, locked = _npz_path(model_name, k), float(locked_all[k])
            sha = fp[f"{spec['stem']}_fold{k}_best"]["sha256"]
            fold_df = cv[cv["cv_fold"] == k].reset_index(drop=True)
            began = time.time()
            if path.exists():
                if not _ids_readable(path):   # saved by the first version of this cell (IDs stored as objects); its AUC was proven then
                    print(f"  (stale file from the first version of this cell, re-scoring: {path.name})")
                    assert inside_out(path), "refusing to delete outside the Phase 6 folder"
                    try:
                        path.unlink()
                    except PermissionError:
                        raise RuntimeError(f"Windows says another program still has {path.name} open. Interrupt any other cell that is "
                                           f"still running, then Kernel > Restart, re-run the earlier cells and try again.") from None
            if path.exists():
                ok, detail, auc = _validate_saved(path, fold_df, model_name, k, locked, tol, sha)
                status = "reused" if ok else "SAVED FILE FAILED"
                if not ok:
                    raise RuntimeError(f"{path.name} exists but does not check out: {detail}. Delete it yourself only if you mean to redo it.")
            else:
                df, probs = _score_fold(model_name, k, cv, P6)
                labels = df["dx"].map(CLASS_TO_IDX).values.astype(np.int64)
                auc = fold_macro_auc(labels, probs)
                ok = abs(auc - locked) <= tol
                status = "PASS" if ok else "FAIL"
                if ok:
                    _write_npz(path, {
                        "image_ids": np.array(df["image_id"].astype(str).tolist(), dtype="U"), "lesion_ids": np.array(df["lesion_id"].astype(str).tolist(), dtype="U"),
                        "labels": labels, "probs": probs, "pred": probs.argmax(axis=1).astype(np.int64),
                        "fold": np.int64(k), "model_kind": np.str_(model_name),
                        "checkpoint_sha256": np.str_(sha),
                        "auc_roc": np.float64(auc), "locked_auc": np.float64(locked), "tolerance": np.float64(tol),
                        "created_at": np.str_(now_iso())})
                    ok, detail, _ = _validate_saved(path, fold_df, model_name, k, locked, tol, sha)   # read-back from disk
                    if not ok:
                        raise RuntimeError(f"Read-back of {path.name} failed: {detail}")
            diff = auc - locked
            print(f"  {model_name:<9}{k:>5}{len(fold_df):>8}{auc:>11.6f}{locked:>13.6f}{diff:>+12.6f}{tol:>11.4f}   {status} ({time.time() - began:.0f} s)")
            table.append({"model": model_name, "fold": k, "n_images": len(fold_df), "auc_now": auc, "auc_locked": locked,
                          "difference": diff, "tolerance": tol, "status": status})
            if not ok:
                raise RuntimeError(f"WIRING PROOF FAILED for {model_name} fold {k}: difference {diff:+.6f}, tolerance {tol}. "
                                   f"Nothing was saved for this fold. Send Claude this printout; do not widen the tolerance.")

    record = {"format_version": 1, "recorded_at": now_iso(), "precision": "bf16 autocast, softmax fp32",
              "batch_size": SCORING_BATCH_SIZE, "device": str(DEVICE), "torch": torch.__version__, "folds": table}
    atomic_write_bytes(BASE_OUT / "scoring_pass_record.json", json_bytes(record))
    assert len(json.loads((BASE_OUT / "scoring_pass_record.json").read_text(encoding="utf-8"))["folds"]) == 10

    def pool(name):
        parts = []
        for k in range(5):
            with np.load(_npz_path(name, k), allow_pickle=False) as f:
                parts.append(f["image_ids"])
        ids = np.concatenate(parts)
        return len(ids) == len(set(ids.tolist())) == P6["expected"]["cv_pool_images"] and set(ids.tolist()) == set(cv["image_id"])
    rows = run_checks("MODULE 04: OUTPUT CHECKS", [
        (f"{n}: 5 files together cover the 8,583 pool images exactly once", lambda n=n: (pool(n), "image IDs equal cv_fold_assignments")) for n in SCORING_MODELS])
    if all(r["passed"] for r in rows):
        print(f"\n  Wiring proof passed for all 10 folds. Files in {BASE_OUT}\n  Module 04 is complete. Next: Module 05 (OOF assembly), after the walk-through.")
    return table


assert globals().get("CHECKPOINTS_OK") is True, "Run 04_01 first; all 10 checkpoints must fit before scoring."
SCORING = run_scoring_pass(P6)


PHASE 6 -- MODULE 04 (3/3): SCORING PASS WITH WIRING PROOF
  model     fold  images    AUC now   AUC locked  difference  tolerance   result
  (stale file from the first version of this cell, re-scoring: resnet50_fold0_predictions.npz)
  resnet50     0    1717   0.963783     0.963783   +0.000000     0.0010   PASS (8 s)
  resnet50     1    1716   0.961477     0.961477   +0.000000     0.0010   PASS (43 s)
  resnet50     2    1716   0.954482     0.954482   +0.000000     0.0010   PASS (45 s)
  resnet50     3    1718   0.962540     0.962540   +0.000000     0.0010   PASS (43 s)
  resnet50     4    1716   0.961671     0.961671   +0.000000     0.0010   PASS (39 s)
  vit_b_16     0    1717   0.964608     0.964608   +0.000000     0.0030   PASS (8 s)
  vit_b_16     1    1716   0.963219     0.963219   +0.000000     0.0030   PASS (7 s)
  vit_b_16     2    1716   0.954354     0.954354   +0.000000     0.0030   PASS (8 s)
  vit_b_16     3    1718   0.962950     0.962950   +0.000000     0.0030   PASS (

# Module 5

In [36]:
"""
Phase 6 -- Module 05: one aligned out-of-fold (OOF) table for the four models.

Requires cells 01_00 to 04_00 (and the 10 files from Module 04).  Just run the cell: it starts at once and
is safe to re-run (an existing OOF file is compared with a fresh build and reused only if identical).

Builds oof/oof_predictions.npz, with row i = the same image for all four models, in cv_fold_assignments.csv order:
  image_ids, lesion_ids, folds, labels, class_names,
  probs_multimodal, probs_ablation   (Phase 3 validation predictions, bf16 autocast)
  probs_resnet50, probs_vit_b_16     (Module 04, bf16 autocast)           each float32, 8583 x 7
Stops before writing if any alignment check fails. Prints no model comparison: only match checks.
Also defines load_oof() for the later modules.  Writes oof/oof_alignment_record.json.
"""

import io
import json

OOF_DIR = OUT / "oof"
OOF_PATH = OOF_DIR / "oof_predictions.npz"
OOF_MODELS = ("multimodal", "ablation", "resnet50", "vit_b_16")
PHASE3_AUC_TOLERANCE = 0.001        # stored Phase 3 predictions vs the best_val_auc_roc recorded in its run config


def _load_fold(model, k):
    """(image_ids, lesion_ids or None, labels, probs, class_names) of one model's file for fold k."""
    if model in ("multimodal", "ablation"):
        path, lesion_key = P6["registry"][f"p3_{model}_fold{k}_val_predictions"], None
    else:
        path, lesion_key = OUT / "baseline_predictions" / f"{model}_fold{k}_predictions.npz", "lesion_ids"
    with np.load(path, allow_pickle=False) as f:
        names = [str(c) for c in f["class_names"]] if "class_names" in f.files else list(CLASSES)
        lesions = f[lesion_key].astype(str) if lesion_key else None
        return f["image_ids"].astype(str), lesions, f["labels"].astype(np.int64), f["probs"].astype(np.float32), names


def load_oof():
    """The saved OOF table as a dict of arrays (file closed again)."""
    with np.load(OOF_PATH, allow_pickle=False) as f:
        return {key: f[key] for key in f.files}


def assemble_oof(P6):
    rules = check_rules_note()
    banner("PHASE 6 -- MODULE 05: OOF ASSEMBLY")
    cv = pd.read_csv(P6["registry"]["p1_cv_folds"])
    ho = pd.read_csv(P6["registry"]["p1_holdout_split"])
    master_ids = cv["image_id"].astype(str).values
    master_lesions = cv["lesion_id"].astype(str).values
    master_folds = cv["cv_fold"].values.astype(np.int64)
    master_labels = cv["dx"].map(CLASS_TO_IDX).values.astype(np.int64)
    N, pos = len(cv), {i: n for n, i in enumerate(master_ids)}
    issues = {key: [] for key in ("ids", "classes", "fold", "labels", "lesions", "once")}
    blocks, per_fold_auc = {}, []

    for m in OOF_MODELS:
        block, seen = np.full((N, 7), np.nan, dtype=np.float32), np.zeros(N, dtype=np.int64)
        for k in range(SETTINGS["n_folds"]):
            ids, lesions, labels, probs, names = _load_fold(m, k)
            if names != list(CLASSES):
                issues["classes"].append(f"{m} fold {k}: {names}")
            idx = np.array([pos.get(i, -1) for i in ids])
            if (idx < 0).any() or probs.shape != (len(ids), 7):
                issues["ids"].append(f"{m} fold {k}: {int((idx < 0).sum())} unknown IDs or shape {probs.shape}")
                continue
            if (master_folds[idx] != k).any():
                issues["fold"].append(f"{m} fold {k}: {int((master_folds[idx] != k).sum())} images belong to another fold")
            if (labels != master_labels[idx]).any():
                issues["labels"].append(f"{m} fold {k}: {int((labels != master_labels[idx]).sum())} labels differ")
            if lesions is not None and (lesions != master_lesions[idx]).any():
                issues["lesions"].append(f"{m} fold {k}: lesion IDs differ")
            block[idx] = probs
            seen[idx] += 1
        if not (seen == 1).all():
            issues["once"].append(f"{m}: {int((seen == 0).sum())} images missing, {int((seen > 1).sum())} repeated")
        blocks[m] = block

    # per-fold macro AUC-ROC of the pooled rows against the locked values
    locked = {"multimodal": P6["locked_auc"]["p3_multimodal"], "ablation": P6["locked_auc"]["p3_ablation"],
              "resnet50": P6["locked_auc"]["p2_script2_resnet50"], "vit_b_16": P6["locked_auc"]["p2_script3_vit_b_16"]}
    tol = {"multimodal": PHASE3_AUC_TOLERANCE, "ablation": PHASE3_AUC_TOLERANCE,
           **rules["scoring_pass"]["wiring_tolerance_abs_auc"]}
    finite = all(np.isfinite(b).all() for b in blocks.values())
    if finite:
        for m in OOF_MODELS:
            for k in range(SETTINGS["n_folds"]):
                rows = master_folds == k
                now = fold_macro_auc(master_labels[rows], blocks[m][rows])
                per_fold_auc.append({"model": m, "fold": k, "auc_now": now, "auc_locked": float(locked[m][k]),
                                     "difference": now - float(locked[m][k]), "tolerance": tol[m]})

    def issue_check(key, text):
        return lambda: (not issues[key], text if not issues[key] else "; ".join(issues[key][:4]))

    def auc_check(models):
        rows = [r for r in per_fold_auc if r["model"] in models]
        worst = max((abs(r["difference"]) for r in rows), default=NAN)
        return (len(rows) == 5 * len(models) and all(abs(r["difference"]) <= r["tolerance"] for r in rows),
                f"largest per-fold difference {worst:.2e} over {len(rows)} folds")

    checks = [
        ("Every file's image IDs are pool images; class order equals the Phase 3 order", lambda: (not issues["ids"] and not issues["classes"], "; ".join((issues["ids"] + issues["classes"])[:4]) or "20 files read")),
        ("Each image sits in the file of its own fold (all four models)", issue_check("fold", "fold membership equals cv_fold_assignments")),
        ("True labels equal the fold table in all four models", issue_check("labels", "labels identical")),
        ("Module 04 lesion IDs equal the fold table", issue_check("lesions", "lesion IDs identical")),
        ("Each image appears exactly once per model (8,583 rows each)", issue_check("once", "no image missing or repeated")),
        ("All probabilities finite; every row sums to 1 (within 1e-4)", lambda: (finite and all(abs(b.sum(axis=1) - 1).max() < 1e-4 for b in blocks.values()), "four 8583 x 7 blocks")),
        ("No holdout image is in the pool", lambda: (len(set(ho["image_id"].astype(str)) & set(master_ids)) == 0, f"{len(ho)} holdout images, 0 in the pool")),
        ("Phase 3 per-fold macro AUC-ROC from the pooled rows equals the recorded value", lambda: auc_check(("multimodal", "ablation"))),
        ("ResNet50 / ViT-B/16 per-fold macro AUC-ROC from the pooled rows equals the locked value", lambda: auc_check(("resnet50", "vit_b_16"))),
    ]
    rows = run_checks("MODULE 05: ALIGNMENT CHECKS (match checks only; no model comparison)", checks)
    if not all(r["passed"] for r in rows):
        print("\n  STOP: nothing was written. Send Claude the FAIL lines.")
        return None

    arrays = {"image_ids": np.array(master_ids.tolist(), dtype="U"), "lesion_ids": np.array(master_lesions.tolist(), dtype="U"),
              "folds": master_folds, "labels": master_labels, "class_names": np.array(list(CLASSES), dtype="U"),
              **{f"probs_{m}": blocks[m] for m in OOF_MODELS},
              "note": np.str_("rows in cv_fold_assignments.csv order; all four models bf16 autocast with fp32 softmax; "
                              "multimodal/ablation from Phase 3 validation predictions, resnet50/vit_b_16 from Module 04")}
    OOF_DIR.mkdir(exist_ok=True)
    if OOF_PATH.exists():
        old = load_oof()
        same = set(old) == set(arrays) and all(np.array_equal(old[key], arrays[key]) for key in arrays if key != "note")
        if not same:
            raise RuntimeError(f"{OOF_PATH.name} exists but differs from a fresh build. Delete it yourself only if you mean to redo it.")
        status = "reused (identical to a fresh build)"
    else:
        buf = io.BytesIO()
        np.savez(buf, **arrays)
        atomic_write_bytes(OOF_PATH, buf.getvalue())
        status = "written"
    back = load_oof()                                                     # read-back from disk
    assert all(np.array_equal(back[key], arrays[key]) for key in arrays if key != "note"), "OOF read-back differs"
    assert back["probs_multimodal"].shape == (N, 7) and len(set(back["image_ids"].tolist())) == N

    record = {"format_version": 1, "recorded_at": now_iso(), "status": status, "n_images": N, "checks": rows,
              "per_fold_auc_from_pooled_rows": per_fold_auc, "phase3_auc_tolerance": PHASE3_AUC_TOLERANCE}
    atomic_write_bytes(OOF_DIR / "oof_alignment_record.json", json_bytes(record))
    print(f"\n  OOF table {status}: {OOF_PATH}")
    print("  Module 05 is complete. Next: Module 06 (Claim 1), after the walk-through.")
    return per_fold_auc


OOF_AUC_ROWS = assemble_oof(P6)


PHASE 6 -- MODULE 05: OOF ASSEMBLY

MODULE 05: ALIGNMENT CHECKS (match checks only; no model comparison)
  [PASS] Every file's image IDs are pool images; class order equals the Phase 3 order  --  20 files read
  [PASS] Each image sits in the file of its own fold (all four models)  --  fold membership equals cv_fold_assignments
  [PASS] True labels equal the fold table in all four models  --  labels identical
  [PASS] Module 04 lesion IDs equal the fold table  --  lesion IDs identical
  [PASS] Each image appears exactly once per model (8,583 rows each)  --  no image missing or repeated
  [PASS] All probabilities finite; every row sums to 1 (within 1e-4)  --  four 8583 x 7 blocks
  [PASS] No holdout image is in the pool  --  1432 holdout images, 0 in the pool
  [PASS] Phase 3 per-fold macro AUC-ROC from the pooled rows equals the recorded value  --  largest per-fold difference 0.00e+00 over 10 folds
  [PASS] ResNet50 / ViT-B/16 per-fold macro AUC-ROC from the pooled rows equals the lock

# Module 6

In [37]:
"""
Phase 6 -- Module 06, cell 1 of 2: real-data null gate (shuffled labels). Runs before any Claim 1 result.

Requires cells 01_00 to 05_00.  Just run the cell (about 5-10 minutes; safe to re-run).

The real out-of-fold scores of the multimodal and image-only models are compared against diagnoses that have
been shuffled BETWEEN LESIONS (20 shuffles, seed 42, 1,000 lesion resamples each). Nothing real is left to find, so
the tests should flag differences about as often as chance. This measures, on the real data, how far DeLong's
assumption of independent images (it ignores lesions) distorts its p-values.
Limits are the ones fixed in Module 03 (NULL_RULES). If the gate fails, nothing in Module 06 runs: send Claude the printout.
Writes claim1/real_data_null_gate.json.  Starts with check_rules_note().
"""

import json
import time

check_rules_note()
assert "NULL_RULES" in globals(), "Run 03_04 first (it defines the limits)."
GATE_PATH = OUT / "claim1" / "real_data_null_gate.json"


def run_null_gate():
    banner("PHASE 6 -- MODULE 06 (1/2): REAL-DATA NULL GATE (shuffled labels, no real result involved)")
    if GATE_PATH.exists():
        saved = json.loads(GATE_PATH.read_text(encoding="utf-8"))
        print(f"  Already recorded {saved['recorded_at']}: gate {'PASSED' if saved['passed'] else 'FAILED'}. Observed: {saved['observed']}")
        return saved["passed"]
    oof = load_oof()
    y, A, B, lesions = oof["labels"], oof["probs_multimodal"], oof["probs_ablation"], oof["lesion_ids"]
    boot = LesionBootstrap(lesions, SETTINGS["bootstrap_resamples"], SETTINGS["bootstrap_seed"])
    code, n_les = boot.codes, boot.n_lesions
    label_of_lesion = np.zeros(n_les, dtype=np.int64)
    label_of_lesion[code] = y
    assert (label_of_lesion[code] == y).all(), "A lesion has images with different diagnoses; shuffling by lesion is not valid."
    rng = np.random.default_rng(SETTINGS["bootstrap_seed"])
    raw, covers, holm_hits = [], [], 0
    print(f"  {n_les} lesions, {len(y)} images, {boot.B} resamples per shuffle")
    for s in range(NULL_N_SHUFFLES):
        began = time.time()
        shuffled = rng.permutation(label_of_lesion)[code]
        t = compare(shuffled, A, B, boot, "multimodal", "ablation")["table"].iloc[:len(CLASSES)]
        p = t["p (raw)"].to_numpy(dtype=float)
        raw += p.tolist()
        covers += ((t["CI low"] <= 0) & (t["CI high"] >= 0)).tolist()
        holm_hits += int((t["p (Holm)"] < SETTINGS["alpha"]).any())
        print(f"  shuffle {s + 1:>2}/{NULL_N_SHUFFLES}: raw p < 0.05 in {int((p < SETTINGS['alpha']).sum())} of {len(p)} classes; "
              f"Holm-significant classes: {int((t['p (Holm)'] < SETTINGS['alpha']).sum())}   ({time.time() - began:.0f} s)")
    raw = np.array(raw)
    obs = {"share_raw_p_below_alpha": float((raw < SETTINGS["alpha"]).mean()), "median_raw_p": float(np.median(raw)),
           "share_ci_containing_zero": float(np.mean(covers)), "share_shuffles_any_holm_significant": holm_hits / NULL_N_SHUFFLES,
           "n_p_values": int(len(raw))}
    r = NULL_RULES
    rows = run_checks("REAL-DATA NULL GATE", [
        ("Raw p < 0.05 no more often than the limit", lambda: (obs["share_raw_p_below_alpha"] <= r["share_raw_p_below_alpha_max"],
            f"{obs['share_raw_p_below_alpha']:.1%} (limit {r['share_raw_p_below_alpha_max']:.0%}, chance 5%)")),
        ("Median raw p high enough", lambda: (obs["median_raw_p"] >= r["median_raw_p_min"], f"{obs['median_raw_p']:.2f} (min {r['median_raw_p_min']}, chance 0.50)")),
        ("Intervals contain 0 often enough", lambda: (obs["share_ci_containing_zero"] >= r["share_ci_containing_zero_min"],
            f"{obs['share_ci_containing_zero']:.1%} (min {r['share_ci_containing_zero_min']:.0%}, chance 95%)")),
        ("Shuffles with any Holm-significant class not too many", lambda: (obs["share_shuffles_any_holm_significant"] <= r["share_shuffles_any_holm_significant_max"],
            f"{obs['share_shuffles_any_holm_significant']:.0%} (max {r['share_shuffles_any_holm_significant_max']:.0%}, chance about 5%)")),
    ])
    passed = all(x["passed"] for x in rows)
    record = {"format_version": 1, "recorded_at": now_iso(), "passed": passed, "rules": r, "observed": obs, "checks": rows,
              "design": {"shuffles": NULL_N_SHUFFLES, "resamples": boot.B, "seed": SETTINGS["bootstrap_seed"], "unit": "lesion labels permuted between lesions",
                         "scores": "real pooled OOF probabilities, multimodal and Phase 3 matched ablation"}}
    atomic_write_bytes(GATE_PATH, json_bytes(record))
    assert json.loads(GATE_PATH.read_text(encoding="utf-8"))["passed"] == passed
    print("\n  Gate PASSED. Next: run 06_01 (Claim 1)." if passed else "\n  GATE FAILED: do not run 06_01. Send Claude this printout; no rule is changed without a recorded decision.")
    return passed


NULL_GATE_PASSED = run_null_gate()


PHASE 6 -- MODULE 06 (1/2): REAL-DATA NULL GATE (shuffled labels, no real result involved)
  6403 lesions, 8583 images, 1000 resamples per shuffle
  shuffle  1/20: raw p < 0.05 in 2 of 7 classes; Holm-significant classes: 0   (11 s)
  shuffle  2/20: raw p < 0.05 in 1 of 7 classes; Holm-significant classes: 0   (11 s)
  shuffle  3/20: raw p < 0.05 in 1 of 7 classes; Holm-significant classes: 0   (11 s)
  shuffle  4/20: raw p < 0.05 in 1 of 7 classes; Holm-significant classes: 0   (12 s)
  shuffle  5/20: raw p < 0.05 in 1 of 7 classes; Holm-significant classes: 0   (11 s)
  shuffle  6/20: raw p < 0.05 in 1 of 7 classes; Holm-significant classes: 0   (11 s)
  shuffle  7/20: raw p < 0.05 in 0 of 7 classes; Holm-significant classes: 0   (11 s)
  shuffle  8/20: raw p < 0.05 in 2 of 7 classes; Holm-significant classes: 0   (11 s)
  shuffle  9/20: raw p < 0.05 in 1 of 7 classes; Holm-significant classes: 0   (11 s)
  shuffle 10/20: raw p < 0.05 in 1 of 7 classes; Holm-significant classes: 1  

In [38]:
"""
Phase 6 -- Module 06, cell 2 of 2: Claim 1 -- multimodal vs the Phase 3 matched image-only ablation.

Requires cells 01_00 to 06_00 and a PASSED real-data null gate.  Just run the cell (about 1-2 minutes).

PRIMARY (the verdict): all 8,583 pooled out-of-fold images. Per class (one-vs-rest): paired DeLong, Holm across the
7 classes, lesion-bootstrap 95% interval (1,000 resamples, seed 42), delta AUC = multimodal minus ablation.
The category (supported / mixed / not detectable / opposed) is chosen by code from the counts using the conditions
fixed in the rules note, and its pre-written sentence template is printed unchanged.
SENSITIVITY (EXPLORATORY, chosen after seeing the Phase 3 results): the same analysis with fold 2's validation images
removed for both models. Shown beside the primary row; it never replaces it.
Macro AUC-ROC is a descriptive row only, never tested.
Writes claim1/claim1_primary_table.csv, claim1_fold2_sensitivity_table.csv, claim1_result.json (written once).
"""

import json

RULES6 = check_rules_note()
CLAIM1_DIR = OUT / "claim1"
RESULT_PATH = CLAIM1_DIR / "claim1_result.json"
NAME_A, NAME_B = "multimodal", "ablation"


def _p(v):
    return "n/a" if not np.isfinite(v) else (f"{v:.2e}" if v < 1e-3 else f"{v:.4f}")


def summarise(table):
    """Counts, class names, category and reported-as line from a compare() table (7 class rows + MACRO)."""
    rows = table.iloc[:len(CLASSES)]
    sup, opp = rows[rows["Reading"] == f"{NAME_A} higher"], rows[rows["Reading"] == f"{NAME_B} higher"]
    rest = rows[~rows["Reading"].isin([f"{NAME_A} higher", f"{NAME_B} higher"])]
    k1, k2, k3 = len(sup), len(opp), len(rest)
    category = ("supported" if k1 >= 1 and k2 == 0 else "mixed" if k1 >= 1 and k2 >= 1
                else "not_detectable" if k1 == 0 and k2 == 0 else "opposed")

    def listing(df):
        return ", ".join(f"{r['Class']} (delta {r['Delta (A - B)']:+.4f} [{r['CI low']:+.4f}, {r['CI high']:+.4f}])" for _, r in df.iterrows()) or "none"
    return {"k1": k1, "k2": k2, "k3": k3, "category": category,
            "supports": sup["Class"].tolist(), "opposes": opp["Class"].tolist(), "not_detectable": rest["Class"].tolist(),
            "disagreements": rows[rows["Reading"].str.startswith("disagreement")][["Class", "Reading"]].values.tolist(),
            "reported_as": f"Supports in {k1} of 7 [{listing(sup)}]; opposes in {k2} of 7 [{listing(opp)}]; not detectable in {k3} of 7 [{', '.join(rest['Class']) or 'none'}]"}


def show_table(table, title):
    d = pd.DataFrame({"Class": table["Class"], "N pos": table["N positive"].astype(int),
                      f"AUC {NAME_A}": table[f"AUC {NAME_A}"].map(lambda v: f"{v:.4f}"),
                      f"AUC {NAME_B}": table[f"AUC {NAME_B}"].map(lambda v: f"{v:.4f}"),
                      "Delta (A-B)": table["Delta (A - B)"].map(lambda v: f"{v:+.4f}"),
                      "95% CI": [f"[{a:+.4f}, {b:+.4f}]" for a, b in zip(table["CI low"], table["CI high"])],
                      "p raw": table["p (raw)"].map(_p), "p Holm": table["p (Holm)"].map(_p),
                      "Undef.": table["Undefined resamples"].astype(int), "Reading": table["Reading"]})
    print(f"\n  {title}\n" + "\n".join("  " + line for line in d.to_string(index=False).splitlines()))


def run_claim1():
    banner("PHASE 6 -- MODULE 06 (2/2): CLAIM 1, MULTIMODAL VS MATCHED IMAGE-ONLY ABLATION")
    gate = json.loads(GATE_PATH.read_text(encoding="utf-8")) if GATE_PATH.exists() else None
    if not gate or not gate["passed"]:
        raise RuntimeError("The real-data null gate has not passed. Run 06_00 first; Claim 1 does not run without it.")
    cats = RULES6["claim1"]["categories"]
    if RESULT_PATH.exists():
        saved = json.loads(RESULT_PATH.read_text(encoding="utf-8"))
        print(f"  Already recorded {saved['recorded_at']} (written once; delete the file yourself only if you mean to redo it).")
        for key in ("primary", "sensitivity"):
            print(f"  {key}: category {saved[key]['category']}; {saved[key]['reported_as']}")
        return saved

    oof = load_oof()
    y, A, B, les, folds = oof["labels"], oof["probs_multimodal"], oof["probs_ablation"], oof["lesion_ids"], oof["folds"]
    out, tables = {}, {}
    for key, keep, label in (("primary", np.ones(len(y), bool), "PRIMARY (verdict): all 8,583 pooled out-of-fold images"),
                             ("sensitivity", folds != 2, "SENSITIVITY (EXPLORATORY): fold 2 validation images removed for both models")):
        boot = LesionBootstrap(les[keep], SETTINGS["bootstrap_resamples"], SETTINGS["bootstrap_seed"])
        t = compare(y[keep], A[keep], B[keep], boot, NAME_A, NAME_B)["table"]
        tables[key] = t
        out[key] = {"n_images": int(keep.sum()), "n_lesions": boot.n_lesions, **summarise(t)}
        show_table(t, f"{label}  ({out[key]['n_images']} images, {out[key]['n_lesions']} lesions)")
        print(f"\n  {out[key]['reported_as']}")
        if out[key]["disagreements"]:
            print(f"  DeLong/bootstrap disagreements: {out[key]['disagreements']}")
        print(f"  Category: {out[key]['category']}")
        flagged = t[t["Flag"] != ""]["Class"].tolist()
        if flagged:
            print(f"  Undefined-resample share above 5%: {flagged}")
        out[key]["sentence_template"] = cats[out[key]["category"]]["sentence"]
    print(f"\n  Pre-written sentence for the PRIMARY category ({out['primary']['category']}), unchanged from the rules note:\n  {out['primary']['sentence_template']}")
    print("  Then, per the rules note: the fold-2 sensitivity result (above, EXPLORATORY) and the holdout check (Module 07, EXPLORATORY).")
    print(f"  Does the category change without fold 2?  {'NO' if out['primary']['category'] == out['sensitivity']['category'] else 'YES: ' + out['primary']['category'] + ' -> ' + out['sensitivity']['category']}")
    print("  Limits: " + " | ".join(RULES6["claim1"]["limits_for_all_four"][1:3]))

    tables["primary"].to_csv(CLAIM1_DIR / "claim1_primary_table.csv", index=False)
    tables["sensitivity"].to_csv(CLAIM1_DIR / "claim1_fold2_sensitivity_table.csv", index=False)
    record = {"format_version": 1, "recorded_at": now_iso(), "null_gate_recorded_at": gate["recorded_at"],
              "settings": {"alpha": SETTINGS["alpha"], "holm_family": 7, "resamples": SETTINGS["bootstrap_resamples"], "seed": SETTINGS["bootstrap_seed"],
                           "difference": "multimodal minus ablation", "test": "paired DeLong per class (one-vs-rest) + lesion bootstrap, both required"},
              "primary": out["primary"], "sensitivity": {**out["sensitivity"], "label": RULES6["claim1"]["fold2_sensitivity_label"]},
              "macro_row_note": "descriptive only, never tested"}
    write_json_once(RESULT_PATH, record)
    back = json.loads(RESULT_PATH.read_text(encoding="utf-8"))
    p_csv, s_csv = pd.read_csv(CLAIM1_DIR / "claim1_primary_table.csv"), pd.read_csv(CLAIM1_DIR / "claim1_fold2_sensitivity_table.csv")
    assert back["primary"]["category"] == out["primary"]["category"] and len(p_csv) == 8 and len(s_csv) == 8, "Claim 1 read-back failed"
    print(f"\n  Saved and read back from disk: {CLAIM1_DIR}\n  Module 06 is complete. Next: Module 07 (holdout check, EXPLORATORY), after the walk-through.")
    return record


CLAIM1 = run_claim1()


PHASE 6 -- MODULE 06 (2/2): CLAIM 1, MULTIMODAL VS MATCHED IMAGE-ONLY ABLATION

  PRIMARY (verdict): all 8,583 pooled out-of-fold images  (8583 images, 6403 lesions)
                Class  N pos AUC multimodal AUC ablation Delta (A-B)             95% CI    p raw   p Holm  Undef.                      Reading
                AKIEC    281         0.8887       0.9127     -0.0241 [-0.0478, -0.0011]   0.0168   0.0841       0 disagreement: bootstrap only
                  BCC    438         0.9548       0.9482     +0.0067 [-0.0080, +0.0216]   0.3529   0.7074       0               not detectable
                  BKL    941         0.8967       0.8882     +0.0085 [-0.0046, +0.0217]   0.1768   0.7074       0               not detectable
                   DF    100         0.9539       0.9040     +0.0499 [+0.0134, +0.0876]   0.0047   0.0285       0            multimodal higher
                  MEL    954         0.8514       0.8417     +0.0097 [-0.0067, +0.0248]   0.1962   0.7074       0     

# Module 7

In [39]:
"""
Phase 6 -- Module 07: holdout check of Claim 1 (EXPLORATORY).

Requires cells 01_00 to 06_01.  Just run the cell (about 1-2 minutes; the result is written once).

The same comparison as Claim 1 (per-class paired DeLong, Holm across 7, lesion bootstrap 1,000 resamples, seed 42,
delta AUC = multimodal minus ablation) on the 1,432 internal-holdout images, from the Phase 4 fp32 prediction files
(no new inference). ONE fold-3 model per arm: the frozen multimodal model and the fold-3 image-only ablation.
Labelled EXPLORATORY; it is shown beside the pooled-OOF verdict and never replaces it. Classes with fewer than 30
positives are flagged low-sample. Writes holdout_check/holdout_check_table.csv and holdout_check_result.json.
"""

import json

RULES6 = check_rules_note()
HOLD_DIR = OUT / "holdout_check"
HOLD_RESULT = HOLD_DIR / "holdout_check_result.json"
HOLD_LABEL = "EXPLORATORY: internal holdout, one fold-3 model per arm (frozen multimodal vs fold-3 image-only ablation); shown beside, never in place of, the pooled out-of-fold verdict"


def run_holdout_check():
    banner("PHASE 6 -- MODULE 07: HOLDOUT CHECK OF CLAIM 1 (EXPLORATORY)")
    if HOLD_RESULT.exists():
        saved = json.loads(HOLD_RESULT.read_text(encoding="utf-8"))
        print(f"  Already recorded {saved['recorded_at']} (written once; delete the file yourself only if you mean to redo it).")
        print(f"  category {saved['summary']['category']}; {saved['summary']['reported_as']}")
        return saved
    ho = pd.read_csv(P6["registry"]["p1_holdout_split"])
    ids, lesions = ho["image_id"].astype(str).values, ho["lesion_id"].astype(str).values
    pos = {i: n for n, i in enumerate(ids)}

    def load(key):
        with np.load(P6["registry"][key], allow_pickle=False) as f:
            fid, labels, probs = f["image_ids"].astype(str), f["labels"].astype(np.int64), f["probs"].astype(np.float32)
            names = [str(c) for c in f["class_names"]]
        idx = np.array([pos.get(i, -1) for i in fid])
        ok = bool((idx >= 0).all() and len(set(idx.tolist())) == len(ids) == len(fid))
        P, L = np.full((len(ids), 7), np.nan, np.float32), np.full(len(ids), -1, np.int64)
        if ok:
            P[idx], L[idx] = probs, labels
        return P, L, names, ok

    A, LA, na, oka = load("p4_multimodal_holdout_predictions")
    B, LB, nb, okb = load("p4_ablation_holdout_predictions")
    overall = pd.read_csv(P6["registry"]["p5_holdout_overall"])
    p5_auc = [to_float(v) for v in overall["Macro AUC-ROC (primary)"]]

    def matches_phase5(P, L):
        if not np.isfinite(P).all():
            return False
        return min(abs(fold_macro_auc(L, P) - v) for v in p5_auc) < 1e-9

    rows = run_checks("MODULE 07: INPUT CHECKS", [
        ("Both files hold exactly the 1,432 holdout images (each once)", lambda: (oka and okb, f"{len(ids)} images, {ho['lesion_id'].nunique()} lesions")),
        ("Class order equals the Phase 3 order in both files", lambda: (na == list(CLASSES) and nb == list(CLASSES), str(na))),
        ("True labels identical in the two files", lambda: (bool((LA == LB).all()) and LA.min() >= 0 and LA.max() <= 6, "labels identical, within 0-6")),
        ("Probabilities finite; rows sum to 1 (within 1e-4)", lambda: (bool(np.isfinite(A).all() and np.isfinite(B).all() and abs(A.sum(1) - 1).max() < 1e-4 and abs(B.sum(1) - 1).max() < 1e-4), "two 1432 x 7 blocks")),
        ("Macro AUC-ROC from these inputs equals a Phase 5 holdout value (both models)", lambda: (matches_phase5(A, LA) and matches_phase5(B, LB), "equal to the Phase 5 holdout_overall table within 1e-9")),
    ])
    if not all(r["passed"] for r in rows):
        print("\n  STOP: nothing was written. Send Claude the FAIL lines.")
        return None

    boot = LesionBootstrap(lesions, SETTINGS["bootstrap_resamples"], SETTINGS["bootstrap_seed"])
    t = compare(LA, A, B, boot, NAME_A, NAME_B)["table"]
    t["Low sample (N<30)"] = [bool(n < LOW_SAMPLE_N) for n in t["N positive"].iloc[:len(CLASSES)]] + [False]
    summary = {"n_images": int(len(ids)), "n_lesions": boot.n_lesions, **summarise(t)}
    show_table(t, f"{HOLD_LABEL}  ({summary['n_images']} images, {summary['n_lesions']} lesions)")
    low = t[t["Low sample (N<30)"]]["Class"].tolist()
    print(f"\n  {summary['reported_as']}")
    if summary["disagreements"]:
        print(f"  DeLong/bootstrap disagreements: {summary['disagreements']}")
    if low:
        print(f"  Low-sample classes (fewer than 30 positives; read with caution): {low}")
    flagged = t[t["Flag"] != ""]["Class"].tolist()
    if flagged:
        print(f"  Undefined-resample share above 5%: {flagged}")
    print(f"  Category by the same rule: {summary['category']}  (EXPLORATORY; the verdict is the pooled out-of-fold row)")

    HOLD_DIR.mkdir(exist_ok=True)
    t.to_csv(HOLD_DIR / "holdout_check_table.csv", index=False)
    record = {"format_version": 1, "recorded_at": now_iso(), "label": HOLD_LABEL, "status": "exploratory",
              "inputs": "Phase 4 fp32 holdout predictions (plain probs), no new inference",
              "settings": {"alpha": SETTINGS["alpha"], "holm_family": 7, "resamples": SETTINGS["bootstrap_resamples"],
                           "seed": SETTINGS["bootstrap_seed"], "difference": "multimodal minus ablation"},
              "summary": summary, "low_sample_classes": low, "input_checks": rows}
    write_json_once(HOLD_RESULT, record)
    back = json.loads(HOLD_RESULT.read_text(encoding="utf-8"))
    assert back["summary"]["category"] == summary["category"] and len(pd.read_csv(HOLD_DIR / "holdout_check_table.csv")) == 8, "Holdout read-back failed"
    print(f"\n  Saved and read back from disk: {HOLD_DIR}\n  Module 07 is complete. Next: Module 08 (baseline comparisons), after the walk-through.")
    return record


HOLDOUT_CHECK = run_holdout_check()


PHASE 6 -- MODULE 07: HOLDOUT CHECK OF CLAIM 1 (EXPLORATORY)

MODULE 07: INPUT CHECKS
  [PASS] Both files hold exactly the 1,432 holdout images (each once)  --  1432 images, 1067 lesions
  [PASS] Class order equals the Phase 3 order in both files  --  ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
  [PASS] True labels identical in the two files  --  labels identical, within 0-6
  [PASS] Probabilities finite; rows sum to 1 (within 1e-4)  --  two 1432 x 7 blocks
  [PASS] Macro AUC-ROC from these inputs equals a Phase 5 holdout value (both models)  --  equal to the Phase 5 holdout_overall table within 1e-9

  5 of 5 passed

  EXPLORATORY: internal holdout, one fold-3 model per arm (frozen multimodal vs fold-3 image-only ablation); shown beside, never in place of, the pooled out-of-fold verdict  (1432 images, 1067 lesions)
                Class  N pos AUC multimodal AUC ablation Delta (A-B)             95% CI    p raw p Holm  Undef.                      Reading
                AKIEC  

# Module 8

In [40]:
"""
Phase 6 -- Module 08: multimodal vs the ResNet50 and ViT-B/16 baselines (SECONDARY, not a verdict).

Requires cells 01_00 to 07_00.  Just run the cell (about 1-2 minutes; the result is written once).

On the same 8,583 pooled out-of-fold images: per-class paired DeLong, Holm across 7 (separately for each comparison),
lesion bootstrap 1,000 resamples (seed 42; BOTH comparisons share the same resamples), delta AUC = multimodal minus baseline.
Counts are reported per comparison; no Claim 1 sentence is chosen. The locked Phase 2 baseline rows are read from the
run configs and shown unchanged as reference. The baselines have no metadata branch and a different training recipe, so a
difference here does not isolate what metadata does.
Writes baselines/multimodal_vs_resnet50_table.csv, multimodal_vs_vit_b_16_table.csv, baseline_comparisons_result.json.
"""

import json

RULES6 = check_rules_note()
BASE08_DIR = OUT / "baselines"
BASE08_RESULT = BASE08_DIR / "baseline_comparisons_result.json"
SECONDARY_LABEL = "SECONDARY: " + "; ".join(RULES6["status_labels"]["secondary"]) + " (pooled out-of-fold, not a verdict)"
CAVEAT = ("The baselines have no metadata branch and were trained under the Phase 2 recipe (no class balancing), the multimodal model under "
          "the Phase 3 recipe; a difference here does not isolate what metadata does. The pooled set mixes five models per arm.")
REFERENCE_ROWS = {"EfficientNet-B3 (Phase 2 baseline)": ("p2_script1_efficientnet_b3_run_config", 0.9316, 0.0298),
                  "ResNet50 (Phase 2 baseline)": ("p2_script2_resnet50_run_config", 0.9608, 0.0036),
                  "ViT-B/16 (Phase 2 baseline)": ("p2_script3_vit_b_16_run_config", 0.9609, 0.0041)}


def summarise_pair(table, a, b):
    rows = table.iloc[:len(CLASSES)]
    hi_a, hi_b = rows[rows["Reading"] == f"{a} higher"], rows[rows["Reading"] == f"{b} higher"]
    rest = rows[~rows["Reading"].isin([f"{a} higher", f"{b} higher"])]

    def listing(df):
        return ", ".join(f"{r['Class']} (delta {r['Delta (A - B)']:+.4f} [{r['CI low']:+.4f}, {r['CI high']:+.4f}])" for _, r in df.iterrows()) or "none"
    return {f"{a}_higher": hi_a["Class"].tolist(), f"{b}_higher": hi_b["Class"].tolist(), "not_detectable": rest["Class"].tolist(),
            "disagreements": rows[rows["Reading"].str.startswith("disagreement")][["Class", "Reading"]].values.tolist(),
            "reported_as": f"{a} higher in {len(hi_a)} of 7 [{listing(hi_a)}]; {b} higher in {len(hi_b)} of 7 [{listing(hi_b)}]; "
                           f"not detectable in {len(rest)} of 7 [{', '.join(rest['Class']) or 'none'}]"}


def show_pair(table, a, b, title):
    d = pd.DataFrame({"Class": table["Class"], "N pos": table["N positive"].astype(int),
                      f"AUC {a}": table[f"AUC {a}"].map(lambda v: f"{v:.4f}"), f"AUC {b}": table[f"AUC {b}"].map(lambda v: f"{v:.4f}"),
                      "Delta (A-B)": table["Delta (A - B)"].map(lambda v: f"{v:+.4f}"),
                      "95% CI": [f"[{x:+.4f}, {y:+.4f}]" for x, y in zip(table["CI low"], table["CI high"])],
                      "p raw": table["p (raw)"].map(_p), "p Holm": table["p (Holm)"].map(_p),
                      "Undef.": table["Undefined resamples"].astype(int), "Reading": table["Reading"]})
    print(f"\n  {title}\n" + "\n".join("  " + line for line in d.to_string(index=False).splitlines()))


def run_baselines():
    banner("PHASE 6 -- MODULE 08: MULTIMODAL VS RESNET50 AND VIT-B/16 (SECONDARY)")
    if BASE08_RESULT.exists():
        saved = json.loads(BASE08_RESULT.read_text(encoding="utf-8"))
        print(f"  Already recorded {saved['recorded_at']} (written once; delete the file yourself only if you mean to redo it).")
        for name, res in saved["comparisons"].items():
            print(f"  {name}: {res['reported_as']}")
        return saved

    reference, ref_ok = {}, True
    for label, (key, mean_exp, sd_exp) in REFERENCE_ROWS.items():
        cfg = json.loads(P6["registry"][key].read_text(encoding="utf-8"))["summary"]["auc_roc"]
        reference[label] = {"mean": float(cfg["mean"]), "sd": float(cfg["sd"])}
        ref_ok &= round(cfg["mean"], 4) == mean_exp and round(cfg["sd"], 4) == sd_exp
    rows = run_checks("MODULE 08: REFERENCE CHECK", [
        ("Locked Phase 2 baseline AUC-ROC (mean +/- SD over 5 folds) equals the values in the plan", lambda: (ref_ok, "; ".join(f"{k}: {v['mean']:.4f} +/- {v['sd']:.4f}" for k, v in reference.items())))])
    if not all(r["passed"] for r in rows):
        print("\n  STOP: nothing was written. Send Claude the FAIL line.")
        return None

    oof = load_oof()
    y, A, les = oof["labels"], oof["probs_multimodal"], oof["lesion_ids"]
    boot = LesionBootstrap(les, SETTINGS["bootstrap_resamples"], SETTINGS["bootstrap_seed"])      # shared by both comparisons
    print(f"\n  {SECONDARY_LABEL}\n  {CAVEAT}")
    results, tables = {}, {}
    for other in ("resnet50", "vit_b_16"):
        t = compare(y, A, oof[f"probs_{other}"], boot, "multimodal", other)["table"]
        tables[other] = t
        results[f"multimodal_vs_{other}"] = {"n_images": int(len(y)), "n_lesions": boot.n_lesions, **summarise_pair(t, "multimodal", other)}
        show_pair(t, "multimodal", other, f"multimodal vs {other}  ({len(y)} images, {boot.n_lesions} lesions; delta = multimodal minus {other})")
        print(f"\n  {results[f'multimodal_vs_{other}']['reported_as']}")
        if results[f"multimodal_vs_{other}"]["disagreements"]:
            print(f"  DeLong/bootstrap disagreements: {results[f'multimodal_vs_{other}']['disagreements']}")
        flagged = t[t["Flag"] != ""]["Class"].tolist()
        if flagged:
            print(f"  Undefined-resample share above 5%: {flagged}")
    print("\n  Locked Phase 2 baseline rows, unchanged (macro AUC-ROC, mean +/- SD over 5 folds, reference only):")
    for k, v in reference.items():
        print(f"    {k}: {v['mean']:.4f} +/- {v['sd']:.4f}")

    BASE08_DIR.mkdir(exist_ok=True)
    for other, t in tables.items():
        t.to_csv(BASE08_DIR / f"multimodal_vs_{other}_table.csv", index=False)
    record = {"format_version": 1, "recorded_at": now_iso(), "label": SECONDARY_LABEL, "caveat": CAVEAT,
              "settings": {"alpha": SETTINGS["alpha"], "holm_family": 7, "resamples": SETTINGS["bootstrap_resamples"], "seed": SETTINGS["bootstrap_seed"],
                           "difference": "multimodal minus baseline", "shared_resamples": True},
              "reference_rows_phase2_locked": reference, "comparisons": results}
    write_json_once(BASE08_RESULT, record)
    back = json.loads(BASE08_RESULT.read_text(encoding="utf-8"))
    assert set(back["comparisons"]) == {"multimodal_vs_resnet50", "multimodal_vs_vit_b_16"}
    assert all(len(pd.read_csv(BASE08_DIR / f"multimodal_vs_{o}_table.csv")) == 8 for o in tables), "Baseline table read-back failed"
    print(f"\n  Saved and read back from disk: {BASE08_DIR}\n  Module 08 is complete. Next: Module 09 (intervals and the generalisation gap), after the walk-through.")
    return record


BASELINE_COMPARISONS = run_baselines()


PHASE 6 -- MODULE 08: MULTIMODAL VS RESNET50 AND VIT-B/16 (SECONDARY)

MODULE 08: REFERENCE CHECK
  [PASS] Locked Phase 2 baseline AUC-ROC (mean +/- SD over 5 folds) equals the values in the plan  --  EfficientNet-B3 (Phase 2 baseline): 0.9316 +/- 0.0298; ResNet50 (Phase 2 baseline): 0.9608 +/- 0.0036; ViT-B/16 (Phase 2 baseline): 0.9609 +/- 0.0041

  1 of 1 passed

  SECONDARY: multimodal vs ResNet50 tests; multimodal vs ViT-B/16 tests (pooled out-of-fold, not a verdict)
  The baselines have no metadata branch and were trained under the Phase 2 recipe (no class balancing), the multimodal model under the Phase 3 recipe; a difference here does not isolate what metadata does. The pooled set mixes five models per arm.

  multimodal vs resnet50  (8583 images, 6403 lesions; delta = multimodal minus resnet50)
                Class  N pos AUC multimodal AUC resnet50 Delta (A-B)             95% CI    p raw   p Holm  Undef.                    Reading
                AKIEC    281         0.8887

# Module 9

In [41]:
"""
Phase 6 -- Module 09, cell 1 of 3: load the Phase 5 predictions and prove Phase 6 reproduces Phase 5's numbers (no resampling yet).

Requires cells 01_00 to 08_00.  Just run the cell (about a minute). Stops (writes nothing) if any check fails.

Loads, for the intervals: the holdout (Phase 4 fp32 predictions, 1,432 images / 1,067 lesions) and BCN20000 primary images
(label >= 0: 16,284 images / 4,591 lesions) for the multimodal and ablation models, plus the Phase 5 SITE-BLIND multimodal run
(EXPLORATORY). The site-blind file is not in Module 01's fingerprint registry, so it is hashed here and the hash is recorded.
Equality gate: the Phase 5 library recomputes every point estimate and it must equal the Phase 5 files (per-class tables,
overall tables, site-blind tables, and the holdout/BCN20000 pairs behind the gap table). Sets DATA09, MS09, SITE_BLIND_SHA.
"""

import json

check_rules_note()
P5RES = DIRS["p5"] / "bcn_results"
SITE_BLIND_NPZ = DIRS["p5"] / "bcn_predictions" / "bcn_predictions_multimodal_site_blind.npz"
UNITS = [("holdout", "multimodal"), ("holdout", "ablation"), ("bcn", "multimodal"), ("bcn", "ablation"), ("bcn", "site_blind")]
INT_DIR = OUT / "intervals"
HEADLINE_NAMES = ["Macro AUC-ROC", "Balanced accuracy", "Macro-F1", "Macro precision", "Macro PR-AUC", "ECE (15 bins)", "Brier (multiclass)", "Accuracy"]
PER_CLASS_KEYS = ["auc_roc", "recall", "specificity", "precision", "f1", "pr_auc"]
PER_CLASS_NAMES = ["AUC-ROC", "Sensitivity", "Specificity", "Precision", "F1", "PR-AUC"]
METRIC_NAMES = HEADLINE_NAMES + ["Macro precision, classes left out"] + [f"{c.upper()} | {n}" for c in CLASSES for n in PER_CLASS_NAMES]


def headline_values(ms):
    """The eight headline numbers, in HEADLINE_NAMES order, from a compute_metric_set result."""
    m = ms["macro"]
    return [m["auc_roc"], ms["balanced_accuracy"], m["f1"], m["precision"], m["pr_auc"],
            ms["calibration"]["ece"], ms["brier"], ms["accuracy"]]


def metric_vector(y, probs):
    """All numbers that get an interval, as one vector in METRIC_NAMES order (NaN = undefined)."""
    ms = compute_metric_set(y, probs, with_sens_at_spec=False)
    v = headline_values(ms) + [float(ms["macro"]["precision_left_out"])]
    for c in CLASSES:
        v += [ms["per_class"][c][k] for k in PER_CLASS_KEYS]
    return np.array(v, dtype=np.float64)


def _load_holdout(model):
    ho = pd.read_csv(P6["registry"]["p1_holdout_split"])
    ids, lesions = ho["image_id"].astype(str).values, ho["lesion_id"].astype(str).values
    pos = {i: n for n, i in enumerate(ids)}
    with np.load(P6["registry"][f"p4_{model}_holdout_predictions"], allow_pickle=False) as f:
        fid, labels, probs = f["image_ids"].astype(str), f["labels"].astype(np.int64), f["probs"].astype(np.float32)
    idx = np.array([pos[i] for i in fid])
    P, L = np.zeros((len(ids), 7), np.float32), np.zeros(len(ids), np.int64)
    P[idx], L[idx] = probs, labels
    return L, P, lesions, ids


def _load_bcn(run):
    path = SITE_BLIND_NPZ if run == "site_blind" else P6["registry"][f"p5_bcn_predictions_{run}"]
    with np.load(path, allow_pickle=False) as f:
        labels, probs = f["labels"].astype(np.int64), f["probs"].astype(np.float32)
        les, isic, names = f["lesion_ids"].astype(str), f["isic_ids"].astype(str), [str(c) for c in f["class_names"]]
    keep = labels >= 0                                   # the 559 SCC images carry label -1 and are not scored
    return labels[keep], probs[keep], les[keep], isic[keep], names


def _gap(x, y):
    x, y = to_float(x), to_float(y)
    return 0.0 if (np.isnan(x) and np.isnan(y)) else abs(x - y)


def _per_class_vs(mine, theirs):
    if mine["Class"].tolist() != theirs["Class"].tolist():
        return False, "class rows differ"
    worst, missing = 0.0, []
    for col in mine.columns:
        if col in ("Precision mode", "Class", "Flag", "Model"):
            continue
        if col not in theirs.columns:
            missing.append(col)
            continue
        worst = max([worst] + [_gap(a, b) for a, b in zip(mine[col], theirs[col])])
    flags = bool((mine["Flag"].fillna("") == theirs["Flag"].fillna("")).all()) if "Flag" in theirs.columns else None
    return (worst < 1e-9 and not missing), f"largest difference {worst:.1e}; columns missing in the Phase 5 file: {missing or 'none'}; flags identical: {flags}"


def _overall_match(rows, theirs):
    """rows: dict model -> one-row Series. Matches each to a Phase 5 row by value (labels are not trusted)."""
    used, notes = set(), []
    for name, mine in rows.items():
        cols = [c for c in mine.index if c not in ("Model", "Precision mode") and c in theirs.columns]
        diffs = [max(_gap(mine[c], theirs.iloc[i][c]) for c in cols) for i in range(len(theirs))]
        best = int(np.argmin(diffs))
        used.add(best)
        notes.append(f"{name}: row {best}, largest difference {diffs[best]:.1e}")
        if not diffs[best] < 1e-9:
            return False, "; ".join(notes)
    return len(used) == len(rows), "; ".join(notes) + " (rows matched by value)"


# ----------------------------------------------------------------------------
# Load everything, compute the point estimates with the Phase 5 library
# ----------------------------------------------------------------------------
DATA09, MS09, LOAD_NOTES = {}, {}, {}
for m in ("multimodal", "ablation"):
    L, P, les, ids = _load_holdout(m)
    DATA09[("holdout", m)] = (L, P, les)
for run in ("multimodal", "ablation", "site_blind"):
    if run == "site_blind" and not SITE_BLIND_NPZ.exists():
        raise RuntimeError(f"Phase 5 site-blind predictions not found: {SITE_BLIND_NPZ}")
    L, P, les, isic, names = _load_bcn(run)
    DATA09[("bcn", run)] = (L, P, les)
    LOAD_NOTES[run] = (isic, names)
SITE_BLIND_SHA = sha256_file(SITE_BLIND_NPZ)
for unit, (L, P, les) in DATA09.items():
    MS09[unit] = compute_metric_set(L, P)                # full set (with sensitivity at specificity) for the equality tables


def _t_data():
    h, b = DATA09[("holdout", "multimodal")], DATA09[("bcn", "multimodal")]
    ok = (len(h[0]) == 1432 and len(set(h[2])) == 1067 and len(b[0]) == 16284 and len(set(b[2])) == 4591
          and all(len(DATA09[("bcn", r)][0]) == 16284 for r in ("ablation", "site_blind")))
    return ok, f"holdout {len(h[0])} images / {len(set(h[2]))} lesions; BCN20000 primary {len(b[0])} images / {len(set(b[2]))} lesions"


def _t_alignment():
    m, a, s = (DATA09[("bcn", r)] for r in ("multimodal", "ablation", "site_blind"))
    ids = {r: LOAD_NOTES[r][0] for r in LOAD_NOTES}
    ok = (np.array_equal(m[0], a[0]) and np.array_equal(m[0], s[0]) and np.array_equal(m[2], a[2]) and np.array_equal(m[2], s[2])
          and np.array_equal(ids["multimodal"], ids["ablation"]) and np.array_equal(ids["multimodal"], ids["site_blind"])
          and all(LOAD_NOTES[r][1] == list(CLASSES) for r in LOAD_NOTES)
          and np.array_equal(DATA09[("holdout", "multimodal")][0], DATA09[("holdout", "ablation")][0])
          and np.array_equal(DATA09[("holdout", "multimodal")][2], DATA09[("holdout", "ablation")][2]))
    return ok, "same images, labels, lesions and class order across the runs of each dataset (resamples can be shared)"


def _per_class_check(ds, run, path):
    mine = metric_table(MS09[(ds, run)])
    return _per_class_vs(mine, pd.read_csv(path))


def _overall_check(ds, runs, path):
    rows = {r: overall_row(MS09[(ds, r)], r).iloc[0] for r in runs}
    return _overall_match(rows, pd.read_csv(path))


def _gap_check():
    theirs = pd.read_csv(P5RES / "bcn_gap_all_headline_metrics.csv")
    hcol, bcol = [c for c in theirs.columns if c.startswith("Holdout")][0], [c for c in theirs.columns if c.startswith("BCN20000") and "minus" not in c][0]
    pairs = [(to_float(r[hcol]), to_float(r[bcol])) for _, r in theirs.iterrows()]
    found, total = 0, 0
    for m in ("multimodal", "ablation"):
        for h, b in zip(headline_values(MS09[("holdout", m)]), headline_values(MS09[("bcn", m)])):
            total += 1
            found += any(abs(h - ph) < 1e-4 and abs(b - pb) < 1e-4 for ph, pb in pairs)      # the Phase 5 file holds rounded text
    return found == total == len(pairs), f"{found} of {total} holdout/BCN20000 pairs found in bcn_gap_all_headline_metrics.csv (rounded text, tolerance 1e-4)"


_rows = run_checks("PHASE 6 -- MODULE 09 (1/3): EQUALITY GATE (Phase 6 point estimates vs the Phase 5 files)", [
    ("Image, lesion and label counts", _t_data),
    ("Same images, labels, lesions and class order across runs", _t_alignment),
    ("Holdout per-class table, multimodal", lambda: _per_class_check("holdout", "multimodal", P6["registry"]["p5_holdout_per_class_multimodal"])),
    ("Holdout per-class table, ablation", lambda: _per_class_check("holdout", "ablation", P6["registry"]["p5_holdout_per_class_ablation"])),
    ("Holdout overall table (both models)", lambda: _overall_check("holdout", ("multimodal", "ablation"), P6["registry"]["p5_holdout_overall"])),
    ("BCN20000 per-class table, multimodal", lambda: _per_class_check("bcn", "multimodal", P6["registry"]["p5_bcn_per_class_multimodal"])),
    ("BCN20000 per-class table, ablation", lambda: _per_class_check("bcn", "ablation", P6["registry"]["p5_bcn_per_class_ablation"])),
    ("BCN20000 overall table (both models)", lambda: _overall_check("bcn", ("multimodal", "ablation"), P6["registry"]["p5_bcn_overall"])),
    ("Site-blind per-class table (EXPLORATORY)", lambda: _per_class_check("bcn", "site_blind", P5RES / "bcn_site_blind_per_class_fp32_EXPLORATORY.csv")),
    ("Site-blind overall table (EXPLORATORY)", lambda: _overall_check("bcn", ("site_blind",), P5RES / "bcn_site_blind_overall_fp32_EXPLORATORY.csv")),
    ("Holdout / BCN20000 pairs behind the Phase 5 gap table", _gap_check),
])
assert all(r["passed"] for r in _rows), "Equality gate failed: nothing else runs. Send Claude the FAIL lines."
print(f"\n  Site-blind predictions file hashed (not in the Module 01 registry): {SITE_BLIND_SHA}")
print("  Gate passed. Next: run 09_01 (the bootstrap; resumable).")


PHASE 6 -- MODULE 09 (1/3): EQUALITY GATE (Phase 6 point estimates vs the Phase 5 files)
  [PASS] Image, lesion and label counts  --  holdout 1432 images / 1067 lesions; BCN20000 primary 16284 images / 4591 lesions
  [PASS] Same images, labels, lesions and class order across runs  --  same images, labels, lesions and class order across the runs of each dataset (resamples can be shared)
  [PASS] Holdout per-class table, multimodal  --  largest difference 1.1e-16; columns missing in the Phase 5 file: none; flags identical: True
  [PASS] Holdout per-class table, ablation  --  largest difference 1.1e-16; columns missing in the Phase 5 file: none; flags identical: True
  [PASS] Holdout overall table (both models)  --  multimodal: row 0, largest difference 1.1e-16; ablation: row 1, largest difference 8.3e-17 (rows matched by value)
  [PASS] BCN20000 per-class table, multimodal  --  largest difference 1.1e-16; columns missing in the Phase 5 file: none; flags identical: True
  [PASS] BCN20000

In [42]:
"""
Phase 6 -- Module 09, cell 2 of 3: lesion-bootstrap draws for every number that gets an interval.

Requires 09_00 (its equality gate must have passed).  Just run the cell (about 10-30 minutes; resumable).

For each of 5 units (holdout multimodal / ablation; BCN20000 primary multimodal / ablation; Phase 5 site-blind, EXPLORATORY):
1,000 lesion resamples, seed 42 (all units of one dataset share the SAME resamples, so differences between models and the
holdout-minus-BCN20000 gap can be taken draw by draw). Each draw holds 51 numbers (8 headline metrics, macro-precision classes
left out, 7 classes x 6 per-class metrics); undefined = NaN, counted later, never filled. Each finished unit is saved at once
(intervals/boot_<dataset>_<run>.npz) and re-checked on a re-run (point estimate recomputed and equal), so a power cut only
costs the unit in progress.
"""

import io
import time

check_rules_note()
assert "DATA09" in globals() and "SITE_BLIND_SHA" in globals(), "Run 09_00 first (its equality gate must pass)."


def _boot_path(ds, run):
    return INT_DIR / f"boot_{ds}_{run}.npz"


def _load_unit(path):
    with np.load(path, allow_pickle=False) as f:
        return {k: f[k] for k in f.files}


def run_bootstrap_units():
    banner("PHASE 6 -- MODULE 09 (2/3): LESION BOOTSTRAP RUNS")
    INT_DIR.mkdir(exist_ok=True)
    B, seed = SETTINGS["bootstrap_resamples"], SETTINGS["bootstrap_seed"]
    boots = {}
    for ds in ("holdout", "bcn"):
        boots[ds] = LesionBootstrap(DATA09[(ds, "multimodal")][2], B, seed)
        print(f"  {ds}: {boots[ds].n_images} images, {boots[ds].n_lesions} lesions, {B} resamples, seed {seed}")
    for unit in UNITS:
        ds, run = unit
        y, P, les = DATA09[unit]
        point = metric_vector(y, P)
        path = _boot_path(ds, run)
        began = time.time()
        if path.exists():
            d = _load_unit(path)
            ok = (d["draws"].shape == (B, len(METRIC_NAMES)) and int(d["seed"]) == seed and np.allclose(d["point"], point, equal_nan=True, atol=1e-12)
                  and np.array_equal(d["counts_checksum"], boots[ds].counts.sum(axis=0)))
            if not ok:
                raise RuntimeError(f"{path.name} exists but does not match a fresh point estimate or the same resamples. Delete it yourself only if you mean to redo it.")
            status = "reused"
        else:
            draws = boots[ds].run(lambda idx: metric_vector(y[idx], P[idx]))
            buf = io.BytesIO()
            np.savez(buf, point=point, draws=draws, names=np.array(METRIC_NAMES, dtype="U"), seed=np.int64(seed), n_resamples=np.int64(B),
                     n_images=np.int64(len(y)), n_lesions=np.int64(boots[ds].n_lesions), counts_checksum=boots[ds].counts.sum(axis=0),
                     created_at=np.str_(now_iso()))
            atomic_write_bytes(path, buf.getvalue())
            d = _load_unit(path)                                          # read-back from disk
            assert np.array_equal(d["draws"], draws, equal_nan=True) and np.allclose(d["point"], point, equal_nan=True), f"Read-back of {path.name} failed"
            status = "computed"
        n_nan = int(np.isnan(d["draws"]).sum())
        print(f"  {ds:<8}{run:<12}{status:<10}{d['draws'].shape[0]} draws x {d['draws'].shape[1]} numbers; undefined entries: {n_nan}   ({time.time() - began:.0f} s)")
    print("\n  All 5 units saved. Next: run 09_02 (tables, generalisation gap, paired differences).")


run_bootstrap_units()


PHASE 6 -- MODULE 09 (2/3): LESION BOOTSTRAP RUNS
  holdout: 1432 images, 1067 lesions, 1000 resamples, seed 42
  bcn: 16284 images, 4591 lesions, 1000 resamples, seed 42
  holdout multimodal  computed  1000 draws x 51 numbers; undefined entries: 0   (24 s)
  holdout ablation    computed  1000 draws x 51 numbers; undefined entries: 0   (24 s)
  bcn     multimodal  computed  1000 draws x 51 numbers; undefined entries: 0   (60 s)
  bcn     ablation    computed  1000 draws x 51 numbers; undefined entries: 0   (60 s)
  bcn     site_blind  computed  1000 draws x 51 numbers; undefined entries: 1000   (54 s)

  All 5 units saved. Next: run 09_02 (tables, generalisation gap, paired differences).


In [43]:
"""
Phase 6 -- Module 09, cell 3 of 3: interval tables, generalisation gap and paired differences.

Requires 09_00 and 09_01.  Just run the cell (a few seconds; the result file is written once).

From the saved draws (percentile 95% interval; undefined draws dropped, counted, flagged above 5%):
  1. headline intervals (8 metrics) and per-class intervals, holdout and BCN20000, both models;
  2. the Phase 5 SITE-BLIND run (EXPLORATORY, separate file);
  3. the generalisation gap = holdout minus BCN20000, per model and metric, interval from the draw-by-draw difference
     (the two datasets are bootstrapped separately). Phase 5's "closest published figures" are printed beside it unchanged,
     as figures that are NOT like-for-like, with no pass mark;
  4. multimodal minus ablation on each dataset (macro AUC-ROC and balanced accuracy), same resamples (EXPLORATORY, descriptive).
Point estimates are the ones proven equal to Phase 5 in 09_00. Writes intervals/*.csv and intervals_result.json.
"""

import json

RULES6 = check_rules_note()
RESULT09 = INT_DIR / "intervals_result.json"
LABEL_SITE_BLIND = "EXPLORATORY: Phase 5 site-blind run (multimodal model, site = unknown for every image)"
LABEL_PAIRED = "EXPLORATORY: multimodal minus ablation on each dataset, same lesion resamples, descriptive (not a test)"
GAP_NOTE = ("Gap = holdout minus BCN20000, each dataset bootstrapped separately, interval from the draw-by-draw difference. "
            "For ECE and Brier a negative gap means the holdout value is lower (better). The published figures are the closest available, "
            "not like-for-like, and no pass mark is applied.")


def _row(ds, run, name, point, draws, status="", extra=""):
    ci = percentile_interval(draws)
    return {"Dataset": ds, "Run": run, "Status": status, "Metric": name, "Point": float(point), "CI low": ci["lo"], "CI high": ci["hi"],
            "Undefined resamples": ci["n_undefined"], "Undefined share": ci["undefined_share"],
            "Flag": "undefined share above 5%" if ci["flagged"] else "", "Note": extra}


def run_tables():
    banner("PHASE 6 -- MODULE 09 (3/3): INTERVAL TABLES, GENERALISATION GAP, PAIRED DIFFERENCES")
    if RESULT09.exists():
        saved = json.loads(RESULT09.read_text(encoding="utf-8"))
        print(f"  Already recorded {saved['recorded_at']} (written once; delete the file yourself only if you mean to redo it).")
        print(f"  Flagged rows: {saved['flagged_rows'] or 'none'}")
        return saved
    D = {unit: _load_unit(_boot_path(*unit)) for unit in UNITS}
    head, perc = [], []
    for unit in UNITS:
        ds, run = unit
        d, y = D[unit], DATA09[unit][0]
        status = "EXPLORATORY" if run == "site_blind" else ""
        for j, name in enumerate(HEADLINE_NAMES):
            extra = ""
            if j == 3:
                extra = f"macro over defined classes; at least one class left out in {float((d['draws'][:, 8] > 0).mean()):.1%} of resamples"
            head.append(_row(ds, run, name, d["point"][j], d["draws"][:, j], status, extra))
        for ci, c in enumerate(CLASSES):
            n_pos = int((y == ci).sum())
            for k, name in enumerate(PER_CLASS_NAMES):
                col = 9 + ci * 6 + k
                r = _row(ds, run, f"{c.upper()} | {name}", d["point"][col], d["draws"][:, col], status, "low sample (N < 30)" if n_pos < LOW_SAMPLE_N else "")
                r["Class"], r["Class metric"], r["N positive"] = c.upper(), name, n_pos
                perc.append(r)
    head, perc = pd.DataFrame(head), pd.DataFrame(perc)

    gap = []
    for m in ("multimodal", "ablation"):
        dh, db = D[("holdout", m)], D[("bcn", m)]
        for j, name in enumerate(HEADLINE_NAMES):
            r = _row("gap", m, name, dh["point"][j] - db["point"][j], dh["draws"][:, j] - db["draws"][:, j])
            r["Holdout"], r["BCN20000"] = float(dh["point"][j]), float(db["point"][j])
            gap.append(r)
    gap = pd.DataFrame(gap).rename(columns={"Point": "Gap (holdout minus BCN20000)", "Run": "Model"}).drop(columns=["Dataset", "Status", "Note"])

    paired = []
    for ds in ("holdout", "bcn"):
        a, b = D[(ds, "multimodal")], D[(ds, "ablation")]
        for j in (0, 1):
            paired.append(_row(ds, "multimodal minus ablation", HEADLINE_NAMES[j], a["point"][j] - b["point"][j], a["draws"][:, j] - b["draws"][:, j], "EXPLORATORY"))
    paired = pd.DataFrame(paired)

    # ---- printout -------------------------------------------------------------------------------------------------
    def cell(r):
        return f"{r['Point']:.4f} [{r['CI low']:.4f}, {r['CI high']:.4f}]"
    for ds, runs, title in (("holdout", ("multimodal", "ablation"), "HOLDOUT (1,432 images, 1,067 lesions)"),
                            ("bcn", ("multimodal", "ablation"), "BCN20000 primary (16,284 images, 4,591 lesions)"),
                            ("bcn", ("site_blind",), LABEL_SITE_BLIND)):
        sub = head[(head["Dataset"] == ds) & head["Run"].isin(runs)]
        wide = pd.DataFrame({run: [cell(r) for _, r in sub[sub["Run"] == run].iterrows()] for run in runs}, index=HEADLINE_NAMES)
        print(f"\n  {title}: point estimate [95% lesion-bootstrap interval]\n" + "\n".join("  " + l for l in wide.to_string().splitlines()))
    g = pd.DataFrame({"Model": gap["Model"], "Metric": gap["Metric"], "Holdout": gap["Holdout"].map(lambda v: f"{v:.4f}"),
                      "BCN20000": gap["BCN20000"].map(lambda v: f"{v:.4f}"),
                      "Gap [95% CI]": [f"{r['Gap (holdout minus BCN20000)']:+.4f} [{r['CI low']:+.4f}, {r['CI high']:+.4f}]" for _, r in gap.iterrows()],
                      "Undef.": gap["Undefined resamples"].astype(int)})
    print(f"\n  GENERALISATION GAP (holdout minus BCN20000)\n  {GAP_NOTE}\n" + "\n".join("  " + l for l in g.to_string(index=False).splitlines()))
    p5gap = pd.read_csv(P5RES / "bcn_gap_table.csv")
    keep = [c for c in ("Model", "Metric", "Unit", "Published figure (closest available)", "Published gap") if c in p5gap.columns]
    print("\n  Phase 5's published figures, unchanged (closest available, NOT like-for-like; no pass mark):\n" + "\n".join("  " + l for l in p5gap[keep].to_string(index=False).splitlines()))
    pr = pd.DataFrame({"Dataset": paired["Dataset"], "Metric": paired["Metric"],
                       "multimodal - ablation [95% CI]": [f"{r['Point']:+.4f} [{r['CI low']:+.4f}, {r['CI high']:+.4f}]" for _, r in paired.iterrows()]})
    print(f"\n  {LABEL_PAIRED}\n" + "\n".join("  " + l for l in pr.to_string(index=False).splitlines()))
    auc = perc[perc["Class metric"] == "AUC-ROC"]
    wide = pd.DataFrame({f"{ds}/{run}": [cell(r) for _, r in auc[(auc["Dataset"] == ds) & (auc["Run"] == run)].iterrows()] for ds, run in UNITS}, index=[c.upper() for c in CLASSES])
    print("\n  PER-CLASS AUC-ROC [95% CI] (all per-class metrics are in the CSV files; site_blind is EXPLORATORY)\n" + "\n".join("  " + l for l in wide.to_string().splitlines()))
    allrows = pd.concat([head.assign(Table="headline"), perc.assign(Table="per-class"), gap.assign(Table="gap"), paired.assign(Table="paired")], ignore_index=True)
    flagged = allrows[allrows["Flag"] != ""]
    flagged_list = [f"{r['Table']}: {r.get('Dataset', '')}/{r.get('Run', r.get('Model', ''))} {r['Metric']} ({r['Undefined share']:.1%})" for _, r in flagged.iterrows()]
    print(f"\n  Rows with an undefined-resample share above 5%: {flagged_list or 'none'}")

    # ---- save and read back ----------------------------------------------------------------------------------------
    main = head["Run"] != "site_blind"
    files = {"headline_intervals.csv": head[main], "per_class_intervals.csv": perc[perc["Run"] != "site_blind"],
             "site_blind_intervals_EXPLORATORY.csv": pd.concat([head[~main], perc[perc["Run"] == "site_blind"]], ignore_index=True),
             "gap_intervals.csv": gap, "paired_differences_EXPLORATORY.csv": paired}
    for name, df in files.items():
        df.to_csv(INT_DIR / name, index=False)
    sizes = {"headline_intervals.csv": 32, "per_class_intervals.csv": 168, "site_blind_intervals_EXPLORATORY.csv": 50, "gap_intervals.csv": 16, "paired_differences_EXPLORATORY.csv": 4}
    record = {"format_version": 1, "recorded_at": now_iso(),
              "settings": {"resamples": SETTINGS["bootstrap_resamples"], "seed": SETTINGS["bootstrap_seed"], "interval": "percentile 95%", "unit": "lesion"},
              "labels": {"site_blind": LABEL_SITE_BLIND, "paired": LABEL_PAIRED, "gap_note": GAP_NOTE},
              "site_blind_file": {"path": str(SITE_BLIND_NPZ), "sha256": SITE_BLIND_SHA, "note": "not in the Module 01 registry; hashed in Module 09"},
              "units": {f"{ds}/{run}": {"n_images": int(len(DATA09[(ds, run)][0])), "n_lesions": int(len(set(DATA09[(ds, run)][2])))} for ds, run in UNITS},
              "files": list(files), "flagged_rows": flagged_list}
    write_json_once(RESULT09, record)
    back = json.loads(RESULT09.read_text(encoding="utf-8"))
    assert all(len(pd.read_csv(INT_DIR / n)) == k for n, k in sizes.items()) and back["files"] == list(files), "Interval table read-back failed"
    print(f"\n  Saved and read back from disk: {INT_DIR}\n  Module 09 is complete. Next: Module 10 (demographic-cell intervals; first in the cut order), after the walk-through.")
    return record


INTERVALS09 = run_tables()


PHASE 6 -- MODULE 09 (3/3): INTERVAL TABLES, GENERALISATION GAP, PAIRED DIFFERENCES

  HOLDOUT (1,432 images, 1,067 lesions): point estimate [95% lesion-bootstrap interval]
                                   multimodal                 ablation
  Macro AUC-ROC       0.9348 [0.9187, 0.9499]  0.9595 [0.9507, 0.9675]
  Balanced accuracy   0.6649 [0.5985, 0.7539]  0.7293 [0.6647, 0.7906]
  Macro-F1            0.6221 [0.5564, 0.6768]  0.6917 [0.6190, 0.7347]
  Macro precision     0.6107 [0.5416, 0.6689]  0.6773 [0.6000, 0.7289]
  Macro PR-AUC        0.6822 [0.6184, 0.7527]  0.7668 [0.7026, 0.8250]
  ECE (15 bins)       0.1429 [0.1221, 0.1707]  0.1409 [0.1195, 0.1643]
  Brier (multiclass)  0.3776 [0.3343, 0.4204]  0.3424 [0.3027, 0.3822]
  Accuracy            0.7640 [0.7377, 0.7919]  0.7849 [0.7600, 0.8098]

  BCN20000 primary (16,284 images, 4,591 lesions): point estimate [95% lesion-bootstrap interval]
                                   multimodal                 ablation
  Macro AUC-ROC  

# Module 10

In [44]:
"""
Phase 6 -- Module 10, cell 1 of 2: rebuild every demographic cell and prove it equals the Phase 5 tables (no resampling yet).

Requires cells 01_00 to 09_00.  Just run the cell (about a minute). Stops (writes nothing) if any check fails.

Gives every image its age group, sex and site (holdout: HAM10000_metadata.tab + holdout_split.csv; BCN20000: bcn_metadata_encoded.csv,
BCN20000's own regions), then recomputes every cell (one class inside one subgroup) with the Phase 5 cell rule and compares it with the
Phase 5 demographic tables: counts, status, AUC-ROC, sensitivity. Sets DEMO10 for cell 10_01.
"""

import re
from pathlib import Path

check_rules_note()
assert "DATA09" in globals(), "Run cells 09_00 to 09_02 first."
HAM_TAB = Path(r"T:\Thesis - Development\Resources\Dataset\HAM10000\HAM10000_metadata.tab")
HAM_TAB_BYTES = 830428                                   # size fingerprinted in Phase 4
DEMO_DIR = OUT / "demographic_intervals"
CELL_TOL = 5e-5                                          # the Phase 5 tables may hold rounded text
AGE_LABELS = ["0-20", "21-40", "41-60", "61-79", "80+", "Unknown"]


def _norm(s):
    return re.sub(r"\s+", " ", str(s).replace("\u2013", "-").replace("\u2014", "-").strip().lower())


def _kind(axis):
    a = _norm(axis)
    for key, words in (("age", ("age",)), ("sex", ("sex",)), ("site", ("site", "region", "locat"))):
        if any(w in a for w in words):
            return key
    raise ValueError(f"Unknown demographic axis: {axis!r}")


def _holdout_groups():
    ho = pd.read_csv(P6["registry"]["p1_holdout_split"])
    tab = pd.read_csv(HAM_TAB, sep="\t").set_index("image_id")
    ids = ho["image_id"].astype(str).values
    assert set(ids) <= set(tab.index), "Some holdout images are missing from HAM10000_metadata.tab"
    t = tab.loc[ids]
    age = pd.to_numeric(t["age"], errors="coerce").values
    code = np.digitize(np.nan_to_num(age, nan=-1), [21, 41, 61, 80])            # 0-20, 21-40, 41-60, 61-79, 80+
    age_label = np.array([AGE_LABELS[5] if np.isnan(a) else AGE_LABELS[c] for a, c in zip(age, code)])
    return {"age": np.array([_norm(x) for x in age_label]), "sex": np.array([_norm(x) for x in t["sex"]]),
            "site": np.array([_norm(x) for x in ho["localization"]])}, int(np.isnan(age).sum())


def _bcn_groups():
    enc = pd.read_csv(P6["registry"]["p5_bcn_metadata_encoded"]).set_index("isic_id")
    isic = LOAD_NOTES["multimodal"][0]
    assert set(isic) <= set(enc.index), "Some BCN20000 images are missing from bcn_metadata_encoded.csv"
    t = enc.loc[isic]
    return {"age": np.array([_norm(x) for x in t["age_bin"]]), "sex": np.array([_norm(x) for x in t["sex"]]),
            "site": np.array([_norm(x) for x in t["bcn_site"]])}


GROUPS10, DEMO_CSV = {}, {}
GROUPS10["holdout"], _n_no_age = _holdout_groups()
GROUPS10["bcn"] = _bcn_groups()
for _ds in ("holdout", "bcn"):
    for _m in ("multimodal", "ablation"):
        DEMO_CSV[(_ds, _m)] = pd.read_csv(P6["registry"][f"p5_{_ds}_demographics_{_m}"])


def _recompute(ds, model):
    """Every Phase 5 cell, rebuilt here; returns (problems, worst numeric difference)."""
    y, P, les = DATA09[(ds, model)]
    pred = P.argmax(axis=1)
    problems, worst = [], 0.0
    for _, r in DEMO_CSV[(ds, model)].iterrows():
        mask = GROUPS10[ds][_kind(r["Axis"])] == _norm(r["Subgroup"])
        ci = [c.lower() for c in CLASSES].index(_norm(r["Class"]))
        yb = (y == ci)[mask]
        cm = cell_metrics(yb, P[mask, ci], (pred == ci)[mask])
        n_pos_les = int(len(set(les[mask][yb])))
        tag = f"{r['Axis']}/{r['Subgroup']}/{r['Class']}"
        if (cm["n_pos"], cm["n_neg"], n_pos_les) != (int(r["N positive"]), int(r["N negative"]), int(r["N positive lesions"])):
            problems.append(f"{tag}: counts {(cm['n_pos'], cm['n_neg'], n_pos_les)} vs Phase 5 {(int(r['N positive']), int(r['N negative']), int(r['N positive lesions']))}")
        elif cm["status"] != r["Status"]:
            problems.append(f"{tag}: status {cm['status']!r} vs {r['Status']!r}")
        else:
            for mine, theirs in ((cm["auc_roc"], to_float(r["AUC-ROC"])), (cm["sensitivity"], to_float(r["Sensitivity"]))):
                d = 0.0 if (np.isnan(mine) and np.isnan(theirs)) else abs(mine - theirs)
                worst = max(worst, d)
                if not d <= CELL_TOL:
                    problems.append(f"{tag}: value {mine} vs {theirs}")
    return problems, worst


def _t_cells(ds, model):
    problems, worst = _recompute(ds, model)
    n = len(DEMO_CSV[(ds, model)])
    return not problems, (f"{n} cells rebuilt and equal to Phase 5 (largest value difference {worst:.1e})" if not problems
                          else f"{len(problems)} of {n} cells differ; first: {problems[:3]}")


def _t_membership():
    bad = []
    for ds in ("holdout", "bcn"):
        n = len(DATA09[(ds, "multimodal")][0])
        for axis in DEMO_CSV[(ds, "multimodal")]["Axis"].unique():
            k = _kind(axis)
            listed = {_norm(s) for s in DEMO_CSV[(ds, "multimodal")].loc[DEMO_CSV[(ds, "multimodal")]["Axis"] == axis, "Subgroup"]}
            seen = set(GROUPS10[ds][k])
            covered = int(np.isin(GROUPS10[ds][k], list(listed)).sum())
            if not seen <= listed or covered != n:
                bad.append(f"{ds}/{axis}: {covered} of {n} images in a listed subgroup; unlisted labels {sorted(seen - listed)}")
    return not bad, "every image falls in exactly one listed subgroup on each axis" if not bad else "; ".join(bad)


def _t_reported():
    out, ok = [], True
    for ds, expect in (("holdout", 22), ("bcn", 75)):
        for m in ("multimodal", "ablation"):
            n = int((DEMO_CSV[(ds, m)]["Status"] == "reported").sum())
            ok &= (n == expect)
            out.append(f"{ds}/{m}: {n}")
    return ok, "reported cells (expected 22 holdout, 75 BCN20000): " + ", ".join(out)


_rows = run_checks("PHASE 6 -- MODULE 10 (1/2): DEMOGRAPHIC CELL GATE (Phase 6 rebuild vs the Phase 5 tables)", [
    ("HAM10000_metadata.tab present, size as fingerprinted in Phase 4", lambda: (HAM_TAB.exists() and HAM_TAB.stat().st_size == HAM_TAB_BYTES,
        f"{HAM_TAB.stat().st_size if HAM_TAB.exists() else 'missing'} bytes; {_n_no_age} holdout images with no recorded age (expected 9)")),
    ("Every image sits in exactly one subgroup on each axis", _t_membership),
    ("Holdout cells, multimodal", lambda: _t_cells("holdout", "multimodal")),
    ("Holdout cells, ablation", lambda: _t_cells("holdout", "ablation")),
    ("BCN20000 cells, multimodal", lambda: _t_cells("bcn", "multimodal")),
    ("BCN20000 cells, ablation", lambda: _t_cells("bcn", "ablation")),
    ("Reported-cell counts", _t_reported),
])
assert all(r["passed"] for r in _rows), "Demographic gate failed: nothing else runs. Send Claude the FAIL lines."
HAM_TAB_SHA = sha256_file(HAM_TAB)
DEMO10 = True
print(f"\n  HAM10000_metadata.tab SHA-256 (recorded in the result file): {HAM_TAB_SHA}")
print("  Gate passed. Next: run 10_01 (bootstrap and tables).")


PHASE 6 -- MODULE 10 (1/2): DEMOGRAPHIC CELL GATE (Phase 6 rebuild vs the Phase 5 tables)
  [PASS] HAM10000_metadata.tab present, size as fingerprinted in Phase 4  --  830428 bytes; 9 holdout images with no recorded age (expected 9)
  [PASS] Every image sits in exactly one subgroup on each axis  --  every image falls in exactly one listed subgroup on each axis
  [PASS] Holdout cells, multimodal  --  168 cells rebuilt and equal to Phase 5 (largest value difference 0.0e+00)
  [PASS] Holdout cells, ablation  --  168 cells rebuilt and equal to Phase 5 (largest value difference 0.0e+00)
  [PASS] BCN20000 cells, multimodal  --  112 cells rebuilt and equal to Phase 5 (largest value difference 0.0e+00)
  [PASS] BCN20000 cells, ablation  --  112 cells rebuilt and equal to Phase 5 (largest value difference 0.0e+00)
  [PASS] Reported-cell counts  --  reported cells (expected 22 holdout, 75 BCN20000): holdout/multimodal: 22, holdout/ablation: 22, bcn/multimodal: 75, bcn/ablation: 75

  7 of 7 pas

In [45]:
"""
Phase 6 -- Module 10, cell 2 of 2: lesion-bootstrap intervals for the demographic cells that Phase 5 marked 'reported'.

Requires 10_00 (its gate must have passed).  Just run the cell (a few minutes; the result file is written once).

For each subgroup that holds at least one 'reported' cell: 1,000 resamples of THAT subgroup's lesions (seed 42; both models use the
same resamples), AUC-ROC and sensitivity of each reported class, percentile 95% interval, undefined resamples counted and flagged
above 5%. Cells that are low-sample or undefined in Phase 5 get no interval and keep their Phase 5 status; AKIEC, DF and VASC
cells without a 'reported' status are additionally written 'not estimable at this sample size'.
Holdout = primary. BCN20000 = EXPLORATORY. Writes demographic_intervals/*.csv and demographic_intervals_result.json.
"""

import json

RULES6 = check_rules_note()
assert "DEMO10" in globals(), "Run 10_00 first (its gate must pass)."
RESULT10 = DEMO_DIR / "demographic_intervals_result.json"
RARE = {"akiec", "df", "vasc"}
NOT_ESTIMABLE = "not estimable at this sample size"
FILES10 = {"holdout": "demographic_intervals_holdout.csv", "bcn": "demographic_intervals_bcn_EXPLORATORY.csv"}
NO_INTERVAL_FILE = "cells_without_interval.csv"


def _fmt_ci(p, lo, hi):
    return "undefined" if np.isnan(p) else f"{p:.3f} [{lo:.3f}, {hi:.3f}]"


def run_demographic_intervals():
    banner("PHASE 6 -- MODULE 10 (2/2): DEMOGRAPHIC-CELL INTERVALS")
    if RESULT10.exists():
        saved = json.loads(RESULT10.read_text(encoding="utf-8"))
        print(f"  Already recorded {saved['recorded_at']} (written once; delete the file yourself only if you mean to redo it).")
        print(f"  Cells with an interval: {saved['n_interval_rows']}; flagged rows: {saved['flagged_rows'] or 'none'}")
        return saved
    DEMO_DIR.mkdir(exist_ok=True)
    B, seed = SETTINGS["bootstrap_resamples"], SETTINGS["bootstrap_seed"]
    cls = [c.lower() for c in CLASSES]
    out_rows, skipped = {"holdout": [], "bcn": []}, []
    for ds in ("holdout", "bcn"):
        y = DATA09[(ds, "multimodal")][0]
        les = DATA09[(ds, "multimodal")][2]
        tables = {m: DEMO_CSV[(ds, m)].reset_index(drop=True) for m in ("multimodal", "ablation")}
        ref = tables["multimodal"]
        assert ref[["Axis", "Subgroup", "Class"]].equals(tables["ablation"][["Axis", "Subgroup", "Class"]]), "Phase 5 grids differ between models"
        pred = {m: DATA09[(ds, m)][1].argmax(axis=1) for m in tables}
        for (axis, sub), grp in ref[ref["Status"] == "reported"].groupby(["Axis", "Subgroup"], sort=False):
            mask = GROUPS10[ds][_kind(axis)] == _norm(sub)
            ii = np.flatnonzero(mask)
            boot = LesionBootstrap(les[ii], B, seed)
            reps = [cls.index(_norm(c)) for c in grp["Class"]]
            yy = y[ii]
            P = {m: DATA09[(ds, m)][1][ii] for m in tables}
            hit = {m: pred[m][ii] for m in tables}

            def vec(idx):
                v = []
                for m in ("multimodal", "ablation"):
                    for ci in reps:
                        yb = (yy[idx] == ci)
                        v.append(binary_auc(yb, P[m][idx, ci]))
                        npos = int(yb.sum())
                        v.append(float((hit[m][idx][yb] == ci).sum() / npos) if npos else NAN)
                return np.array(v, dtype=np.float64)
            draws = boot.run(vec)
            for mi, m in enumerate(("multimodal", "ablation")):
                for k, ci in enumerate(reps):
                    col = (mi * len(reps) + k) * 2
                    r = tables[m][(tables[m]["Axis"] == axis) & (tables[m]["Subgroup"] == sub) & (tables[m]["Class"] == grp["Class"].iloc[k])].iloc[0]
                    a_ci, s_ci = percentile_interval(draws[:, col]), percentile_interval(draws[:, col + 1])
                    a_pt, s_pt = to_float(r["AUC-ROC"]), to_float(r["Sensitivity"])
                    flags = [n for n, c in (("AUC-ROC", a_ci), ("Sensitivity", s_ci)) if c["flagged"]]
                    out_rows[ds].append({
                        "Dataset": ds, "Status label": "EXPLORATORY" if ds == "bcn" else "primary", "Model": m, "Axis": axis, "Subgroup": sub,
                        "Class": r["Class"], "N positive": int(r["N positive"]), "N negative": int(r["N negative"]),
                        "N positive lesions": int(r["N positive lesions"]), "N lesions in subgroup": boot.n_lesions,
                        "AUC-ROC": a_pt, "AUC-ROC CI low": a_ci["lo"], "AUC-ROC CI high": a_ci["hi"],
                        "AUC-ROC undefined resamples": a_ci["n_undefined"], "AUC-ROC undefined share": a_ci["undefined_share"],
                        "Sensitivity": s_pt, "Sensitivity CI low": s_ci["lo"], "Sensitivity CI high": s_ci["hi"],
                        "Sensitivity undefined resamples": s_ci["n_undefined"], "Sensitivity undefined share": s_ci["undefined_share"],
                        "Flag": ("undefined share above 5%: " + ", ".join(flags)) if flags else "", "Phase 5 status": r["Status"]})
        for m in tables:                                     # cells that get no interval
            t = tables[m]
            for _, r in t[t["Status"] != "reported"].iterrows():
                rare = _norm(r["Class"]) in RARE
                skipped.append({"Dataset": ds, "Status label": "EXPLORATORY" if ds == "bcn" else "primary", "Model": m, "Axis": r["Axis"],
                                "Subgroup": r["Subgroup"], "Class": r["Class"], "N positive": int(r["N positive"]), "N negative": int(r["N negative"]),
                                "Tier": r["Tier"], "Phase 5 status": r["Status"], "Interval": NOT_ESTIMABLE if rare else "none (Phase 5 flag kept)"})
    frames = {ds: pd.DataFrame(rows) for ds, rows in out_rows.items()}
    skipped = pd.DataFrame(skipped)

    # ---- printout -------------------------------------------------------------------------------------------------
    print("  Cells are one class inside one subgroup. Point estimate [95% lesion-bootstrap interval]. Never a macro value inside a subgroup.")
    for ds, title in (("holdout", "HOLDOUT (primary): 22 reported cells per model"), ("bcn", "BCN20000 (EXPLORATORY): 75 reported cells per model")):
        f = frames[ds]
        mm, ab = f[f["Model"] == "multimodal"].reset_index(drop=True), f[f["Model"] == "ablation"].reset_index(drop=True)
        wide = pd.DataFrame({"Axis": mm["Axis"], "Subgroup": mm["Subgroup"], "Class": mm["Class"],
                             "N pos/lesions": [f"{a}/{b}" for a, b in zip(mm["N positive"], mm["N positive lesions"])],
                             "AUC multimodal": [_fmt_ci(r["AUC-ROC"], r["AUC-ROC CI low"], r["AUC-ROC CI high"]) for _, r in mm.iterrows()],
                             "AUC ablation": [_fmt_ci(r["AUC-ROC"], r["AUC-ROC CI low"], r["AUC-ROC CI high"]) for _, r in ab.iterrows()],
                             "Sens multimodal": [_fmt_ci(r["Sensitivity"], r["Sensitivity CI low"], r["Sensitivity CI high"]) for _, r in mm.iterrows()],
                             "Sens ablation": [_fmt_ci(r["Sensitivity"], r["Sensitivity CI low"], r["Sensitivity CI high"]) for _, r in ab.iterrows()]})
        print(f"\n  {title}\n" + "\n".join("  " + l for l in wide.to_string(index=False).splitlines()))
    for ds in ("holdout", "bcn"):
        s = skipped[(skipped["Dataset"] == ds) & (skipped["Model"] == "multimodal")]
        print(f"\n  {ds}: {len(s)} cells per model get no interval. By Phase 5 status: {s['Phase 5 status'].value_counts().to_dict()}; "
              f"AKIEC/DF/VASC written '{NOT_ESTIMABLE}': {int((s['Interval'] == NOT_ESTIMABLE).sum())}")
    allrows = pd.concat(frames.values(), ignore_index=True)
    flagged_list = [f"{r['Dataset']}/{r['Model']} {r['Axis']}/{r['Subgroup']}/{r['Class']} ({r['Flag']})" for _, r in allrows[allrows["Flag"] != ""].iterrows()]
    print(f"\n  Rows with an undefined-resample share above 5%: {flagged_list or 'none'}")
    n_few = int((allrows["N positive lesions"] < CELL_MIN).sum())
    print(f"  Information: {n_few} of {len(allrows)} interval rows rest on fewer than {CELL_MIN} positive lesions (cells are tiered by images, as in Phase 5).")

    # ---- save and read back ----------------------------------------------------------------------------------------
    for ds, df in frames.items():
        df.to_csv(DEMO_DIR / FILES10[ds], index=False)
    skipped.to_csv(DEMO_DIR / NO_INTERVAL_FILE, index=False)
    sizes = {FILES10["holdout"]: 44, FILES10["bcn"]: 150, NO_INTERVAL_FILE: int(len(skipped))}
    record = {"format_version": 1, "recorded_at": now_iso(),
              "settings": {"resamples": B, "seed": seed, "interval": "percentile 95%", "unit": "lesions of the cell's subgroup",
                           "metrics": ["AUC-ROC", "Sensitivity"], "models_share_resamples": True},
              "labels": {"holdout": "primary", "bcn": "EXPLORATORY: BCN20000 demographic-cell intervals"},
              "ham_metadata_tab": {"path": str(HAM_TAB), "bytes": HAM_TAB_BYTES, "sha256": HAM_TAB_SHA},
              "n_interval_rows": int(len(allrows)), "files": list(sizes), "flagged_rows": flagged_list,
              "not_estimable_wording": NOT_ESTIMABLE}
    write_json_once(RESULT10, record)
    back = json.loads(RESULT10.read_text(encoding="utf-8"))
    assert all(len(pd.read_csv(DEMO_DIR / n)) == k for n, k in sizes.items()) and back["files"] == list(sizes), "Demographic interval read-back failed"
    print(f"\n  Saved and read back from disk: {DEMO_DIR}\n  Module 10 is complete. Next: Module 11 (results assembly), after the walk-through.")
    return record


DEMO_INTERVALS = run_demographic_intervals()


PHASE 6 -- MODULE 10 (2/2): DEMOGRAPHIC-CELL INTERVALS
  Cells are one class inside one subgroup. Point estimate [95% lesion-bootstrap interval]. Never a macro value inside a subgroup.

  HOLDOUT (primary): 22 reported cells per model
             Axis        Subgroup Class N pos/lesions       AUC multimodal         AUC ablation      Sens multimodal        Sens ablation
        Age group           41-60   BKL         58/37 0.911 [0.837, 0.960] 0.931 [0.863, 0.977] 0.776 [0.636, 0.915] 0.759 [0.615, 0.889]
        Age group           41-60   MEL         72/37 0.898 [0.837, 0.945] 0.928 [0.885, 0.961] 0.722 [0.568, 0.857] 0.681 [0.540, 0.811]
        Age group           41-60    NV       438/380 0.951 [0.929, 0.969] 0.954 [0.934, 0.971] 0.826 [0.780, 0.873] 0.872 [0.833, 0.909]
        Age group           61-79   BCC         45/28 0.900 [0.827, 0.967] 0.967 [0.934, 0.991] 0.622 [0.439, 0.806] 0.800 [0.634, 0.950]
        Age group           61-79   BKL         55/39 0.868 [0.804, 0.927]

# Module 11

In [51]:
"""
Phase 6 -- Module 11, cell 1 of 3: assemble the final tables (no new numbers).

Requires cells 01_00 to 10_01.  Just run the cell (a few seconds; the record is written once).

Copies and combines the finished result files into assembly/ and checks them against their sources:
  table_1_holdout_with_intervals.csv     (headline + per-class, both models)
  table_2_bcn20000_with_intervals.csv    (headline + per-class, both models)  + table_2b_generalisation_gap.csv + table_2c_published_figures_phase5.csv
  table_3_xai_validation_phase4.csv      (Phase 4 Claim 2 comparison, unchanged)
  table_4_demographics.csv               (all 560 Phase 5 cells, intervals on the 'reported' ones only)
  table_5_model_comparisons.csv          (Claim 1 primary / fold-2 / holdout check / ResNet50 / ViT-B/16, each with its status label)
  table_5b_phase2_reference.csv          (locked Phase 2 baseline rows)
"""

import json

RULES6 = check_rules_note()
ASM_DIR = OUT / "assembly"
RESULT11A = ASM_DIR / "assembly_tables_record.json"
NOT_EST = "not estimable at this sample size"
SRC = {"headline": OUT / "intervals" / "headline_intervals.csv", "perclass": OUT / "intervals" / "per_class_intervals.csv",
       "gap": OUT / "intervals" / "gap_intervals.csv", "claim1": OUT / "claim1" / "claim1_result.json",
       "c1_primary": OUT / "claim1" / "claim1_primary_table.csv", "c1_fold2": OUT / "claim1" / "claim1_fold2_sensitivity_table.csv",
       "hold_json": OUT / "holdout_check" / "holdout_check_result.json", "hold_tab": OUT / "holdout_check" / "holdout_check_table.csv",
       "base_json": OUT / "baselines" / "baseline_comparisons_result.json",
       "resnet": OUT / "baselines" / "multimodal_vs_resnet50_table.csv", "vit": OUT / "baselines" / "multimodal_vs_vit_b_16_table.csv",
       "di_hold": OUT / "demographic_intervals" / "demographic_intervals_holdout.csv", "di_bcn": OUT / "demographic_intervals" / "demographic_intervals_bcn_EXPLORATORY.csv",
       "p5_gap": P6["registry"]["p5_bcn_gap_table"], "p4_claim2": P6["registry"]["p4_claim2_paired_comparison"]}
RARE_CLASSES = {"akiec", "df", "vasc"}
UNCHANGED_UPSTREAM = {"table_2c_published_figures_phase5.csv", "table_3_xai_validation_phase4.csv"}      # copied as they are


def _build():
    S = {k: (pd.read_csv(v) if str(v).endswith(".csv") else json.loads(v.read_text(encoding="utf-8"))) for k, v in SRC.items()}
    head, perc = S["headline"], S["perclass"]
    t1 = pd.concat([head[head["Dataset"] == "holdout"].assign(Table="headline"), perc[perc["Dataset"] == "holdout"].assign(Table="per-class")], ignore_index=True)
    t2 = pd.concat([head[head["Dataset"] == "bcn"].assign(Table="headline"), perc[perc["Dataset"] == "bcn"].assign(Table="per-class")], ignore_index=True)
    t2["Status"] = "EXPLORATORY: BCN20000 is external validation, not the primary verdict set"
    # demographics
    parts = []
    for ds, label, key in (("holdout", "HAM10000 holdout (primary)", "di_hold"), ("bcn", "BCN20000 (EXPLORATORY)", "di_bcn")):
        iv = S[key]
        take = ["Model", "Axis", "Subgroup", "Class", "AUC-ROC CI low", "AUC-ROC CI high", "AUC-ROC undefined resamples",
                "Sensitivity CI low", "Sensitivity CI high", "Sensitivity undefined resamples", "N lesions in subgroup"]
        for m in ("multimodal", "ablation"):
            base = pd.read_csv(P6["registry"][f"p5_{ds}_demographics_{m}"])
            merged = base.merge(iv[iv["Model"] == m][take].drop(columns=["Model"]), on=["Axis", "Subgroup", "Class"], how="left")
            merged.insert(0, "Set", label)
            merged.insert(1, "Run", m)
            merged["Interval note"] = [("95% lesion-bootstrap interval" if s == "reported" else NOT_EST if str(c).strip().lower() in RARE_CLASSES
                                        else "none (Phase 5 flag kept)") for s, c in zip(merged["Status"], merged["Class"])]
            parts.append(merged)
    t4 = pd.concat(parts, ignore_index=True)
    # model comparisons
    c1, hc, bl = S["claim1"], S["hold_json"], S["base_json"]
    lab = {"primary": "VERDICT: Claim 1 on pooled out-of-fold predictions", "fold2": c1["sensitivity"]["label"], "hold": hc["label"], "base": bl["label"]}
    stack = []
    for name, tab, status in (("multimodal vs ablation (Claim 1, pooled OOF)", S["c1_primary"], lab["primary"]),
                              ("multimodal vs ablation, fold 2 removed", S["c1_fold2"], lab["fold2"]),
                              ("multimodal vs ablation, holdout check", S["hold_tab"], lab["hold"]),
                              ("multimodal vs ResNet50", S["resnet"], lab["base"]), ("multimodal vs ViT-B/16", S["vit"], lab["base"])):
        t = tab.copy()
        t.insert(0, "Comparison", name)
        t.insert(1, "Status", status)
        stack.append(t)
    t5 = pd.concat(stack, ignore_index=True)
    t5b = pd.DataFrame([{"Model": k, "Macro AUC-ROC mean": v["mean"], "SD over 5 folds": v["sd"], "Status": "locked Phase 2 value, reference only"}
                        for k, v in bl["reference_rows_phase2_locked"].items()])
    return S, {"table_1_holdout_with_intervals.csv": t1, "table_2_bcn20000_with_intervals.csv": t2, "table_2b_generalisation_gap.csv": S["gap"],
               "table_2c_published_figures_phase5.csv": S["p5_gap"], "table_3_xai_validation_phase4.csv": S["p4_claim2"],
               "table_4_demographics.csv": t4, "table_5_model_comparisons.csv": t5, "table_5b_phase2_reference.csv": t5b}


def run_assembly():
    banner("PHASE 6 -- MODULE 11 (1/3): ASSEMBLE THE TABLES")
    if RESULT11A.exists():
        saved = json.loads(RESULT11A.read_text(encoding="utf-8"))
        print(f"  Already recorded {saved['recorded_at']} (written once; delete the file yourself only if you mean to redo it).")
        for f, e in saved["files"].items():
            print(f"  {f}: {e['rows']} rows")
        return saved
    missing = [k for k, v in SRC.items() if not v.exists()]
    if missing:
        raise RuntimeError(f"Result files missing: {missing}. Run the earlier modules first.")
    S, T = _build()

    def _t_counts():
        n = {k: len(v) for k, v in T.items()}
        ok = (n["table_1_holdout_with_intervals.csv"] == 100 and n["table_2_bcn20000_with_intervals.csv"] == 100 and n["table_2b_generalisation_gap.csv"] == 16
              and n["table_3_xai_validation_phase4.csv"] == EXPECTED["claim2_comparison_rows"] and n["table_4_demographics.csv"] == 560
              and n["table_5_model_comparisons.csv"] == 40 and n["table_5b_phase2_reference.csv"] == 3)
        return ok, ", ".join(f"{k.split('.')[0]}: {v}" for k, v in n.items())

    def _t_gap():
        g, h = S["gap"], S["headline"]
        worst = 0.0
        for _, r in g.iterrows():
            ho = h[(h["Dataset"] == "holdout") & (h["Run"] == r["Model"]) & (h["Metric"] == r["Metric"])]["Point"].iloc[0]
            bc = h[(h["Dataset"] == "bcn") & (h["Run"] == r["Model"]) & (h["Metric"] == r["Metric"])]["Point"].iloc[0]
            worst = max(worst, abs(r["Gap (holdout minus BCN20000)"] - (ho - bc)), abs(r["Holdout"] - ho), abs(r["BCN20000"] - bc))
        return worst < 1e-12, f"all 16 gaps equal holdout minus BCN20000 recomputed from the saved headline table (largest difference {worst:.1e})"

    def _t_demo():
        t4 = T["table_4_demographics.csv"]
        rep = t4["Status"] == "reported"
        has = t4["AUC-ROC CI low"].notna() | t4["Sensitivity CI low"].notna()
        ok = int(rep.sum()) == 194 and bool((has == rep).all()) and int(t4["Interval note"].eq(NOT_EST).sum()) == 72 * 2 + 22 * 2
        return ok, (f"560 cells; {int(rep.sum())} reported cells have an interval and no other cell has one; "
                    f"{int(t4['Interval note'].eq(NOT_EST).sum())} AKIEC/DF/VASC cells without a 'reported' status read '{NOT_EST}'")

    def _t_compare():
        t5 = T["table_5_model_comparisons.csv"]
        prim = t5[t5["Comparison"].str.contains("Claim 1")]
        ok = prim["Reading"].iloc[:7].tolist() == S["c1_primary"]["Reading"].iloc[:7].tolist() and t5["Comparison"].nunique() == 5
        return ok, f"5 comparisons x 8 rows; every row carries its status label; Claim 1 primary category (from the result file): {S['claim1']['primary']['category']}"

    def _t_wording():
        hits = [f for f, df in T.items() if f not in UNCHANGED_UPSTREAM and any(isinstance(v, str) and "significan" in v.lower() for v in df.astype(object).values.ravel())]
        return not hits, "no table we built uses 'significant' (every comparison row carries its delta, CI and Holm p in the same row)" if not hits else f"found in {hits}"

    rows = run_checks("MODULE 11 (1/3): TABLE CHECKS", [("Row counts of every table", _t_counts), ("Gap arithmetic from the saved tables", _t_gap),
                                                    ("Demographic table: intervals only on reported cells", _t_demo), ("Model comparison table", _t_compare),
                                                    ("Wording: no 'significant' in any table", _t_wording)])
    if not all(r["passed"] for r in rows):
        print("\n  STOP: nothing was written. Send Claude the FAIL lines.")
        return None
    ASM_DIR.mkdir(exist_ok=True)
    files = {}
    for name, df in T.items():
        atomic_write_bytes(ASM_DIR / name, df.to_csv(index=False).encode("utf-8"))
        back = pd.read_csv(ASM_DIR / name)
        assert back.shape == df.shape, f"Read-back of {name} failed"
        files[name] = {"rows": int(len(df)), "columns": int(df.shape[1]), "sha256": sha256_file(ASM_DIR / name)}
    sources = {k: sha256_file(v) for k, v in SRC.items()}
    write_json_once(RESULT11A, {"format_version": 1, "recorded_at": now_iso(), "files": files, "source_sha256": sources,
                                "checks": rows, "primary_category": S["claim1"]["primary"]["category"]})
    print(f"\n  Tables written and read back from disk: {ASM_DIR}")
    for f, e in files.items():
        print(f"    {f}: {e['rows']} rows x {e['columns']} columns")
    print("\n  Phase 4 Claim 2 comparison, unchanged (Table 3):\n" + "\n".join("  " + l for l in T["table_3_xai_validation_phase4.csv"].to_string(index=False).splitlines()))
    print("\n  Next: run 11_01 (literature table, success criteria, Claim 1 sentences).")
    return json.loads(RESULT11A.read_text(encoding="utf-8"))


ASSEMBLY11 = run_assembly()


PHASE 6 -- MODULE 11 (1/3): ASSEMBLE THE TABLES

MODULE 11 (1/3): TABLE CHECKS
  [PASS] Row counts of every table  --  table_1_holdout_with_intervals: 100, table_2_bcn20000_with_intervals: 100, table_2b_generalisation_gap: 16, table_2c_published_figures_phase5: 6, table_3_xai_validation_phase4: 17, table_4_demographics: 560, table_5_model_comparisons: 40, table_5b_phase2_reference: 3
  [PASS] Gap arithmetic from the saved tables  --  all 16 gaps equal holdout minus BCN20000 recomputed from the saved headline table (largest difference 1.7e-16)
  [PASS] Demographic table: intervals only on reported cells  --  560 cells; 194 reported cells have an interval and no other cell has one; 188 AKIEC/DF/VASC cells without a 'reported' status read 'not estimable at this sample size'
  [PASS] Model comparison table  --  5 comparisons x 8 rows; every row carries its status label; Claim 1 primary category (from the result file): supported
  [PASS] Wording: no 'significant' in any table  --  no table

In [52]:
"""
Phase 6 -- Module 11, cell 2 of 3: literature table, success criteria, stretch goals, filled Claim 1 sentences.

Requires 11_00.  Just run the cell (a few seconds; the record is written once).

Everything is read from the finished files; nothing is re-run. Claims 3 and 4 are written numbers first, criterion second ('met' is never a
headline). Claim 2 is quoted from the Phase 4 table (checked against it). The Claim 1 sentence is the one the rules note pre-wrote for the
category the rule produced; only its placeholders are filled. Writes assembly/literature_table.csv, success_criteria.csv,
stretch_goals.csv and criteria_and_literature_record.json.
"""

import json

RULES6 = check_rules_note()
assert RESULT11A.exists(), "Run 11_00 first."
RESULT11B = ASM_DIR / "criteria_and_literature_record.json"
LITERATURE = [
    ("Javaid et al. (2025)", "Grad-CAM Dice / IoU", "83.0% / 71.7%",
     "Closest published figure, different protocol: 28x28 resolution; mean of seven per-class values (their Table 8); the paper does not state which mask set, how many images or how the heatmap was binarised. Compared with this thesis's primary final-layer results only"),
    ("Saeed et al. (2025)", "Metadata-fusion AUC", "93.2% / 91.1%",
     "Different datasets (ISIC 2018/2019 vs HAM10000); visual Grad-CAM only, no quantitative validation"),
    ("Gamage et al. (2024)", "XAI IoU", "69.47%",
     "Context only (different quantity): bounding-box localisation score against boxes from ISIC 2017 masks; binary melanoma-vs-nevus ViT"),
    ("Ou et al. (2022)", "Metadata AUC gain", "0.901 -> 0.947",
     "Different dataset (PAD-UFES-20); dual-attention vs MetaBlock; caveat narrowed, not removed"),
]
LIT_NOTE = ("Every row is the closest published figure, not a benchmark, and no row is ranked. The 'same masks as Javaid and Gamage' statement is not supported "
            "(Phase 4 close-out). Combalia et al. is the Phase 5 anchor for the generalisation gap, not a literature-table row.")
CLAIM2_TARGETS = {"multimodal Dice 48.0": 48.0, "image-only Dice 62.6": 62.6, "difference -14.62": -14.62, "CI low -16.28": -16.28, "CI high -12.92": -12.92}


def _fill(template, summary, table):
    """Fills only the placeholders of a pre-written sentence from a finished table."""
    t = table.iloc[:7].set_index("Class")

    def vals(names):
        return ", ".join(f"{t.loc[n, 'Delta (A - B)']:+.4f}" for n in names)

    def cis(names):
        return ", ".join(f"[{t.loc[n, 'CI low']:+.4f}, {t.loc[n, 'CI high']:+.4f}]" for n in names)
    s, sup, opp, nd = template, summary["supports"], summary["opposes"], summary["not_detectable"]
    for ph, rep in (("[K₁]", str(summary["k1"])),):
        s = s.replace(ph, rep)
    if summary["category"] == "supported":
        s = s.replace("[names]", ", ".join(sup), 1).replace("[values]", vals(sup), 1).replace("[values]", cis(sup), 1)
    elif summary["category"] == "mixed":
        for names in (sup, opp, nd):
            s = s.replace("[names]", ", ".join(names) or "none", 1)
    elif summary["category"] == "opposed":
        s = s.replace("[names]", ", ".join(opp), 1)
    return s.replace("Table X", "Table 5")


def _contains_all(df, targets):
    nums = pd.to_numeric(pd.Series(df.astype(object).values.ravel()), errors="coerce").dropna().values
    found = {}
    for label, v in targets.items():
        tol = 0.06 if abs(v) > 20 else 0.006
        found[label] = bool(np.any(np.abs(nums - v) < tol) or np.any(np.abs(nums * 100 - v) < tol))
    return found


def run_criteria():
    banner("PHASE 6 -- MODULE 11 (2/3): LITERATURE TABLE, SUCCESS CRITERIA, CLAIM 1 SENTENCES")
    if RESULT11B.exists():
        saved = json.loads(RESULT11B.read_text(encoding="utf-8"))
        print(f"  Already recorded {saved['recorded_at']} (written once; delete the file yourself only if you mean to redo it).")
        print(f"  Claim 1 primary sentence: {saved['claim1_sentences']['primary']['sentence']}")
        return saved
    c1 = json.loads((OUT / "claim1" / "claim1_result.json").read_text(encoding="utf-8"))
    hc = json.loads((OUT / "holdout_check" / "holdout_check_result.json").read_text(encoding="utf-8"))
    bl = json.loads((OUT / "baselines" / "baseline_comparisons_result.json").read_text(encoding="utf-8"))
    gap = pd.read_csv(ASM_DIR / "table_2b_generalisation_gap.csv")
    cl2 = pd.read_csv(ASM_DIR / "table_3_xai_validation_phase4.csv")
    cats = RULES6["claim1"]["categories"]
    tabs = {"primary": pd.read_csv(OUT / "claim1" / "claim1_primary_table.csv"), "fold2": pd.read_csv(OUT / "claim1" / "claim1_fold2_sensitivity_table.csv"),
            "holdout": pd.read_csv(OUT / "holdout_check" / "holdout_check_table.csv")}
    summ = {"primary": c1["primary"], "fold2": c1["sensitivity"], "holdout": hc["summary"]}
    labels = {"primary": "VERDICT: pooled out-of-fold predictions", "fold2": c1["sensitivity"]["label"], "holdout": hc["label"]}
    sentences = {k: {"label": labels[k], "category": summ[k]["category"], "reported_as": summ[k]["reported_as"],
                     "sentence": _fill(cats[summ[k]["category"]]["sentence"], summ[k], tabs[k])} for k in summ}

    g = gap.set_index(["Model", "Metric"])
    G = lambda m, k: g.loc[(m, k)]
    ci = lambda m, k: f"{G(m, k)['Gap (holdout minus BCN20000)']:+.4f} [{G(m, k)['CI low']:+.4f}, {G(m, k)['CI high']:+.4f}]"
    c2found = _contains_all(cl2, CLAIM2_TARGETS)
    claim2_measured = ("Phase 4 (quoted unchanged): multimodal Dice 48.0 vs image-only 62.6 on the primary layer; difference -14.62 [-16.28, -12.92] (Otsu measure); "
                       "not detectable on the top-50% measure" if all(c2found.values()) else "see Table 3 (Phase 4 Claim 2 comparison, unchanged)")
    t4 = pd.read_csv(ASM_DIR / "table_4_demographics.csv")
    dem = {}
    for key, label in (("holdout", "HAM10000 holdout (primary)"), ("bcn", "BCN20000 (EXPLORATORY)")):
        d = t4[(t4["Set"] == label) & (t4["Run"] == "multimodal")]
        dem[key] = {"n": len(d), "rep": int((d["Status"] == "reported").sum()), "low": int(d["Status"].str.startswith("low sample").sum()),
                    "und": int(d["Status"].str.startswith("undefined").sum()),
                    "rare_reported": int(((d["Status"] == "reported") & d["Class"].str.lower().isin(RARE_CLASSES)).sum())}
    base = bl["comparisons"]
    criteria = pd.DataFrame([
        {"Claim": "1 Metadata fusion", "Criterion (Methodology 9.1)": "Per-class AUC-ROC difference, multimodal minus matched ablation, Holm-corrected DeLong and bootstrap CI both required, with delta and CI",
         "Measured": f"Primary: {sentences['primary']['category']}; {summ['primary']['reported_as']}. Fold-2 sensitivity (EXPLORATORY): {sentences['fold2']['category']}. Holdout check (EXPLORATORY): {sentences['holdout']['category']}",
         "State": "Reported as the rule gives it; the category describes counts and is not a pass or fail"},
        {"Claim": "2 XAI validation", "Criterion (Methodology 9.1)": "Per-class, per-layer Dice/IoU with N and flag; multimodal at least image-only on the primary layer",
         "Measured": claim2_measured, "State": "LOWER (Phase 4), reported plainly; nothing re-run"},
        {"Claim": "3 Generalisation", "Criterion (Methodology 9.1)": "BCN20000 evaluated; gap compared with both anchors",
         "Measured": (f"Macro AUC-ROC gap {ci('multimodal', 'Macro AUC-ROC')} multimodal and {ci('ablation', 'Macro AUC-ROC')} ablation (anchor 0.075); "
                      f"balanced-accuracy gap {G('multimodal', 'Balanced accuracy')['Gap (holdout minus BCN20000)'] * 100:.1f} and {G('ablation', 'Balanced accuracy')['Gap (holdout minus BCN20000)'] * 100:.1f} points (anchors 23.2 and 22.3; closest published figures, not like-for-like)"),
         "State": "Criterion satisfied by measuring and reporting; no pass mark"},
        {"Claim": "4 Demographic disaggregation", "Criterion (Methodology 9.1)": "Every cell reported under the two-tier system; not skin tone",
         "Measured": (f"Holdout {dem['holdout']['n']} cells per model: {dem['holdout']['rep']} reported, {dem['holdout']['low']} low sample, {dem['holdout']['und']} undefined; "
                      f"BCN20000 {dem['bcn']['n']}: {dem['bcn']['rep']}, {dem['bcn']['low']}, {dem['bcn']['und']}. "
                      + ("AKIEC, DF and VASC never reach 30 positives in any holdout subgroup. " if dem["holdout"]["rare_reported"] == 0 else "")
                      + f"Intervals on the {dem['holdout']['rep']} and {dem['bcn']['rep']} reported cells only"),
         "State": "Criterion satisfied by counting and reporting every cell; skin tone not addressed"}])
    gmm = lambda m, k: G(m, k)["Gap (holdout minus BCN20000)"]
    stretch = pd.DataFrame([
        {"Stretch goal": "Dice/IoU above Javaid (83.0% / 71.7%)", "State": "Not met, and not like-for-like (different protocol)", "Source": "Phase 4 table and literature table"},
        {"Stretch goal": "Intermediate layer above final layer", "State": "Not seen (mid-layer scores lower on all four scores)", "Source": "Phase 4 close-out"},
        {"Stretch goal": "BCN20000 gap smaller than Combalia's",
         "State": f"Not met for either model (AUC-ROC gap {gmm('multimodal', 'Macro AUC-ROC'):.3f} and {gmm('ablation', 'Macro AUC-ROC'):.3f} vs 0.075; balanced accuracy {gmm('multimodal', 'Balanced accuracy') * 100:.1f} and {gmm('ablation', 'Balanced accuracy') * 100:.1f} points vs 23.2 and 22.3); closest published figures, not like-for-like",
         "Source": "Module 09"},
        {"Stretch goal": "Multimodal outperforms both baselines",
         "State": f"Not seen: ResNet50 higher in {len(base['multimodal_vs_resnet50']['resnet50_higher'])} of 7 classes and ViT-B/16 higher in {len(base['multimodal_vs_vit_b_16']['vit_b_16_higher'])} of 7; multimodal higher in 0 of 7 against either (SECONDARY; recipes differ)",
         "Source": "Module 08"},
        {"Stretch goal": "B4 alongside B3", "State": "Not run (Ghani, 2026-10-09; B3 locked outright)", "Source": "Planning"}])
    lit = pd.DataFrame(LITERATURE, columns=["Paper", "Compared on", "Their number", "Caveat to carry"])

    def _t_sentences():
        bad = [k for k, v in sentences.items() if any(x in v["sentence"] for x in ("[names]", "[values]", "[K", "Table X"))]
        return not bad, "no placeholder left in the three Claim 1 sentences" if not bad else f"placeholders left in {bad}"

    def _t_cat():
        return (sentences["primary"]["category"] == c1["primary"]["category"] and all(v["sentence"].startswith(tuple("IThNo")) for v in sentences.values())), \
            f"categories: primary {sentences['primary']['category']}, fold-2 {sentences['fold2']['category']}, holdout {sentences['holdout']['category']}"

    def _t_lit():
        text = " ".join(" ".join(r) for r in LITERATURE).lower() + LIT_NOTE.lower().replace("not a benchmark", "")
        return len(lit) == 4 and "benchmark" not in text and "state-of-the-art" not in text and "context only" in lit.iloc[2]["Caveat to carry"].lower(), \
            "4 rows; none called a benchmark or ranked; Gamage marked context only"

    def _t_c2():
        return True, ("Phase 4 Claim 2 numbers found in Table 3: " + ", ".join(f"{k}: {'yes' if v else 'NO'}" for k, v in c2found.items())
                      + ("" if all(c2found.values()) else "  (the digest will point to Table 3 instead of quoting them)"))

    rows = run_checks("MODULE 11 (2/3): CHECKS", [("Claim 1 sentences filled, no placeholder left", _t_sentences), ("Categories match the Claim 1 result file", _t_cat),
                                                  ("Literature table wording", _t_lit), ("Claim 2 numbers quoted from Phase 4 are in its table", _t_c2)])
    if not all(r["passed"] for r in rows):
        print("\n  STOP: nothing was written. Send Claude the FAIL lines.")
        return None
    for name, df in (("literature_table.csv", lit), ("success_criteria.csv", criteria), ("stretch_goals.csv", stretch)):
        atomic_write_bytes(ASM_DIR / name, df.to_csv(index=False).encode("utf-8"))
        assert len(pd.read_csv(ASM_DIR / name)) == len(df), f"Read-back of {name} failed"
    record = {"format_version": 1, "recorded_at": now_iso(), "claim1_sentences": sentences, "claim2_quote_found": c2found, "claim2_measured": claim2_measured,
              "literature_note": LIT_NOTE, "files": {n: sha256_file(ASM_DIR / n) for n in ("literature_table.csv", "success_criteria.csv", "stretch_goals.csv")}, "checks": rows}
    write_json_once(RESULT11B, record)
    for k in ("primary", "fold2", "holdout"):
        print(f"\n  Claim 1 sentence, {sentences[k]['label']}\n  {sentences[k]['sentence']}")
    print("\n  Success criteria:\n" + "\n".join(f"  {r['Claim']}: {r['Measured']}\n      -> {r['State']}" for _, r in criteria.iterrows()))
    print("\n  Stretch goals, as they stand:\n" + "\n".join(f"  {r['Stretch goal']}: {r['State']}" for _, r in stretch.iterrows()))
    print(f"\n  {LIT_NOTE}")
    print("\n  Saved and read back from disk. Next: run 11_02 (results digest).")
    return record


CRITERIA11 = run_criteria()


PHASE 6 -- MODULE 11 (2/3): LITERATURE TABLE, SUCCESS CRITERIA, CLAIM 1 SENTENCES

MODULE 11 (2/3): CHECKS
  [PASS] Claim 1 sentences filled, no placeholder left  --  no placeholder left in the three Claim 1 sentences
  [PASS] Categories match the Claim 1 result file  --  categories: primary supported, fold-2 opposed, holdout opposed
  [PASS] Literature table wording  --  4 rows; none called a benchmark or ranked; Gamage marked context only
  [PASS] Claim 2 numbers quoted from Phase 4 are in its table  --  Phase 4 Claim 2 numbers found in Table 3: multimodal Dice 48.0: yes, image-only Dice 62.6: yes, difference -14.62: yes, CI low -16.28: yes, CI high -12.92: yes

  4 of 4 passed

  Claim 1 sentence, VERDICT: pooled out-of-fold predictions
  In this setup, adding age, sex and site via MetaBlock was associated with higher AUC-ROC in 2 of 7 classes (DF, NV, ΔAUC +0.0499, +0.0120, 95% CI [+0.0134, +0.0876], [+0.0051, +0.0186]) and no detectable difference in the rest.

  Claim 1 sentence

In [53]:
"""
Phase 6 -- Module 11, cell 3 of 3: write results_digest.md and check it against the files it quotes.

Requires 11_00 and 11_01.  Just run the cell (a few seconds; the digest is written once).

The digest is built only from the finished result files (no number is typed in by hand except the fixed anchors 0.075 / 23.2 / 22.3).
Claims 1 and 2 are two separate findings with one cross-reference sentence; Claims 3 and 4 are numbers first, criterion second; 'met' is
never a headline. Before writing, the files it quotes are re-hashed against the records of 11_00 and 11_01. Writes
assembly/results_digest.md and assembly/results_digest.sha256.json.
"""

import json
import re

RULES6 = check_rules_note()
assert RESULT11A.exists() and RESULT11B.exists(), "Run 11_00 and 11_01 first."
DIGEST = ASM_DIR / "results_digest.md"
DIGEST_SHA = ASM_DIR / "results_digest.sha256.json"
CROSS_REF = ("Claims 1 and 2 are two separate findings, each stated on its own measure (AUC-ROC for Claim 1; Dice/IoU against the lesion mask for Claim 2). "
             "Both compare the multimodal model with the same matched image-only ablation and both involve the MetaBlock metadata branch; "
             "no link between them was tested, and any joint explanation is a hypothesis.")


def _md(df):
    cols = [str(c) for c in df.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    lines += ["| " + " | ".join(str(v) for v in row) + " |" for row in df.astype(object).values.tolist()]
    return "\n".join(lines)


def _build_digest():
    A = {n.split(".")[0]: pd.read_csv(ASM_DIR / n) for n in json.loads(RESULT11A.read_text(encoding="utf-8"))["files"]}
    rec_b = json.loads(RESULT11B.read_text(encoding="utf-8"))
    crit, stretch, lit = (pd.read_csv(ASM_DIR / f) for f in ("success_criteria.csv", "stretch_goals.csv", "literature_table.csv"))
    sent = rec_b["claim1_sentences"]
    bl = json.loads((OUT / "baselines" / "baseline_comparisons_result.json").read_text(encoding="utf-8"))
    paired = pd.read_csv(OUT / "intervals" / "paired_differences_EXPLORATORY.csv")
    sb = pd.read_csv(OUT / "intervals" / "site_blind_intervals_EXPLORATORY.csv")
    gap, t4 = A["table_2b_generalisation_gap"], A["table_4_demographics"]
    note_sha = json.loads(SIDECAR_PATH.read_text(encoding="utf-8"))["sha256"]
    c1 = json.loads((OUT / "claim1" / "claim1_result.json").read_text(encoding="utf-8"))
    hc = json.loads((OUT / "holdout_check" / "holdout_check_result.json").read_text(encoding="utf-8"))

    def dis(summary):
        d = summary.get("disagreements") or []
        return ("DeLong/bootstrap disagreements (listed, not resolved; counted as not detectable): " + "; ".join(f"{c} ({r.replace('disagreement: ', '')})" for c, r in d) + ".") if d else "DeLong/bootstrap disagreements: none."

    p = A["table_5_model_comparisons"]
    prim = p[p["Comparison"].str.contains("Claim 1")]
    show = pd.DataFrame({"Class": prim["Class"], "N pos": prim["N positive"].astype(int), "AUC multimodal": prim["AUC multimodal"].map("{:.4f}".format),
                         "AUC ablation": prim["AUC ablation"].map("{:.4f}".format), "Delta (multimodal - ablation)": prim["Delta (A - B)"].map("{:+.4f}".format),
                         "95% CI": [f"[{a:+.4f}, {b:+.4f}]" for a, b in zip(prim["CI low"], prim["CI high"])],
                         "Holm p": prim["p (Holm)"].map(lambda v: "n/a" if not np.isfinite(v) else f"{v:.2e}" if v < 1e-3 else f"{v:.4f}"), "Reading": prim["Reading"]})
    gtab = pd.DataFrame({"Model": gap["Model"], "Metric": gap["Metric"], "Holdout": gap["Holdout"].map("{:.4f}".format), "BCN20000": gap["BCN20000"].map("{:.4f}".format),
                         "Gap (holdout minus BCN20000) [95% CI]": [f"{r['Gap (holdout minus BCN20000)']:+.4f} [{r['CI low']:+.4f}, {r['CI high']:+.4f}]" for _, r in gap.iterrows()]})
    ptab = pd.DataFrame({"Dataset": paired["Dataset"], "Metric": paired["Metric"],
                         "Multimodal minus ablation [95% CI]": [f"{r['Point']:+.4f} [{r['CI low']:+.4f}, {r['CI high']:+.4f}]" for _, r in paired.iterrows()]})
    d = {}
    for key, label in (("holdout", "HAM10000 holdout (primary)"), ("bcn", "BCN20000 (EXPLORATORY)")):
        x = t4[(t4["Set"] == label) & (t4["Run"] == "multimodal")]
        d[key] = (len(x), int((x["Status"] == "reported").sum()), int(x["Status"].str.startswith("low sample").sum()), int(x["Status"].str.startswith("undefined").sum()),
                  int(((x["Status"] == "reported") & x["Class"].str.lower().isin(["akiec", "df", "vasc"])).sum()))
    n_iv = int(t4[t4["Run"] == "multimodal"]["AUC-ROC CI low"].notna().sum() + t4[t4["Run"] == "ablation"]["AUC-ROC CI low"].notna().sum())
    base = bl["comparisons"]
    ref = A["table_5b_phase2_reference"]
    sbm = sb[(sb["Metric"] == "Macro AUC-ROC")].iloc[0]

    L = ["# Phase 6 results digest", "",
         f"Written {now_iso()} from the finished result files. Rules note SHA-256 `{note_sha}`. Nothing here was re-run; every number is read from a file listed at the end.", "",
         "Labels used throughout: **VERDICT** (the pooled out-of-fold Claim 1 row), **EXPLORATORY** (chosen or run after seeing earlier results; shown beside, never in place of, the verdict), **SECONDARY** (baseline comparisons).", "",
         "## Claim 1: metadata fusion (multimodal vs the matched image-only ablation)", "",
         f"**{sent['primary']['label']}.** {sent['primary']['sentence']}", "",
         f"Reported as: {sent['primary']['reported_as']}. {dis(c1['primary'])}", "", _md(show), "",
         f"**{sent['fold2']['label']}** Category: {sent['fold2']['category']}. {sent['fold2']['reported_as']}. {dis(c1['sensitivity'])}", "",
         f"{sent['fold2']['sentence']}", "",
         f"**{sent['holdout']['label']}** Category: {sent['holdout']['category']}. {sent['holdout']['reported_as']}. {dis(hc['summary'])}", "",
         f"{sent['holdout']['sentence']}", "",
         "Multimodal minus ablation on each dataset, same lesion resamples (EXPLORATORY, descriptive, not a test):", "", _md(ptab), "",
         "Any reason offered for these results (a weak signal in age, sex and site, the image already carrying the same cues, the recipe, the failed Phase 3 ablation fold) is a hypothesis that was not tested. "
         f"The Phase 5 site-blind run (EXPLORATORY; macro AUC-ROC {sbm['Point']:.4f} [{sbm['CI low']:.4f}, {sbm['CI high']:.4f}]) shows how much the frozen model leaned on its site input on BCN20000; it is not a measure of what the metadata is worth.", "",
         "The pooled out-of-fold set mixes five models per arm and is optimistic (selection on the same folds). DeLong ignores lesion clustering; the lesion bootstrap covers the intervals and the second condition, not the DeLong p-value itself.", "",
         "## Claim 2: XAI validation", "", crit[crit["Claim"].str.startswith("2")]["Measured"].iloc[0] + ". Table 3 is the Phase 4 comparison, unchanged.", "", CROSS_REF, "",
         "## Claim 3: generalisation to BCN20000", "",
         "Gaps (holdout minus BCN20000; each dataset bootstrapped separately, interval from the draw-by-draw difference). For ECE and Brier a negative gap means the holdout value is lower (better).", "", _md(gtab), "",
         "Closest published figures from Phase 5, not like-for-like, no pass mark: AUROC drop 0.075 (best 2019 challenge algorithm, AUROC 0.981 to 0.907); balanced-accuracy drop 23.2 points (best algorithm, 82.0% to 58.8%) and 22.3 points (mean over all algorithms, SD 8.6). The gap mixes hospital, camera and preprocessing differences that cannot be separated here.", "",
         "The criterion (BCN20000 evaluated and the gap compared with both anchors) was satisfied by measuring and reporting; no pass mark is applied.", "",
         "## Claim 4: demographic disaggregation", "",
         f"Holdout (primary), per model: {d['holdout'][0]} cells; {d['holdout'][1]} reported, {d['holdout'][2]} low sample, {d['holdout'][3]} undefined. "
         f"BCN20000 (EXPLORATORY): {d['bcn'][0]} cells; {d['bcn'][1]} reported, {d['bcn'][2]} low sample, {d['bcn'][3]} undefined. "
         + ("AKIEC, DF and VASC never reach 30 positives in any holdout subgroup, so none of their holdout cells has an interval ('not estimable at this sample size'). " if d["holdout"][4] == 0 else "")
         + f"Intervals ({n_iv} rows over both models and datasets) are on reported cells only; low-sample and undefined cells keep their Phase 5 flags (Table 4). "
         "Site is both a model input and a disaggregation axis, so the finding is consistency given site as an input, not unconditional fairness. Skin tone is not addressed.", "",
         "The criterion (every cell reported under the two-tier system) was satisfied by counting and reporting every cell.", "",
         "## Secondary: ResNet50 and ViT-B/16 (SECONDARY, not a verdict)", "",
         f"Against ResNet50: {base['multimodal_vs_resnet50']['reported_as']}. {dis(base['multimodal_vs_resnet50'])} Against ViT-B/16: {base['multimodal_vs_vit_b_16']['reported_as']}. {dis(base['multimodal_vs_vit_b_16'])} "
         + bl["caveat"], "",
         "Locked Phase 2 baseline rows, reference only:", "", _md(ref.round(4)), "",
         "## Literature (closest published figures; not a benchmark, not ranked)", "", _md(lit), "", rec_b["literature_note"], "",
         "## Success criteria and stretch goals, as they stand", "", _md(crit), "", _md(stretch), "",
         "## Other limits to carry", "",
         "- Phase 3 predictions are bf16, Phases 4 and 5 are fp32; no table mixes them without a label.",
         "- ViT-B/16 is not bit-reproducible; its wiring tolerance was fixed in advance (0.003).",
         "- The holdout and BCN20000 rows use one fold-3 model per arm.",
         "- Several rows were chosen after seeing earlier results (fold-2 sensitivity, holdout check, paired differences); each is labelled EXPLORATORY.",
         "- Wording: every statement about metadata is limited to this setup (three fields, this fusion method, this dataset).", "",
         "## Files quoted", ""]
    files = json.loads(RESULT11A.read_text(encoding="utf-8"))["files"]
    L += [_md(pd.DataFrame([{"File (assembly/)": n, "Rows": e["rows"], "SHA-256 (first 16)": e["sha256"][:16]} for n, e in files.items()]))]
    return "\n".join(L) + "\n"


def run_digest():
    banner("PHASE 6 -- MODULE 11 (3/3): RESULTS DIGEST")
    if DIGEST.exists():
        side = json.loads(DIGEST_SHA.read_text(encoding="utf-8"))
        ok = sha256_file(DIGEST) == side["sha256"]
        print(f"  Already written {side['recorded_at']}; hash {'matches' if ok else 'DOES NOT MATCH'} its record ({side['sha256'][:16]}...).")
        return side
    ra, rb = json.loads(RESULT11A.read_text(encoding="utf-8")), json.loads(RESULT11B.read_text(encoding="utf-8"))

    def _t_hash():
        bad = [n for n, e in ra["files"].items() if sha256_file(ASM_DIR / n) != e["sha256"]] + [n for n, h in rb["files"].items() if sha256_file(ASM_DIR / n) != h]
        bad += [k for k, v in ra["source_sha256"].items() if sha256_file(SRC[k]) != v]
        return not bad, "every assembled table and every source file equals the hash recorded when the tables were built" if not bad else f"changed since: {bad}"
    text = _build_digest()

    def _t_words():
        low = text.lower()
        heads = [l for l in text.splitlines() if l.startswith("#")]
        bad = [w for w in ("metadata does not help", "significan") if w in low]
        bad += [h for h in heads if re.search(r"\bmet\b", h.lower())]
        return not bad, "no 'significant', no 'metadata does not help', no 'met' headline" if not bad else f"found: {bad}"

    def _t_form():
        need = ["## Claim 1", "## Claim 2", "## Claim 3", "## Claim 4", CROSS_REF[:40], "closest published", "EXPLORATORY"]
        miss = [n for n in need if n not in text]
        return not miss, "all four claims, the cross-reference sentence and the labels are present" if not miss else f"missing: {miss}"
    rows = run_checks("MODULE 11 (3/3): DIGEST CHECKS", [("Quoted files unchanged since they were assembled", _t_hash), ("Wording rules", _t_words), ("Structure", _t_form)])
    if not all(r["passed"] for r in rows):
        print("\n  STOP: nothing was written. Send Claude the FAIL lines.")
        return None
    atomic_write_bytes(DIGEST, text.encode("utf-8"))
    back = DIGEST.read_text(encoding="utf-8")
    assert back == text, "Digest read-back failed"
    side = {"format_version": 1, "recorded_at": now_iso(), "sha256": sha256_file(DIGEST), "bytes": DIGEST.stat().st_size, "lines": len(text.splitlines())}
    write_json_once(DIGEST_SHA, side)
    print(f"\n  Digest written and read back from disk: {DIGEST}  ({side['lines']} lines)\n")
    print(text)
    print("  Module 11 is complete. Next: Module 12 (exit gate), after the walk-through.")
    return side


DIGEST11 = run_digest()


PHASE 6 -- MODULE 11 (3/3): RESULTS DIGEST

MODULE 11 (3/3): DIGEST CHECKS
  [PASS] Quoted files unchanged since they were assembled  --  every assembled table and every source file equals the hash recorded when the tables were built
  [PASS] Wording rules  --  no 'significant', no 'metadata does not help', no 'met' headline
  [PASS] Structure  --  all four claims, the cross-reference sentence and the labels are present

  3 of 3 passed

  Digest written and read back from disk: T:\Thesis - Development\Resources\Artifacts\Phase 6 - P1\assembly\results_digest.md  (143 lines)

# Phase 6 results digest

Written 2026-10-09T15:48:59+06:00 from the finished result files. Rules note SHA-256 `ccb068c346db6e1c6abcfaf4ad283dd2d9edd85b1f67b2791854d93b8dbd4c5e`. Nothing here was re-run; every number is read from a file listed at the end.

Labels used throughout: **VERDICT** (the pooled out-of-fold Claim 1 row), **EXPLORATORY** (chosen or run after seeing earlier results; shown beside, never in pl

# Module 12

In [54]:
"""
Phase 6 -- Module 12: the exit gate. Closes Phase 6.

Requires cells 01_00 to 11_02.  Just run the cell (a few minutes: it re-hashes the 71 frozen inputs, including the checkpoints).

Checks, in order (every one must pass, or nothing is written):
  1. every frozen Phase 1-5 input still equals its hash from the start of Phase 6 (the read-only guard);
  2. the rules note is unchanged and older than every Phase 6 result file;
  3. every expected result file exists, and every json / csv file reads back;
  4. the earlier gates recorded in the result files all passed (input checks, known-answer tests and null checks, real-data null gate,
     scoring wiring proof, OOF alignment, holdout input checks);
  5. the hashes recorded along the way still match (assembled tables and the files they quote, site-blind predictions file,
     HAM10000_metadata.tab, results digest);
  6. the anchor figures 0.075 / 23.2 / 22.3 equal Phase 5's gap table and appear in the digest; the digest wording rules still hold.
If all pass it writes final/run_config.json and final/phase6_exit_gate.json (each once).
"""

import json

RULES6 = check_rules_note()
FINAL_DIR = OUT / "final"
RUN_CONFIG = FINAL_DIR / "run_config.json"
EXIT_GATE = FINAL_DIR / "phase6_exit_gate.json"
NOTE_SHA_EXPECTED = "ccb068c346db6e1c6abcfaf4ad283dd2d9edd85b1f67b2791854d93b8dbd4c5e"
ANCHORS = {"AUROC drop (best 2019 algorithm)": 0.075, "balanced-accuracy drop, best algorithm (points)": 23.2, "balanced-accuracy drop, mean of all algorithms (points)": 22.3}
EXPECTED_FILES = (
    ["phase6_start_fingerprints.json", "phase6_input_checks.json", "rules/phase6_rules_note.json", "rules/phase6_rules_note.sha256.json",
     "assembly/module03_test_record.json", "baseline_predictions/scoring_pass_record.json", "oof/oof_predictions.npz", "oof/oof_alignment_record.json",
     "claim1/real_data_null_gate.json", "claim1/claim1_primary_table.csv", "claim1/claim1_fold2_sensitivity_table.csv", "claim1/claim1_result.json",
     "holdout_check/holdout_check_table.csv", "holdout_check/holdout_check_result.json",
     "baselines/multimodal_vs_resnet50_table.csv", "baselines/multimodal_vs_vit_b_16_table.csv", "baselines/baseline_comparisons_result.json",
     "intervals/headline_intervals.csv", "intervals/per_class_intervals.csv", "intervals/site_blind_intervals_EXPLORATORY.csv", "intervals/gap_intervals.csv",
     "intervals/paired_differences_EXPLORATORY.csv", "intervals/intervals_result.json",
     "demographic_intervals/demographic_intervals_holdout.csv", "demographic_intervals/demographic_intervals_bcn_EXPLORATORY.csv",
     "demographic_intervals/cells_without_interval.csv", "demographic_intervals/demographic_intervals_result.json",
     "assembly/table_1_holdout_with_intervals.csv", "assembly/table_2_bcn20000_with_intervals.csv", "assembly/table_2b_generalisation_gap.csv",
     "assembly/table_2c_published_figures_phase5.csv", "assembly/table_3_xai_validation_phase4.csv", "assembly/table_4_demographics.csv",
     "assembly/table_5_model_comparisons.csv", "assembly/table_5b_phase2_reference.csv", "assembly/literature_table.csv", "assembly/success_criteria.csv",
     "assembly/stretch_goals.csv", "assembly/assembly_tables_record.json", "assembly/criteria_and_literature_record.json", "assembly/results_digest.md",
     "assembly/results_digest.sha256.json"]
    + [f"baseline_predictions/{m}_fold{k}_predictions.npz" for m in ("resnet50", "vit_b_16") for k in range(5)]
    + [f"intervals/boot_{ds}_{run}.npz" for ds, run in (("holdout", "multimodal"), ("holdout", "ablation"), ("bcn", "multimodal"), ("bcn", "ablation"), ("bcn", "site_blind"))])
_J = lambda rel: json.loads((OUT / rel).read_text(encoding="utf-8"))
STATE = {}


def _t_inputs():
    saved = _J("phase6_start_fingerprints.json")["entries"]
    reg = build_registry()
    now = {e["name"]: fingerprint_entry(e) for e in reg if e["path"].exists()}
    changed = [n for n in saved if n not in now or now[n]["sha256"] != saved[n]["sha256"] or now[n]["bytes"] != saved[n]["bytes"]]
    new = [n for n in now if n not in saved]
    STATE["n_inputs"] = len(saved)
    return not (changed or new), (f"all {len(saved)} frozen inputs equal their start-of-Phase-6 hashes ({sum(v['bytes'] for v in now.values()) / 1e9:.2f} GB)"
                                  if not (changed or new) else f"changed: {changed}; not in the start record: {new}")


def _t_rules():
    side = _J("rules/phase6_rules_note.sha256.json")["sha256"]
    note = OUT / "rules" / "phase6_rules_note.json"
    results = find_result_files()
    first = min(results, key=lambda p: p.stat().st_mtime)
    ok_time = note.stat().st_mtime < first.stat().st_mtime
    STATE["note_sha"] = sha256_file(note)
    return (sha256_file(note) == side == NOTE_SHA_EXPECTED and ok_time), \
        (f"rules note SHA-256 {sha256_file(note)[:16]}... equals its sidecar and the recorded value; older than all {len(results)} result files "
         f"(earliest: {first.relative_to(OUT)})")


def _t_files():
    missing = [r for r in EXPECTED_FILES if not (OUT / r).exists()]
    bad = []
    for r in EXPECTED_FILES:
        p = OUT / r
        try:
            if p.suffix == ".json":
                json.loads(p.read_text(encoding="utf-8"))
            elif p.suffix == ".csv":
                pd.read_csv(p)
        except Exception as exc:
            bad.append(f"{r}: {type(exc).__name__}")
    extra = sorted(str(p.relative_to(OUT)).replace("\\", "/") for p in find_result_files() if str(p.relative_to(OUT)).replace("\\", "/") not in EXPECTED_FILES
                   and p.parent != FINAL_DIR)
    STATE["extra_files"] = extra
    return not (missing or bad), (f"all {len(EXPECTED_FILES)} expected files exist and read back" + (f"; other files present (information): {extra}" if extra else "")
                                  if not (missing or bad) else f"missing: {missing}; unreadable: {bad}")


def _t_earlier_gates():
    probs = []
    if not _J("phase6_input_checks.json")["all_passed"]:
        probs.append("input checks")
    m3 = _J("assembly/module03_test_record.json")
    if not m3["all_passed"]:
        probs.append("module 03 tests")
    if not _J("claim1/real_data_null_gate.json")["passed"]:
        probs.append("real-data null gate")
    folds = _J("baseline_predictions/scoring_pass_record.json")["folds"]
    if len(folds) != 10 or any(abs(f["difference"]) > f["tolerance"] for f in folds):
        probs.append("scoring wiring proof")
    oof = _J("oof/oof_alignment_record.json")
    if not all(c["passed"] for c in oof["checks"]):
        probs.append("OOF alignment")
    if not all(c["passed"] for c in _J("holdout_check/holdout_check_result.json")["input_checks"]):
        probs.append("holdout input checks")
    STATE["wiring_worst"] = max(abs(f["difference"]) for f in folds)
    return not probs, (f"input checks, {len(m3['known_answer_tests'])} known-answer tests + {len(m3['null_checks'])} null checks, real-data null gate, "
                       f"10 of 10 scoring folds (largest difference {STATE['wiring_worst']:.1e}), {len(oof['checks'])} OOF checks, holdout input checks: all passed"
                       if not probs else f"not passed: {probs}")


def _t_recorded_hashes():
    bad = []
    a, b = _J("assembly/assembly_tables_record.json"), _J("assembly/criteria_and_literature_record.json")
    bad += [n for n, e in a["files"].items() if sha256_file(OUT / "assembly" / n) != e["sha256"]]
    bad += [n for n, h in b["files"].items() if sha256_file(OUT / "assembly" / n) != h]
    bad += [f"source:{k}" for k, v in a["source_sha256"].items() if sha256_file(SRC[k]) != v]
    iv = _J("intervals/intervals_result.json")["site_blind_file"]
    if sha256_file(Path(iv["path"])) != iv["sha256"]:
        bad.append("site-blind predictions file")
    ham = _J("demographic_intervals/demographic_intervals_result.json")["ham_metadata_tab"]
    if sha256_file(Path(ham["path"])) != ham["sha256"]:
        bad.append("HAM10000_metadata.tab")
    if sha256_file(OUT / "assembly" / "results_digest.md") != _J("assembly/results_digest.sha256.json")["sha256"]:
        bad.append("results digest")
    STATE["site_blind_sha"], STATE["ham_sha"] = iv["sha256"], ham["sha256"]
    return not bad, (f"{len(a['files']) + len(b['files'])} assembled files, {len(a['source_sha256'])} quoted source files, the site-blind file, HAM10000_metadata.tab and the digest all equal their recorded hashes"
                     if not bad else f"changed since recorded: {bad}")


def _t_anchors_wording():
    gap = pd.read_csv(P6["registry"]["p5_bcn_gap_table"])
    col = [c for c in gap.columns if c.lower().startswith("published gap")][0]
    vals = sorted({round(float(v), 3) for v in gap[col]})
    text = (OUT / "assembly" / "results_digest.md").read_text(encoding="utf-8")
    low = text.lower()
    heads = [l for l in text.splitlines() if l.startswith("#")]
    bad = [w for w in ("metadata does not help", "significan") if w in low] + [h for h in heads if re.search(r"\bmet\b", h.lower())]
    ok = vals == sorted(ANCHORS.values()) and all(s in text for s in ("0.075", "23.2", "22.3")) and not bad
    return ok, f"Phase 5 published gaps {vals} = anchors {sorted(ANCHORS.values())}; all three in the digest; wording rules hold" if ok else f"gaps {vals}; wording problems {bad}"


import re
rows = run_checks("PHASE 6 -- MODULE 12: EXIT GATE", [
    ("Frozen Phase 1-5 inputs unchanged (read-only guard)", _t_inputs), ("Rules note unchanged and recorded before every result", _t_rules),
    ("Every expected result file exists and reads back", _t_files), ("Earlier gates recorded as passed", _t_earlier_gates),
    ("Recorded hashes still match", _t_recorded_hashes), ("Anchor figures and digest wording", _t_anchors_wording)])


def write_final():
    if not all(r["passed"] for r in rows):
        print("\n  EXIT GATE FAILED: nothing was written. Send Claude the FAIL lines; no rule is changed without a recorded decision.")
        return None
    FINAL_DIR.mkdir(exist_ok=True)
    if RUN_CONFIG.exists() or EXIT_GATE.exists():
        print("\n  Final records already exist (written once); the checks above were re-run against them and passed.")
        return _J("final/phase6_exit_gate.json")
    files = {r: {"bytes": (OUT / r).stat().st_size, "sha256": sha256_file(OUT / r)} for r in EXPECTED_FILES}
    cfg = {"format_version": 1, "recorded_at": now_iso(), "phase": "Phase 6 - Statistical Comparison & Results Assembly",
           "settings": SETTINGS, "expected_counts": EXPECTED, "classes": list(CLASSES),
           "methods": {"paired_test": "per-class one-vs-rest DeLong (Sun & Xu fast form), Holm across 7", "bootstrap": "lesion-level, 1,000 resamples, seed 42, percentile 95%",
                       "metric_library": "unchanged copy of Phase 5's (Module 03); reproduces the Phase 5 tables to 1e-16 (Modules 03 and 09)",
                       "scoring_precision_baselines": "bf16 autocast (ResNet50, ViT-B/16)", "wiring_tolerances": {"resnet50": 0.001, "vit_b_16": 0.003}},
           "anchors_used": ANCHORS, "rules_note_sha256": STATE["note_sha"], "start_fingerprints_sha256": sha256_file(OUT / "phase6_start_fingerprints.json"),
           "n_frozen_inputs_rechecked": STATE["n_inputs"], "site_blind_predictions_sha256": STATE["site_blind_sha"], "ham10000_metadata_tab_sha256": STATE["ham_sha"],
           "environment": environment_snapshot(), "outputs": files}
    gate = {"format_version": 1, "recorded_at": now_iso(), "all_passed": True, "checks": rows, "n_outputs": len(files),
            "other_files_present": STATE["extra_files"], "scoring_wiring_largest_difference": STATE["wiring_worst"],
            "next": "Cross-check against the teammate's Phase 6 notebook: comparison only, no rule changes."}
    write_json_once(RUN_CONFIG, cfg)
    write_json_once(EXIT_GATE, gate)
    assert _J("final/phase6_exit_gate.json")["all_passed"] and len(_J("final/run_config.json")["outputs"]) == len(EXPECTED_FILES), "Final record read-back failed"
    print(f"\n  Written and read back: {RUN_CONFIG}\n                         {EXIT_GATE}")
    print("\n  PHASE 6 EXIT GATE PASSED. Phase 6 is closed.")
    return gate


EXIT12 = write_final()


PHASE 6 -- MODULE 12: EXIT GATE
  [PASS] Frozen Phase 1-5 inputs unchanged (read-only guard)  --  all 71 frozen inputs equal their start-of-Phase-6 hashes (2.55 GB)
  [PASS] Rules note unchanged and recorded before every result  --  rules note SHA-256 ccb068c346db6e1c... equals its sidecar and the recorded value; older than all 53 result files (earliest: assembly\module03_test_record.json)
  [PASS] Every expected result file exists and reads back  --  all 57 expected files exist and read back
  [PASS] Earlier gates recorded as passed  --  input checks, 11 known-answer tests + 2 null checks, real-data null gate, 10 of 10 scoring folds (largest difference 0.0e+00), 9 OOF checks, holdout input checks: all passed
  [PASS] Recorded hashes still match  --  11 assembled files, 15 quoted source files, the site-blind file, HAM10000_metadata.tab and the digest all equal their recorded hashes
  [PASS] Anchor figures and digest wording  --  Phase 5 published gaps [0.075, 22.3, 23.2] = anchors [0.